<a href="https://colab.research.google.com/github/tommbk50-hub/Cheminformatics/blob/main/pyOpenMS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Why use OpenMS

OpenMS is an open-source, C++ framework hosted on Github <https://github.com/OpenMS/OpenMS> for analyzing large volumes of mass spectrometry data. It has been specially designed for analyzing high performance LC-MS data but over recent times, has been extended to analyze data generated by other techniques.

https://pyopenms.readthedocs.io/en/latest/user_guide/background.html


Installing these programmes by uv is much faster than installing via pip. Here is why uv excels with this specific stack:

1. **Massive Parallel Downloads & Compilations:** Libraries like rdkit, matchms, ms2deepscore, and their dependencies (numba, scipy, matplotlib, torch) are heavy. Standard pip downloads and builds them sequentially, whereas uv processes downloads, metadata extraction, and wheel building in parallel.

2. **Rust-Powered Engine:** uv is written in Rust and operates directly at the system level. It bypasses Python's interpreter overhead for resolving dependency graphs.

3. **Immediate Mismatch Resolution:** In your notebook's install log, pip spent significant time downloading and building packages before finally failing late with: ERROR: Package 'neims-pytorch' requires a different Python: 3.13.16 not in '==3.12.*'. uv resolves the entire dependency tree before downloading any binaries. It would have detected this environment conflict within milliseconds, saving you minutes of wasted download time.

### 1 · Install the pinned Python stack with `uv`

- **What it does:** installs every library the notebook needs (RDKit, pyOpenMS, pyteomics, hplc-py, Plotly, ipywidgets, google-genai) in one resolved pass.
- **How:** `uv pip install --system` resolves and downloads all wheels in parallel, which is far faster than sequential `pip` installs in Colab.
- **Why pinned versions:** the MS/cheminformatics stack is tightly coupled (numpy ↔ numba ↔ scipy), so exact pins avoid ABI breakage mid-session.
- **Run once per runtime** — re-running is a no-op when the wheels are already present.


In [ ]:
#@title 1. Install required Python libraries with pinned versions
!uv pip install --system -q \
    "ipyketcher" \
    "rdkit>=2023.9.1" \
    "mols2grid==2.0.0" \
    "pyteomics==4.6.3" \
    "pyopenms>=3.1.0" \
    "psims==1.3.1" \
    "lxml>=5.1.0" \
    "hdf5plugin==4.3.0" \
    "pandas>=2.0.0,<3.0.0" \
    "pubchempy==1.0.4" \
    "mendeleev" \
    "marimo==0.2.5" \
    "wigglystuff==0.1.3" \
    "matchms" \
    "onnxruntime>=1.17.1" \
    "bottleneck==1.3.8" \
    "plotly>=5.18.0"

!pip -q install kaleido==0.1.0

import os
import glob
import pandas as pd
import matplotlib.colors as mcolors

# Helper to convert matplotlib colors to Plotly-compatible RGB strings
def to_plotly_color(color_val):
    """Converts any matplotlib-compatible color (including RGB tuples) to an rgb(r,g,b) string for Plotly."""
    rgba = mcolors.to_rgba(color_val)
    return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"

# 1. Clone hplc-py for chromatographic deconvolution
if not os.path.exists('hplc-py'):
    !git clone -q https://github.com/cremerlab/hplc-py.git

### 2 · Download the mzML dataset and build the HPLC trace

- **What it does:** fetches the reference `FeatureFinderMetaboIdent_1_input.mzML` file (skipped if it already exists locally) and parses it with `pyteomics.mzml`.
- **How:** iterates every MS1 spectrum, reads the scan start time and total ion current, and assembles them into the `hplc_df` DataFrame (`Time`, `Signal`) that every later cell consumes.
- **Output:** a matplotlib preview of the raw chromatogram so you can sanity-check the retention-time window before deconvolution.
- **Swap in your own data** by replacing the download with an upload — everything downstream only depends on `hplc_df` and the mzML path.


In [ ]:
#@title 2. Plot HPLC Chromatogram, Convert the HPLC Trace into a Pandas dataframe
import urllib.request
import os
import matplotlib.pyplot as plt
from pyteomics import mzml
import pandas as pd

# 1. Download the new dataset
url_new = "https://raw.githubusercontent.com/OpenMS/OpenMS/develop/src/tests/topp/FeatureFinderMetaboIdent_1_input.mzML"
filename_new = "FeatureFinderMetaboIdent_1_input.mzML"

if not os.path.exists(filename_new):
    print(f"Downloading {filename_new}...")
    urllib.request.urlretrieve(url_new, filename_new)

# 2. Parse the new dataset
print(f"Parsing {filename_new}...")
rt_new = []
tic_new = []

try:
    with mzml.read(filename_new) as reader:
        for spectrum in reader:
            if spectrum['ms level'] == 1:
                scan = spectrum.get('scanList', {}).get('scan', [{}])[0]
                rt = scan.get('scan start time', 0)
                tic = spectrum.get('total ion current', 0)
                rt_new.append(rt)
                tic_new.append(tic)

    # Create a pandas DataFrame from the parsed data
    hplc_df = pd.DataFrame({
        'Signal': tic_new,
        'Time': rt_new
    })

    print("HPLC Trace DataFrame successfully created:")
    print(hplc_df.head())

    # 3. Plot the new HPLC trace using the DataFrame
    plt.figure(figsize=(10, 6))
    plt.plot(hplc_df['Time'], hplc_df['Signal'], color='green', linewidth=1.5)
    plt.title('HPLC Trace: FeatureFinderMetaboIdent_1_input.mzML')
    plt.xlabel('Retention Time (seconds)')
    plt.ylabel('Total Ion Current (Signal)')
    plt.grid(True)
    plt.tight_layout()
    # Cache this Figure (before matplotlib's inline backend auto-closes it after
    # this cell finishes) so later cells can feed it into Gemini's vision context.
    globals().setdefault('_notebook_figure_registry', {})['raw_hplc_chromatogram'] = plt.gcf()

    plt.show()

except Exception as e:
    print(f"An error occurred: {e}")

### 3 · Enter the molecules you are looking for

- **What it does:** renders an editable ipywidgets table (optionally an `ipyketcher` sketcher) for the candidate molecules — name, classification (SM / product / byproduct) and SMILES.
- **How:** each row is validated with RDKit on entry; `get_candidates_from_table()` is the single accessor every later cell (and every Gemini tool) uses to read your list.
- **Why it matters:** these SMILES drive the theoretical adduct/dimer matrices, cLogP/TPSA polarity checks and the peak-assignment confidence scoring.
- **Fallback:** if the widget table is empty, a built-in nucleoside list is used so the notebook always runs end-to-end.


In [ ]:
#@title 3. List SMILES to find
import pandas as pd
import os
import sys
import traitlets
import subprocess
import ipywidgets as widgets
from rdkit import Chem
from rdkit.Chem import Descriptors

# Dynamic patch to solve 'TypeError: type "List" is not subscriptable' on Python 3.13 inside ipyketcher[cite: 1]
if not hasattr(traitlets.List, '__class_getitem__'):
    traitlets.List.__class_getitem__ = classmethod(lambda cls, item: cls)

# --- PATCH IPYKETCHER BUG ---[cite: 1]
ipk_init = f"/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages/ipyketcher/__init__.py"
if os.path.exists(ipk_init):
    with open(ipk_init, 'r') as f:
        content = f.read()
    if "traitlets.List[str]()" in content:
        with open(ipk_init, 'w') as f:
            f.write(content.replace("traitlets.List[str]()", "traitlets.List()"))
# ----------------------------

# Force reload in case the broken version was already cached in memory[cite: 1]
if 'ipyketcher' in sys.modules:
    del sys.modules['ipyketcher']
import ipyketcher
from IPython.display import display, HTML

# 1. Initialize and display the molecular sketching canvas[cite: 1]
editor = ipyketcher.KetcherEditor()
print("🎨 Canvas initialized. (Draw molecules to add them to your interactive table below)")
display(editor)

# Helper function to calculate exact mass via RDKit
def calculate_exact_mass(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol:
        return f"{Descriptors.ExactMolWt(mol):.4f}"
    return "Invalid"

# Define reaction scheme list dynamically with the requested molecules[cite: 1]
reaction_smiles = [
    ("adenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
    ("n-deoxyadenosine", "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)C1"),
    ("n-inosine", "Product", "O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
    ("n2-o-methylcytidine", "Product", "CO[C@@H]1[C@H](O[C@H](CO)[C@@H]1O)n2ccc(N)nc2=O"),
    ("n5-methylcytidine", "Product", "Cc1cn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)nc1N"),
    ("n5-formylcytidine", "Product", "Nc1nc(=O)n([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)cc1C=O")
]

# Create a UI Row for the interactive table
def create_table_row(comp_no, name, classification, smiles):
    w_no = widgets.HTML(value=f"<b>{comp_no}</b>", layout=widgets.Layout(width='40px', padding='0 5px'))
    w_name = widgets.Text(value=name, placeholder='Molecule Name', layout=widgets.Layout(width='160px', padding='0 5px'))
    w_class = widgets.Dropdown(
        options=['Reactant', 'Reagent', 'Product'],
        value=classification,
        layout=widgets.Layout(width='100px', padding='0 5px')
    )
    w_smiles = widgets.Text(value=smiles, layout=widgets.Layout(width='320px', padding='0 5px'))
    w_mass = widgets.HTML(value=calculate_exact_mass(smiles), layout=widgets.Layout(width='80px', padding='0 5px'))

    return widgets.HBox([w_no, w_name, w_class, w_smiles, w_mass])

# Build Table Header
header_row = widgets.HBox([
    widgets.HTML(value="<b>No.</b>", layout=widgets.Layout(width='40px', padding='0 5px')),
    widgets.HTML(value="<b>Name</b>", layout=widgets.Layout(width='160px', padding='0 5px')),
    widgets.HTML(value="<b>Classification</b>", layout=widgets.Layout(width='100px', padding='0 5px')),
    widgets.HTML(value="<b>SMILES String</b>", layout=widgets.Layout(width='320px', padding='0 5px')),
    widgets.HTML(value="<b>Exact Mass</b>", layout=widgets.Layout(width='80px', padding='0 5px'))
])

# Populate initial table rows
table_rows = [create_table_row(i + 1, name, cls, smi) for i, (name, cls, smi) in enumerate(reaction_smiles)]
table_container = widgets.VBox([header_row] + table_rows)

btn_update = widgets.Button(description='Update List', button_style='success')
out = widgets.Output()

def on_update_clicked(b):
    with out:
        out.clear_output()
        # ipyketcher stores current smiles in editor.smiles[cite: 1]
        drawn_smi = getattr(editor, 'smiles', None)
        if drawn_smi:
            # Check for duplicates based on existing SMILES in the table
            existing_smiles = [row.children[3].value for row in table_container.children[1:]]
            if drawn_smi not in existing_smiles:
                new_index = len(table_container.children) # Header counts as 1, so len equals the next compound number
                new_row = create_table_row(new_index, "Drawn Molecule", "Reactant", drawn_smi)

                # Append the new row to the table container
                table_container.children = list(table_container.children) + [new_row]
                print(f"Added to table: {drawn_smi}")
            else:
                print(f"Molecule already exists in the table: {drawn_smi}")
        else:
            print("No SMILES found in editor. Please draw a molecule first.")[cite: 1]

btn_update.on_click(on_update_clicked)

display(table_container)
display(widgets.HBox([btn_update]))
display(out)

Here we use an **LC-MS1 alignment/extraction pipeline**.

 We can use the OpenMS **Python wrapper, pyOpenMS**, directly inside this colab notebook to perform a precise, high-resolution Extracted Ion Chromatogram (EIC) extraction and retention time alignment for your target molecular masses.

### 4 · Unified peak detection & dominant-m/z confidence scoring

- **What it does:** optimises the prominence threshold (L-curve elbow), deconvolutes overlapping peaks with `hplc-py`, and produces `detected_df`.
- **How:** `scipy.signal.find_peaks` locates apexes, peak bases map onto fitted peak windows, and the MS1 scans inside each window are summed into the top-10 `dominant m/z` list.
- **Scoring:** every candidate molecule is screened across the ESI+ adduct table, then scored on mass accuracy (ppm), dominant-ion rank, isotope-envelope agreement and corroborating adducts/in-source losses.
- **Outputs used downstream:** `detected_df`, `peak_assignments`, `peak_ms1_spectra`, `chrom` and the pyOpenMS experiment `od`.


In [ ]:
#@title 4. Unified Peak Detection & Dominant-m/z Confidence Scoring (dataframe = detected_df)

import subprocess
import sys
import os
import re
import warnings

try:
    import scipy.signal
except ImportError:
    print("🔧 Installing missing dependencies (scipy, numpy) via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "numpy", "scipy"], capture_output=True)
    import scipy.signal
    print("✅ Repair complete.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets
from pyteomics import mzml
import pyopenms as po
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

warnings.filterwarnings('ignore')

# 1. Ensure hplc-py and pyOpenMS experiment are loaded
if 'hplc-py' not in sys.path:
    sys.path.append(os.path.abspath('hplc-py'))
from hplc.quant import Chromatogram

mzml_filename = "FeatureFinderMetaboIdent_1_input.mzML"
if not os.path.exists(mzml_filename):
    mzml_filename = "/content/FeatureFinderMetaboIdent_1_input.mzML"

if 'od' not in globals() and os.path.exists(mzml_filename):
    od = po.MSExperiment()
    po.FileHandler().loadExperiment(mzml_filename, od)

signal_raw = hplc_df['Signal'].values
signal_norm = (signal_raw - signal_raw.min()) / (signal_raw.max() - signal_raw.min())

# --- 2. Dynamic Hyperparameter Optimization (Elbow Method) ---
print("Running Dynamic Hyperparameter Optimization...")
prominence_range = np.logspace(-2, -0.5, 8)
buffer_val = 0
results = []
chrom_base = Chromatogram(hplc_df, cols={'time': 'Time', 'signal': 'Signal'}, time_window=[200, 300])

for p in prominence_range:
    try:
        peaks = chrom_base.fit_peaks(prominence=p, buffer=buffer_val)
        results.append((p, len(peaks)))
    except Exception:
        pass

if not results:
    default_prominence = 0.05
    print("Optimization failed. Using default prominence: 0.05")
else:
    proms_all = np.array([x[0] for x in results])
    counts_all = np.array([x[1] for x in results])

    max_idx = np.argmax(counts_all)
    valid_proms = proms_all[max_idx:]
    valid_counts = counts_all[max_idx:]

    if len(valid_proms) > 1:
        p1 = np.array([np.log10(valid_proms[0]), valid_counts[0]])
        p2 = np.array([np.log10(valid_proms[-1]), valid_counts[-1]])

        distances = []
        for i in range(len(valid_proms)):
            p3 = np.array([np.log10(valid_proms[i]), valid_counts[i]])
            if np.linalg.norm(p2 - p1) == 0:
                dist = 0
            else:
                cross_prod_2d = (p2[0] - p1[0]) * (p1[1] - p3[1]) - (p2[1] - p1[1]) * (p1[0] - p3[0])
                dist = np.abs(cross_prod_2d) / np.linalg.norm(p2 - p1)
            distances.append(dist)

        optimal_idx_relative = np.argmax(distances)
        optimal_prominence = valid_proms[optimal_idx_relative]
        optimal_count = valid_counts[optimal_idx_relative]
    else:
        optimal_prominence = valid_proms[0]
        optimal_count = valid_counts[0]

    default_prominence = optimal_prominence
    prominence_val = optimal_prominence

    display(Markdown("---"))
    display(Markdown(f"### ✅ Optimization Complete"))
    display(Markdown(f"**Optimal Prominence Threshold:** `{optimal_prominence:.4f}` | **Peaks Resolved:** `{optimal_count}`"))

    fig_opt, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [2.5, 1]})
    ax1.plot(proms_all, counts_all, marker='o', linestyle='-', color='#2c3e50', linewidth=2, label='Fitted Peaks')
    ax1.axvline(optimal_prominence, color='#e74c3c', linestyle='--', linewidth=2, label=f'Optimal Elbow ({optimal_prominence:.4f})')
    ax1.scatter([optimal_prominence], [optimal_count], color='#e74c3c', s=150, zorder=5)
    ax1.set_xscale('log')
    ax1.set_xticks(prominence_range)
    ax1.set_xticklabels([f"{x:.4f}" for x in prominence_range], rotation=45)
    ax1.set_title('Hyperparameter Optimization: Signal-to-Noise Ratio (L-Curve)', fontsize=13, pad=12)
    ax1.set_xlabel('Prominence Threshold (Log Scale)', fontsize=11)
    ax1.set_ylabel('Number of Deconvoluted Peaks', fontsize=11)
    ax1.legend(fontsize=10)
    ax1.grid(True, which="both", ls="--", alpha=0.4)

    ax2.axis('tight')
    ax2.axis('off')
    table_data = [[f"{p:.4f}", int(c)] for p, c in zip(proms_all, counts_all)]
    col_labels = ['Prominence Threshold', 'Number of\nDeconvoluted Peaks']
    table = ax2.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.1, 1.8)

    optimal_row_idx = list(proms_all).index(optimal_prominence) + 1
    for (row, col), cell in table.get_celld().items():
        if row == optimal_row_idx:
            cell.set_edgecolor('red')
            cell.set_linewidth(2)
    ax2.set_title('Tabular Data', fontsize=12, pad=12)
    plt.tight_layout()
    plt.show()

# --- 3. Candidate Fetching & Theoretical Ion / Isotope Generators ---
def get_candidates_from_table():
    if 'table_container' in globals():
        candidates = []
        for row in table_container.children[1:]:
            name = row.children[1].value.strip()
            classification = row.children[2].value
            smi = row.children[3].value.strip()
            if smi:
                candidates.append((name, classification, smi))
        if candidates:
            return candidates
    return [
        ("adenosine",           "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
        ("n-deoxyadenosine",    "Product", "Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)C1"),
        ("n-inosine",           "Product", "O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O"),
        ("n2-o-methylcytidine", "Product", "CO[C@@H]1[C@H](O[C@H](CO)[C@@H]1O)n2ccc(N)nc2=O"),
        ("n5-methylcytidine",   "Product", "Cc1cn([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)c(=O)nc1N"),
        ("n5-formylcytidine",   "Product", "Nc1nc(=O)n([C@@H]2O[C@H](CO)[C@@H](O)[C@H]2O)cc1C=O")
    ]

esi_pos_adducts = [
    ("[M+H]+",       1, 1.007276,  1),
    ("[M+NH4]+",     1, 18.033823, 1),
    ("[M+Na]+",      1, 22.989218, 1),
    ("[M+CH3OH+H]+", 1, 33.033489, 1),
    ("[M+K]+",       1, 38.963158, 1),
    ("[M+ACN+H]+",   1, 42.033823, 1),
    ("[2M+H]+",      2, 1.007276,  1),
    ("[2M+Na]+",     2, 22.989218, 1),
]

insource_neutral_losses = [
    ("[M+H-Ribose]+",      -132.042259),
    ("[M+H-Deoxyribose]+", -116.047344),
    ("[M+H-MeRibose]+",    -146.057909),
    ("[M+H-H2O]+",         -18.010565),
]

def parse_dominant_mz_string(dom_str):
    if not dom_str or dom_str == "None":
        return []
    parsed = []
    for item in str(dom_str).split(","):
        m = re.search(r"([0-9]+\.[0-9]+)\s*\(([0-9.]+)%\)", item.strip())
        if m:
            parsed.append((float(m.group(1)), float(m.group(2))))
    return parsed

def compute_confidence_from_dominant_mz(best_group, dom_mz_list, full_window_spectrum):
    adducts = best_group["adducts"]
    primary = adducts[0]
    mol = best_group["mols"][0]
    exact_mass = best_group["exact_mass"]

    # Mass Accuracy Score
    mean_abs_ppm = np.mean([abs(a["ppm"]) for a in adducts])
    ppm_score = max(0.0, 35.0 * (1.0 - (mean_abs_ppm / 15.0)))

    # Dominant m/z Rank
    matched_dom_rel_int = 0.0
    for a in adducts:
        for dom_mz, dom_rel in dom_mz_list:
            if abs(dom_mz - a["calc_mz"]) <= 0.015:
                matched_dom_rel_int = max(matched_dom_rel_int, dom_rel)

    if matched_dom_rel_int >= 95.0:
        dom_rank_score = 35.0
    elif matched_dom_rel_int > 0:
        dom_rank_score = 12.0 + 23.0 * (matched_dom_rel_int / 100.0)
    else:
        dom_rank_score = 5.0

    # Isotope Envelope Verification
    num_C = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)
    num_N = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 7)
    num_S = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 16)

    theo_m1_rel = (num_C * 1.07) + (num_N * 0.37) + (num_S * 0.76)
    if theo_m1_rel == 0:
        theo_m1_rel = 12.0

    m0_mz = primary["calc_mz"]
    m1_target = m0_mz + 1.003355
    m2_target = m0_mz + 2.006710

    iso_score = 0.0
    iso_evidence = []

    m1_obs_rel = None
    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m1_target) <= 0.015 and matched_dom_rel_int > 0:
            m1_obs_rel = (dom_rel / matched_dom_rel_int) * 100.0
            break

    if m1_obs_rel is None and full_window_spectrum and matched_dom_rel_int > 0:
        m0_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m0_mz) <= 0.015], default=0)
        m1_int = max([inten for mz_k, inten in full_window_spectrum.items() if abs(mz_k - m1_target) <= 0.015], default=0)
        if m0_int > 0 and m1_int > 0:
            m1_obs_rel = (m1_int / m0_int) * 100.0

    if m1_obs_rel is not None:
        rel_diff = abs(m1_obs_rel - theo_m1_rel)
        iso_score += max(6.0, 16.0 * (1.0 - min(rel_diff / 15.0, 1.0)))
        iso_evidence.append(f"[M+H+1]+ iso ({m1_obs_rel:.0f}% vs {theo_m1_rel:.0f}% theo)")

    for dom_mz, dom_rel in dom_mz_list:
        if abs(dom_mz - m2_target) <= 0.015:
            iso_score += 4.0
            iso_evidence.append("[M+H+2]+ iso")
            break

    # Multi-Adduct & In-Source Fragments
    corrob_score = 0.0
    if len(adducts) > 1:
        corrob_score += min(7.0, (len(adducts) - 1) * 4.5)

    mh_mz = exact_mass + 1.007276
    for loss_name, shift in insource_neutral_losses:
        frag_mz = mh_mz + shift
        if frag_mz > 80.0:
            for dom_mz, dom_rel in dom_mz_list:
                if abs(dom_mz - frag_mz) <= 0.015:
                    corrob_score += 5.0
                    iso_evidence.append(f"{loss_name} ({dom_mz:.3f})")
                    break

    corrob_score = min(10.0, corrob_score)
    total_conf = round(min(99.9, ppm_score + dom_rank_score + iso_score + corrob_score), 1)

    evidence_summary = f"Dom.Rank: {matched_dom_rel_int:.0f}%"
    if iso_evidence:
        evidence_summary += " | " + ", ".join(iso_evidence[:2])

    return total_conf, evidence_summary

# --- 4. Interactive UI & Unified Execution Pipeline ---
prominence_input = widgets.FloatText(
    value=default_prominence,
    description='Prominence:',
    step=0.001,
    style={'description_width': 'initial'}
)

rerun_button = widgets.Button(
    description='Rerun Peak Fitting & Assignment',
    button_style='primary',
    icon='refresh',
    layout=widgets.Layout(width='240px')
)

output_area = widgets.Output()

# Global stores shared with Cell 5
peak_ms1_spectra = {}
chrom = None
current_prominence_val = default_prominence

def run_unified_pipeline(prom_val):
    global detected_df, peak_assignments, peak_ms1_spectra, chrom, current_prominence_val
    current_prominence_val = prom_val

    with output_area:
        clear_output(wait=True)
        print(f"Using prominence threshold: {prom_val:.4f}")

        # Topographical peak detection
        peak_indices, _ = scipy.signal.find_peaks(signal_norm, prominence=prom_val)
        _, left_bases, right_bases = scipy.signal.peak_prominences(signal_norm, peak_indices)

        start_times = hplc_df['Time'].iloc[left_bases].values
        end_times = hplc_df['Time'].iloc[right_bases].values

        print(f"Detected {len(peak_indices)} raw topographical anomalies. Deconvoluting true peaks...")
        chrom = Chromatogram(hplc_df, cols={'time': 'Time', 'signal': 'Signal'}, time_window=[200, 300])
        peaks = chrom.fit_peaks(prominence=prom_val, buffer=0).reset_index(drop=True)

        mapped_starts, mapped_ends, mapped_topos = [], [], []
        peak_rts = hplc_df['Time'].values[peak_indices]

        for rt in peaks['retention_time']:
            if len(peak_rts) > 0:
                closest_idx = np.argmin(np.abs(peak_rts - rt))
                mapped_starts.append(start_times[closest_idx])
                mapped_ends.append(end_times[closest_idx])
                mapped_topos.append(peak_rts[closest_idx])
            else:
                mapped_starts.append(np.nan)
                mapped_ends.append(np.nan)
                mapped_topos.append(np.nan)

        peaks['peak start time'] = mapped_starts
        peaks['peak end time'] = mapped_ends
        peaks['topographical_apex'] = mapped_topos

        if peaks['area'].sum() > 0:
            peaks['normalised area'] = (peaks['area'] / peaks['area'].sum() * 100).map('{:.2f}%'.format)
        else:
            peaks['normalised area'] = '0.00%'

        # Extract top 10 dominant m/z signals
        peak_spectra = {i: {} for i in range(len(peaks))}
        if os.path.exists(mzml_filename):
            with mzml.read(mzml_filename) as reader:
                for spectrum in reader:
                    if spectrum['ms level'] == 1:
                        scan_data = spectrum.get('scanList', {}).get('scan', [{}])
                        rt = scan_data[0].get('scan start time', -1) if isinstance(scan_data, list) and scan_data else (scan_data.get('scan start time', -1) if isinstance(scan_data, dict) else -1)
                        mz_array = spectrum.get('m/z array', [])
                        int_array = spectrum.get('intensity array', [])

                        if len(mz_array) > 0:
                            mzs_rounded = np.round(mz_array, 3)
                            for i, row in peaks.iterrows():
                                if not pd.isna(row['peak start time']) and row['peak start time'] <= rt <= row['peak end time']:
                                    for m, intensity in zip(mzs_rounded, int_array):
                                        peak_spectra[i][m] = peak_spectra[i].get(m, 0) + intensity

        TOP_N_PEAKS = 10
        MIN_REL_INTENSITY = 2.0
        dominant_mz_col = []
        for i in range(len(peaks)):
            spectra = peak_spectra[i]
            if not spectra:
                dominant_mz_col.append("None")
                continue
            sorted_mzs = sorted(spectra.items(), key=lambda x: x[1], reverse=True)
            max_int = sorted_mzs[0][1]
            top_strs = []
            for m, inten in sorted_mzs:
                rel_int = (inten / max_int) * 100
                if rel_int >= MIN_REL_INTENSITY:
                    top_strs.append(f"{m:.3f} ({rel_int:.0f}%)")
                if len(top_strs) >= TOP_N_PEAKS:
                    break
            dominant_mz_col.append(", ".join(top_strs) if top_strs else "None")

        peaks['dominant m/z'] = dominant_mz_col

        # Screen ESI+ adducts & score
        candidates = get_candidates_from_table()
        theoretical_ions = []
        for name, classification, smi in candidates:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                exact_mass = Descriptors.ExactMolWt(mol)
                formula = rdMolDescriptors.CalcMolFormula(mol)
                for adduct_name, mult, shift, z in esi_pos_adducts:
                    calc_mz = (exact_mass * mult + shift) / z
                    theoretical_ions.append({
                        "name": f"{name} ({classification})",
                        "raw_name": name,
                        "classification": classification,
                        "smiles": smi,
                        "mol": mol,
                        "formula": formula,
                        "exact_mass": round(exact_mass, 4),
                        "adduct": adduct_name,
                        "calc_mz": calc_mz
                    })

        PPM_TOLERANCE = 15.0
        MIN_PEAK_INTENSITY = 5e5
        peak_assignments = {}

        if 'od' in globals():
            for idx_row, peak_row in peaks.iterrows():
                p_id = int(peak_row['peak_id'])
                rt_start = peak_row['peak start time']
                rt_end = peak_row['peak end time']
                rt_apex = peak_row['retention_time']
                dom_mz_parsed = parse_dominant_mz_string(peak_row['dominant m/z'])

                if pd.isna(rt_start) or pd.isna(rt_end):
                    continue

                window_scans = [
                    spec for spec in od
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end
                ]
                if not window_scans:
                    continue

                mass_group_hits = {}
                for ion in theoretical_ions:
                    target_mz = ion["calc_mz"]
                    tol_da = target_mz * (PPM_TOLERANCE * 1e-6)

                    max_int = 0.0
                    obs_mz_at_max = target_mz
                    rt_at_max = rt_apex

                    for spec in window_scans:
                        mzs, ints = spec.get_peaks()
                        mask = (mzs >= target_mz - tol_da) & (mzs <= target_mz + tol_da)
                        if np.any(mask):
                            best_idx = np.argmax(ints[mask])
                            peak_int = float(ints[mask][best_idx])
                            if peak_int > max_int:
                                max_int = peak_int
                                obs_mz_at_max = float(mzs[mask][best_idx])
                                rt_at_max = float(spec.getRT())

                    if max_int >= MIN_PEAK_INTENSITY and abs(rt_at_max - rt_apex) <= 6.0:
                        em = ion["exact_mass"]
                        if em not in mass_group_hits:
                            mass_group_hits[em] = {
                                "exact_mass": em,
                                "formula": ion["formula"],
                                "names": [],
                                "raw_names": [],
                                "mols": [],
                                "total_intensity": 0.0,
                                "adducts": {}
                            }
                        if ion["name"] not in mass_group_hits[em]["names"]:
                            mass_group_hits[em]["names"].append(ion["name"])
                            mass_group_hits[em]["raw_names"].append(ion["raw_name"])
                            mass_group_hits[em]["mols"].append(ion["mol"])

                        if ion["adduct"] not in mass_group_hits[em]["adducts"]:
                            ppm_err = ((obs_mz_at_max - target_mz) / target_mz) * 1e6
                            mass_group_hits[em]["adducts"].setdefault(ion["adduct"], {
                                "adduct": ion["adduct"],
                                "obs_mz": obs_mz_at_max,
                                "calc_mz": target_mz,
                                "ppm": ppm_err,
                                "intensity": max_int
                            })
                            mass_group_hits[em]["total_intensity"] += max_int

                if mass_group_hits:
                    best_group = max(mass_group_hits.values(), key=lambda x: x["total_intensity"])
                    sorted_adducts = sorted(best_group["adducts"].values(), key=lambda x: x["intensity"], reverse=True)
                    best_group["adducts"] = sorted_adducts

                    is_isobaric = len(best_group["raw_names"]) > 1
                    iso_tag = f" [Isobaric Isomers, {best_group['formula']}]" if is_isobaric else ""

                    conf_score, evidence_str = compute_confidence_from_dominant_mz(
                        best_group, dom_mz_parsed, peak_spectra.get(idx_row, {})
                    )

                    peak_assignments[p_id] = {
                        "display_name": " / ".join(best_group["raw_names"]),
                        "full_assignment": " / ".join(best_group["names"]) + iso_tag,
                        "is_isobaric": is_isobaric,
                        "formula": best_group["formula"],
                        "mols": best_group["mols"],
                        "adducts": sorted_adducts,
                        "confidence": conf_score,
                        "evidence": evidence_str
                    }

        assignment_col = []
        confidence_col = []
        for idx_row, row in peaks.iterrows():
            p_id = int(row['peak_id'])
            if p_id in peak_assignments:
                assign_info = peak_assignments[p_id]
                top_adduct = assign_info["adducts"][0]
                assignment_col.append(
                    f"{assign_info['full_assignment']} [{top_adduct['adduct']}: {top_adduct['obs_mz']:.4f} m/z ({top_adduct['ppm']:+.1f} ppm)]"
                )
                confidence_col.append(f"{assign_info['confidence']:.1f}% ({assign_info['evidence']})")
            else:
                dom_list = parse_dominant_mz_string(row['dominant m/z'])
                base_str = f"Base m/z: {dom_list[0][0]:.3f}" if dom_list else "No MS1 signal"
                assignment_col.append(f"Unassigned ({base_str})")
                confidence_col.append("N/A")

        peaks['Assignment'] = assignment_col
        peaks['Confidence (%)'] = confidence_col
        detected_df = peaks.copy()

        # Populate the per-peak MS1 spectra store for Cell 5
        peak_ms1_spectra = {int(row['peak_id']): peak_spectra.get(idx_row, {}) for idx_row, row in peaks.iterrows()}

        def highlight_peaks(row):
            colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
            c = mcolors.to_rgba(colors[(int(row['peak_id']) - 1) % len(colors)], alpha=0.4)
            bg_color = f'rgba({int(c[0]*255)}, {int(c[1]*255)}, {int(c[2]*255)}, {c[3]})'
            return [f'background-color: {bg_color}' for _ in row]

        display(detected_df.style.apply(highlight_peaks, axis=1))
        print(f"\nFinal deconvoluted peak count: {len(detected_df)} | Assigned peaks: {len(peak_assignments)}")
        print("✅ Data ready! Run Cell 5 below to render the interactive chromatogram & MS1 comparison widgets.")

def on_rerun_clicked(b):
    run_unified_pipeline(prominence_input.value)

rerun_button.on_click(on_rerun_clicked)

display(widgets.HBox([prominence_input, rerun_button]))
display(output_area)

run_unified_pipeline(default_prominence)

### 5 · Interactive chromatogram with hover structure cards (full version)

- **What it does:** rebuilds the deconvoluted chromatogram in Plotly, with one filled trace per deconvoluted component and an RDKit 2D structure card that floats next to the cursor on hover.
- **How:** each peak's assignment is rendered to a PNG/SVG card, embedded as base64 in a JavaScript tooltip bound to `plotly_hover` events.
- **Extras:** multi-peak MS1 comparison widgets let you overlay the mass spectra of several peaks, and a Kaleido PNG snapshot is cached in `_notebook_figure_registry` for the Step 4 vision QC.


In [ ]:
#@title 5. Plot Deconvoluted HPLC Chromatogram with Hover Structure Cards & Multi-Peak MS1 Comparison

import subprocess
import sys
import io
import json
import base64
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from rdkit.Chem import Draw

try:
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
except ImportError:
    print("🔧 Installing missing dependencies (plotly, kaleido) via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "plotly", "kaleido"], capture_output=True)
    import plotly.graph_objects as go
    import plotly.io as pio
    from plotly.subplots import make_subplots
    print("✅ Repair complete.")

try:
    import kaleido
except ImportError:
    print("🔧 Installing/reloading missing kaleido via uv...")
    subprocess.run(["uv", "pip", "install", "--system", "--upgrade", "kaleido"], capture_output=True)
    print("✅ Kaleido installation complete.")

# Ensure Plotly renders natively in Colab
if 'google.colab' in sys.modules:
    pio.renderers.default = 'colab'

if 'to_plotly_color' not in globals():
    def to_plotly_color(color_val):
        rgba = mcolors.to_rgba(color_val)
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"

if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
    print("⚠️ Required data (`detected_df`, `chrom`) not found. Please run Cell 4 first.")
else:
    def _peak_comparison_label(p_id):
        """Human-readable label for a peak_id, reused by the chromatogram legend and
        the MS1 spectra comparison selector/subplot titles."""
        if 'peak_assignments' in globals() and p_id in peak_assignments:
            a_info = peak_assignments[p_id]
            iso_short = " [Isobaric Isomers]" if a_info["is_isobaric"] else ""
            adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts']])
            return f"Peak {p_id}: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
        if 'detected_df' in globals():
            dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
            if len(dom_rows):
                dom_str = dom_rows.values[0]
                first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
                return f"Peak {p_id}: Unassigned (Base m/z: {first_ion})"
        return f"Peak {p_id}: Unassigned"

    # --- 1. Build Clean Interactive Plotly Chromatogram with Floating Hover Structure Cards ---
    FIG_WIDTH_PX, FIG_HEIGHT_PX = 1300, 800
    MARGIN_L, MARGIN_R, MARGIN_T, MARGIN_B = 80, 40, 65, 210
    X_RANGE = [190, 310]

    time_arr = chrom.df[chrom.time_col].values
    raw_signal = chrom.df[chrom.int_col].values
    convolved = np.sum(chrom.unmixed_chromatograms, axis=1) if chrom.peaks is not None else None
    y_max_plot = float(np.max(raw_signal))
    prom_val = globals().get('current_prominence_val', 0.05)

    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    # Pre-build rich HTML hover cards (including Base64-encoded RDKit 2D structure PNGs) per peak
    hover_cards_by_peak = {}
    qc_struct_images = {}

    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        norm_area = peak_row.get('normalised area', 'N/A')
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            mol_images = [Draw.MolToImage(m, size=(175, 145)) for m in assign["mols"]]
            if len(mol_images) > 1:
                total_w = sum(im.width for im in mol_images)
                struct_img = Image.new('RGBA', (total_w, mol_images[0].height), (255, 255, 255, 255))
                x_off = 0
                for im in mol_images:
                    struct_img.paste(im, (x_off, 0))
                    x_off += im.width
            else:
                struct_img = mol_images[0]

            qc_struct_images[p_id] = struct_img

            buf = io.BytesIO()
            struct_img.save(buf, format='PNG')
            b64_png = base64.b64encode(buf.getvalue()).decode('utf-8')

            adduct_lines = "<br>".join([
                f"<b>{a['adduct']}</b>: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ])
            iso_badge = (
                f"<div style='color:#b45309; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"⚠️ Isobaric Isomers ({assign['formula']})</div>"
            ) if assign["is_isobaric"] else ""

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:10px; background:#ffffff; "
                f"padding:10px 12px; box-shadow:0 8px 20px rgba(0,0,0,0.18); font-family:sans-serif; "
                f"color:#111827; min-width:220px; max-width:390px; text-align:center;'>"
                f"<div style='font-weight:700; font-size:13px; border-bottom:1px solid #e5e7eb; "
                f"padding-bottom:4px; margin-bottom:4px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s): {assign['display_name']}</div>"
                f"{iso_badge}"
                f"<div style='font-size:11.5px; color:#374151; line-height:1.45; margin-bottom:6px;'>"
                f"{adduct_lines}<br>"
                f"<b>Area:</b> {norm_area} &nbsp;|&nbsp; "
                f"<b>Confidence:</b> <span style='color:#047857; font-weight:700;'>{assign['confidence']:.1f}%</span>"
                f"</div>"
                f"<img src='data:image/png;base64,{b64_png}' "
                f"style='max-width:100%; height:auto; display:block; margin:0 auto; border-radius:4px;'/>"
                f"</div>"
            )
        else:
            dom_str = str(peak_row.get('dominant m/z', 'None'))
            dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
            top_ions_str = "<br>".join(dom_tokens[:4]) if dom_tokens else "N/A"

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:8px; background:#f9fafb; "
                f"padding:8px 12px; box-shadow:0 6px 16px rgba(0,0,0,0.15); font-family:sans-serif; "
                f"color:#1f2937; min-width:190px; text-align:left;'>"
                f"<div style='font-weight:700; font-size:12.5px; border-bottom:1px solid #d1d5db; "
                f"padding-bottom:3px; margin-bottom:5px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned</div>"
                f"<div style='font-size:11px; color:#4b5563; line-height:1.4;'>"
                f"<b>Norm. Area:</b> {norm_area}<br>"
                f"<b>Dominant m/z:</b><br>{top_ions_str}</div>"
                f"</div>"
            )

    chrom_fig = go.Figure()

    # Raw & convolved traces
    chrom_fig.add_trace(go.Scatter(
        x=time_arr, y=raw_signal, mode='lines', name='raw chromatogram',
        line=dict(color='black', width=1.6),
        hovertemplate='<b>Raw Chromatogram</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
    ))
    if convolved is not None:
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=convolved, mode='lines', name='inferred mixture',
            line=dict(color='red', width=1.4, dash='dash'),
            hovertemplate='<b>Inferred Mixture</b><br>RT=%{x:.2f}s<br>Mixture=%{y:.3e}<extra></extra>'
        ))

    # Deconvoluted peak traces — tagged with peak_id in `meta` so the floating hover card triggers anywhere on the peak
    if chrom.peaks is not None:
        for g, _ in chrom.peaks.groupby('peak_id'):
            p_id = int(g)
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            component = chrom.unmixed_chromatograms[:, p_id - 1]
            legend_label = _peak_comparison_label(p_id)
            chrom_fig.add_trace(go.Scatter(
                x=time_arr, y=component, mode='lines', fill='tozeroy',
                line=dict(color=peak_color, width=1.0), opacity=0.55,
                name=legend_label,
                meta=p_id,
                hoverinfo='none'
            ))

    shapes, compact_annotations = [], []

    # Add subtle drop-lines, compact "P1, P2..." badges, and interactive apex hover markers
    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
        apex_y = float(raw_signal[c_idx])
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        shapes.append(dict(
            type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y,
            line=dict(color='dodgerblue', width=1.1, dash='dot'), opacity=0.55
        ))

        # Small, non-intrusive badge right above the peak apex
        compact_annotations.append(dict(
            x=rt_apex, y=apex_y + y_max_plot * 0.035, xref='x', yref='y',
            text=f"<b>P{p_id}</b>",
            showarrow=False,
            font=dict(size=10, color='#111827'),
            bgcolor='rgba(255,255,255,0.88)',
            bordercolor=peak_color,
            borderwidth=1.5,
            borderpad=2
        ))

        # Apex marker so hovering directly on the peak top always triggers that peak's structure card
        chrom_fig.add_trace(go.Scatter(
            x=[rt_apex], y=[apex_y],
            mode='markers',
            marker=dict(size=9, color=peak_color, line=dict(width=1.5, color='white')),
            showlegend=False,
            meta=p_id,
            hoverinfo='none'
        ))

    chrom_fig.update_layout(
        shapes=shapes,
        annotations=compact_annotations,
        title=dict(
            text=(
                f"HPLC Chromatogram Deconvolution & Isobaric Screening (Prominence: {prom_val:.4f}) "
                f"— Hover over any peak to view 2D structure & MS1 details"
            ),
            font=dict(size=15, color='#111827')
        ),
        xaxis=dict(
            title='Retention Time (seconds)',
            range=X_RANGE,
            rangeslider=dict(visible=True, thickness=0.10)
        ),
        yaxis=dict(
            title='Total Ion Count (baseline corrected)',
            range=[0, y_max_plot * 1.18]
        ),
        legend=dict(
            orientation='h',
            x=0.5, y=-0.26,
            xanchor='center', yanchor='top',
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#d1d5db',
            borderwidth=1
        ),
        margin=dict(l=MARGIN_L, r=MARGIN_R, t=MARGIN_T, b=MARGIN_B),
        template='plotly_white',
        width=FIG_WIDTH_PX,
        height=FIG_HEIGHT_PX,
        hovermode='closest'
    )

    # Render interactive HTML with custom floating hover card listener
    plot_div_id = "hplc_interactive_chrom_div"
    fig_html = chrom_fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=plot_div_id)
    cards_json = json.dumps({str(k): v for k, v in hover_cards_by_peak.items()})

    custom_hover_html = f"""
    <div style="position:relative; width:{FIG_WIDTH_PX}px;">
        {fig_html}
        <div id="{plot_div_id}_tooltip" style="display:none; position:fixed; pointer-events:none; z-index:99999; transition: opacity 0.08s ease-in-out;"></div>
    </div>
    <script>
    (function() {{
        var cards = {cards_json};
        var gd = document.getElementById('{plot_div_id}');
        var tooltip = document.getElementById('{plot_div_id}_tooltip');

        function attachHoverEvents() {{
            if (!gd || typeof gd.on !== 'function') {{
                setTimeout(attachHoverEvents, 150);
                return;
            }}
            gd.on('plotly_hover', function(data) {{
                if (!data || !data.points || data.points.length === 0) return;
                var pt = data.points[0];
                var p_id = pt.data.meta;
                if (p_id && cards[String(p_id)]) {{
                    tooltip.innerHTML = cards[String(p_id)];
                    tooltip.style.display = 'block';
                    if (data.event) {{
                        var x = data.event.clientX + 18;
                        var y = data.event.clientY - 30;
                        var rect = tooltip.getBoundingClientRect();
                        if (x + rect.width > window.innerWidth - 15) {{
                            x = data.event.clientX - rect.width - 18;
                        }}
                        if (y + rect.height > window.innerHeight - 15) {{
                            y = window.innerHeight - rect.height - 15;
                        }}
                        if (y < 10) y = 10;
                        tooltip.style.left = x + 'px';
                        tooltip.style.top = y + 'px';
                    }}
                }}
            }});
            gd.on('plotly_unhover', function() {{
                tooltip.style.display = 'none';
            }});
            gd.addEventListener('mousemove', function(e) {{
                if (tooltip.style.display === 'block') {{
                    var x = e.clientX + 18;
                    var y = e.clientY - 30;
                    var rect = tooltip.getBoundingClientRect();
                    if (x + rect.width > window.innerWidth - 15) {{
                        x = e.clientX - rect.width - 18;
                    }}
                    if (y + rect.height > window.innerHeight - 15) {{
                        y = window.innerHeight - rect.height - 15;
                    }}
                    if (y < 10) y = 10;
                    tooltip.style.left = x + 'px';
                    tooltip.style.top = y + 'px';
                }}
            }});
        }}
        attachHoverEvents();
    }})();
    </script>
    """
    display(HTML(custom_hover_html))

    # Cache a static PNG snapshot via kaleido for the Multimodal Vision QC cell (Cell 9).
    # We place the structure cards in a clean top-row banner (above y_max_plot) solely for the
    # static QC snapshot so Gemini's vision model can still inspect the 2D structures without
    # obscuring the peaks!
    try:
        qc_fig = go.Figure(chrom_fig)
        qc_images = []
        qc_annotations = list(compact_annotations)
        banner_xs = [202.0, 226.0, 250.0, 274.0, 298.0]
        for idx_p, (p_id, struct_img) in enumerate(sorted(qc_struct_images.items())):
            bx = banner_xs[idx_p % len(banner_xs)]
            by = y_max_plot * 1.48
            assign = peak_assignments[p_id]
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            qc_annotations.append(dict(
                x=bx, y=by + y_max_plot * 0.04, xref='x', yref='y',
                text=f"<b>Peak {p_id}: {assign['display_name']}</b> ({assign['confidence']:.0f}%)",
                showarrow=False, font=dict(size=9.5, color='#111827'),
                bgcolor='white', bordercolor=peak_color, borderwidth=1.5
            ))
            qc_images.append(dict(
                source=struct_img, xref='x', yref='y',
                x=bx, y=by,
                sizex=18.0, sizey=y_max_plot * 0.32,
                xanchor='center', yanchor='top', layer='above'
            ))
        qc_fig.update_layout(
            yaxis=dict(title='Total Ion Count (baseline corrected)', range=[0, y_max_plot * 1.62]),
            annotations=qc_annotations,
            images=qc_images
        )
        chromatogram_png_bytes = qc_fig.to_image(format='png', scale=2)
        globals().setdefault('_notebook_figure_registry', {})['deconvoluted_chromatogram_with_structures'] = chromatogram_png_bytes
    except Exception as e:
        print(f"⚠️ Could not rasterize chromatogram for vision QC (kaleido missing?): {e}")

    globals()['_last_chromatogram_figure'] = chrom_fig

    # --- 2. Interactive Multi-Peak MS1 Stick-Spectrum Comparison Widget ---
    ms_peak_selector = widgets.SelectMultiple(
        options=[(_peak_comparison_label(p_id), p_id) for p_id in sorted(peak_ms1_spectra.keys())],
        description='Compare Peaks:',
        rows=6,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='600px')
    )

    ms_output_area = widgets.Output()

    def build_ms_comparison_figure(selected_peak_ids):
        """Builds a stacked, zoom-linked Plotly figure overlaying the MS1 stick spectrum of
        every selected peak so isotope envelopes, co-eluting isomers and fine (sub-0.01 Da)
        mass differences can be inspected side-by-side."""
        mpl_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        n = max(len(selected_peak_ids), 1)

        fig_ms = make_subplots(
            rows=n, cols=1, shared_xaxes=True,
            vertical_spacing=min(0.12, 0.6 / n) if n > 1 else 0.0,
            subplot_titles=[_peak_comparison_label(p) for p in selected_peak_ids]
        )

        for row_i, p_id in enumerate(selected_peak_ids, start=1):
            plotly_color_val = to_plotly_color(mpl_colors[(p_id - 1) % len(mpl_colors)])
            spectrum = peak_ms1_spectra.get(p_id, {})

            if spectrum:
                mzs = np.array(sorted(spectrum.keys()))
                intens = np.array([spectrum[m] for m in mzs])
                max_int = intens.max()
                rel = (intens / max_int * 100.0) if max_int > 0 else intens

                stick_x = np.repeat(mzs, 3)
                stick_y = np.zeros(stick_x.shape, dtype=float)
                stick_y[1::3] = rel
                stick_y[2::3] = np.nan

                fig_ms.add_trace(
                    go.Scatter(
                        x=stick_x, y=stick_y, mode='lines',
                        line=dict(color=plotly_color_val, width=1.3),
                        hoverinfo='skip', showlegend=False
                    ),
                    row=row_i, col=1
                )
                fig_ms.add_trace(
                    go.Scatter(
                        x=mzs, y=rel, mode='markers',
                        marker=dict(color=plotly_color_val, size=5),
                        customdata=intens, showlegend=False,
                        hovertemplate=(
                            f"Peak {p_id}<br>m/z=%{{x:.4f}}<br>rel. int.=%{{y:.1f}}%"
                            "<br>abs. int.=%{customdata:.3e}<extra></extra>"
                        )
                    ),
                    row=row_i, col=1
                )

                if p_id in peak_assignments:
                    for adduct in peak_assignments[p_id]['adducts'][:3]:
                        fig_ms.add_vline(
                            x=adduct['calc_mz'], line=dict(color=plotly_color_val, width=1.0, dash='dot'),
                            annotation_text=f"{adduct['adduct']} calc.", annotation_font_size=8,
                            annotation_position="bottom", row=row_i, col=1
                        )
            else:
                fig_ms.add_annotation(
                    text="No MS1 signal captured in this peak's elution window",
                    x=0.5, y=0.5, xref=f'x{row_i} domain', yref=f'y{row_i} domain',
                    showarrow=False, font=dict(color='#9ca3af', size=11)
                )

            fig_ms.update_yaxes(title_text="Rel. Int. (%)", range=[0, 120], row=row_i, col=1)

        fig_ms.update_xaxes(matches='x', title_text="m/z", row=n, col=1)
        fig_ms.update_layout(
            width=FIG_WIDTH_PX,
            height=max(300, 260 * n),
            template='plotly_white',
            title_text="Multi-Peak MS1 Spectra Comparison — Isotope Envelopes, Co-eluting Isomers & Fine Mass Differences",
            margin=dict(t=90, b=50),
            hovermode='closest'
        )
        return fig_ms

    def render_ms_comparison(change=None):
        with ms_output_area:
            clear_output(wait=True)
            selected = list(ms_peak_selector.value)
            if not selected:
                print("Select one or more peaks above to overlay/stack their MS1 spectra for comparison.")
                return
            build_ms_comparison_figure(selected).show()

    ms_peak_selector.observe(render_ms_comparison, names='value')

    display(widgets.HTML(
        "<b>🔬 Multi-Peak MS1 Spectra Comparison</b> — select 2+ peaks below to overlay/stack their "
        "MS1 stick spectra on a shared, zoomable m/z axis and inspect isotope envelopes, co-eluting "
        "isomers, and fine mass differences side-by-side."
    ))
    display(ms_peak_selector)
    display(ms_output_area)

    # Pre-select the first 3 peaks and trigger initial render
    preselected = tuple(p_id for _, p_id in ms_peak_selector.options[:3])
    ms_peak_selector.value = preselected
    if preselected:
        render_ms_comparison()

### 5b · Compact re-render of the interactive chromatogram

- **What it does:** a slimmer variant of the cell above that re-renders the same interactive chromatogram and refreshes the cached PNG snapshot.
- **When to use it:** after changing the prominence threshold or re-assigning peaks, when you only want the plot refreshed rather than the whole pipeline re-run.
- **Shared state:** it reads the same `detected_df` / `peak_assignments` globals, so the two cells never disagree.


In [ ]:
#@title 5. Plot Deconvoluted HPLC Chromatogram with Hover Structure Cards & Multi-Peak MS1 Comparison

import subprocess, sys, io, json, base64
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from rdkit.Chem import Draw
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

if 'google.colab' in sys.modules:
    pio.renderers.default = 'colab'

if 'to_plotly_color' not in globals():
    def to_plotly_color(color_val):
        rgba = mcolors.to_rgba(color_val)
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"

def _peak_comparison_label(p_id):
    if 'peak_assignments' in globals() and p_id in peak_assignments:
        a_info = peak_assignments[p_id]
        iso_short = " [Isobaric Isomers]" if a_info.get("is_isobaric", False) else ""
        adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts']])
        return f"Peak {p_id}: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
    if 'detected_df' in globals():
        dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
        if len(dom_rows):
            dom_str = dom_rows.values[0]
            first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
            return f"Peak {p_id}: Unassigned (Base m/z: {first_ion})"
    return f"Peak {p_id}: Unassigned"

def build_interactive_chromatogram_html(div_id: str = "hplc_interactive_chrom_div") -> str:
    """Builds the interactive Plotly chromatogram HTML with floating RDKit 2D hover cards
    and updates the static Kaleido PNG snapshot in `_notebook_figure_registry`."""
    FIG_WIDTH_PX, FIG_HEIGHT_PX = 1300, 800
    MARGIN_L, MARGIN_R, MARGIN_T, MARGIN_B = 80, 40, 65, 210
    X_RANGE = [190, 310]

    time_arr = chrom.df[chrom.time_col].values
    raw_signal = chrom.df[chrom.int_col].values
    convolved = np.sum(chrom.unmixed_chromatograms, axis=1) if chrom.peaks is not None else None
    y_max_plot = float(np.max(raw_signal))
    prom_val = globals().get('current_prominence_val', 0.05)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    hover_cards_by_peak = {}
    qc_struct_images = {}

    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        norm_area = peak_row.get('normalised area', 'N/A')
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            mol_images = [Draw.MolToImage(m, size=(175, 145)) for m in assign["mols"]]
            if len(mol_images) > 1:
                total_w = sum(im.width for im in mol_images)
                struct_img = Image.new('RGBA', (total_w, mol_images[0].height), (255, 255, 255, 255))
                x_off = 0
                for im in mol_images:
                    struct_img.paste(im, (x_off, 0))
                    x_off += im.width
            else:
                struct_img = mol_images[0]

            qc_struct_images[p_id] = struct_img
            buf = io.BytesIO()
            struct_img.save(buf, format='PNG')
            b64_png = base64.b64encode(buf.getvalue()).decode('utf-8')

            adduct_lines = "<br>".join([
                f"<b>{a['adduct']}</b>: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ])
            iso_badge = (
                f"<div style='color:#b45309; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"⚠️ Isobaric Isomers ({assign['formula']})</div>"
            ) if assign.get("is_isobaric", False) else ""

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:10px; background:#ffffff; "
                f"padding:10px 12px; box-shadow:0 8px 20px rgba(0,0,0,0.18); font-family:sans-serif; "
                f"color:#111827; min-width:220px; max-width:390px; text-align:center;'>"
                f"<div style='font-weight:700; font-size:13px; border-bottom:1px solid #e5e7eb; "
                f"padding-bottom:4px; margin-bottom:4px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s): {assign['display_name']}</div>"
                f"{iso_badge}"
                f"<div style='font-size:11.5px; color:#374151; line-height:1.45; margin-bottom:6px;'>"
                f"{adduct_lines}<br>"
                f"<b>Area:</b> {norm_area} &nbsp;|&nbsp; "
                f"<b>Confidence:</b> <span style='color:#047857; font-weight:700;'>{assign['confidence']:.1f}%</span>"
                f"</div>"
                f"<img src='data:image/png;base64,{b64_png}' "
                f"style='max-width:100%; height:auto; display:block; margin:0 auto; border-radius:4px;'/>"
                f"</div>"
            )
        else:
            dom_str = str(peak_row.get('dominant m/z', 'None'))
            dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
            top_ions_str = "<br>".join(dom_tokens[:4]) if dom_tokens else "N/A"
            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:8px; background:#f9fafb; "
                f"padding:8px 12px; box-shadow:0 6px 16px rgba(0,0,0,0.15); font-family:sans-serif; "
                f"color:#1f2937; min-width:190px; text-align:left;'>"
                f"<div style='font-weight:700; font-size:12.5px; border-bottom:1px solid #d1d5db; "
                f"padding-bottom:3px; margin-bottom:5px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned</div>"
                f"<div style='font-size:11px; color:#4b5563; line-height:1.4;'>"
                f"<b>Norm. Area:</b> {norm_area}<br>"
                f"<b>Dominant m/z:</b><br>{top_ions_str}</div>"
                f"</div>"
            )

    chrom_fig = go.Figure()
    chrom_fig.add_trace(go.Scatter(
        x=time_arr, y=raw_signal, mode='lines', name='raw chromatogram',
        line=dict(color='black', width=1.6)
    ))
    if convolved is not None:
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=convolved, mode='lines', name='inferred mixture',
            line=dict(color='red', width=1.4, dash='dash')
        ))

    if chrom.peaks is not None:
        for g, _ in chrom.peaks.groupby('peak_id'):
            p_id = int(g)
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            component = chrom.unmixed_chromatograms[:, p_id - 1]
            chrom_fig.add_trace(go.Scatter(
                x=time_arr, y=component, mode='lines', fill='tozeroy',
                line=dict(color=peak_color, width=1.0), opacity=0.55,
                name=_peak_comparison_label(p_id), meta=p_id, hoverinfo='none'
            ))

    shapes, compact_annotations = [], []
    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
        apex_y = float(raw_signal[c_idx])
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        shapes.append(dict(
            type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y,
            line=dict(color='dodgerblue', width=1.1, dash='dot'), opacity=0.55
        ))
        label_badge = f"<b>P{p_id}: {peak_assignments[p_id]['display_name']}</b>" if p_id in peak_assignments else f"<b>P{p_id}</b>"
        compact_annotations.append(dict(
            x=rt_apex, y=apex_y + y_max_plot * 0.035, xref='x', yref='y',
            text=label_badge, showarrow=False, font=dict(size=10, color='#111827'),
            bgcolor='rgba(255,255,255,0.88)', bordercolor=peak_color, borderwidth=1.5, borderpad=2
        ))
        chrom_fig.add_trace(go.Scatter(
            x=[rt_apex], y=[apex_y], mode='markers',
            marker=dict(size=9, color=peak_color, line=dict(width=1.5, color='white')),
            showlegend=False, meta=p_id, hoverinfo='none'
        ))

    chrom_fig.update_layout(
        shapes=shapes, annotations=compact_annotations,
        title=dict(text=f"HPLC Chromatogram Deconvolution & Assignments (Prominence: {prom_val:.4f})", font=dict(size=15)),
        xaxis=dict(title='Retention Time (seconds)', range=X_RANGE, rangeslider=dict(visible=True, thickness=0.10)),
        yaxis=dict(title='Total Ion Count (baseline corrected)', range=[0, y_max_plot * 1.18]),
        legend=dict(orientation='h', x=0.5, y=-0.26, xanchor='center', yanchor='top', font=dict(size=10)),
        margin=dict(l=MARGIN_L, r=MARGIN_R, t=MARGIN_T, b=MARGIN_B),
        template='plotly_white', width=FIG_WIDTH_PX, height=FIG_HEIGHT_PX, hovermode='closest'
    )

    globals()['_last_chromatogram_figure'] = chrom_fig
    fig_html = chrom_fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=div_id)
    cards_json = json.dumps({str(k): v for k, v in hover_cards_by_peak.items()})

    return f"""
    <div style="position:relative; width:{FIG_WIDTH_PX}px;">
        {fig_html}
        <div id="{div_id}_tooltip" style="display:none; position:fixed; pointer-events:none; z-index:99999;"></div>
    </div>
    <script>
    (function() {{
        var cards = {cards_json};
        var gd = document.getElementById('{div_id}');
        var tooltip = document.getElementById('{div_id}_tooltip');
        function attachHoverEvents() {{
            if (!gd || typeof gd.on !== 'function') {{ setTimeout(attachHoverEvents, 150); return; }}
            gd.on('plotly_hover', function(data) {{
                if (!data || !data.points || !data.points.length) return;
                var p_id = data.points[0].data.meta;
                if (p_id && cards[String(p_id)]) {{
                    tooltip.innerHTML = cards[String(p_id)];
                    tooltip.style.display = 'block';
                }}
            }});
            gd.on('plotly_unhover', function() {{ tooltip.style.display = 'none'; }});
            gd.addEventListener('mousemove', function(e) {{
                if (tooltip.style.display === 'block') {{
                    tooltip.style.left = (e.clientX + 18) + 'px';
                    tooltip.style.top = (e.clientY - 30) + 'px';
                }}
            }});
        }}
        attachHoverEvents();
    }})();
    </script>
    """

if 'detected_df' in globals() and chrom is not None:
    display(HTML(build_interactive_chromatogram_html("hplc_interactive_chrom_div")))



First, add a new code cell to install the official Google GenAI SDK, the MCP client, and the RDKit MCP server package:

### Install the Gemini SDK and MCP bridge

- **What it does:** installs `google-genai` (native function calling), `mcp` and `rdkit` so Gemini can drive the cheminformatics tools defined below.
- **Why native function calling:** spinning a JSON-RPC MCP subprocess inside Colab is brittle; declaring the tools directly to the SDK gives the same capability with none of the I/O plumbing.


In [ ]:
#@title 1. Install the RDKit MCP Bridge and Server

!uv pip install --system -q google-genai mcp rdkit

Since spinning up a separate JSON-RPC subprocess server inside a transient Colab cell can sometimes fight with Colab's I/O structure, the most robust way to give Gemini "MCP-like" tool-use capabilities in a notebook is to define the tools explicitly using Gemini's native function-calling API (google-genai SDK):

### Step 2 · Self-healing RDKit tools, fragmenter & 2D SVG generator

- **What it does:** defines the deterministic chemistry helpers Gemini can call — SMILES repair (`safe_mol_from_smiles`), exact-mass/adduct maths, an in-silico ESI+ fragmenter and an SVG structure renderer.
- **Self-healing:** unkekulised aromatic nitrogens (missing `[nH]`) are repaired automatically instead of failing the whole tool call.
- **Fragmenter:** enumerates single non-ring bond cleavages plus nucleoside-specific glycosidic/cross-ring pathways, returning `[F+H]+` m/z values that can be checked against real scans.


In [ ]:
#@title Step 2: Self-Healing RDKit Tools, Automated Fragmenter & 2D SVG Generator
import re
import base64
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from rdkit.Chem.Draw import rdMolDraw2D

# 1. Silence noisy C++ stderr logs from RDKit so Colab output stays clean
RDLogger.DisableLog('rdApp.*')

_rendered_svg_registry = {}

def safe_mol_from_smiles(smiles_or_str: str):
    """
    Robust SMILES parser that automatically repairs common LLM kekulization mistakes
    (e.g., missing [nH] in cleaved purines/indoles/imidazoles like 'Nc1ncnc2c1ncn2').
    Returns (mol, canonical_smiles, error_msg).
    """
    s = smiles_or_str.strip()
    if not s:
        return None, None, "Empty input string."

    # Check if the model accidentally passed a Molecular Formula (e.g., 'C10H13N5O4')
    if re.fullmatch(r'([A-Z][a-z]?\d*)+([+-]\d*)?', s) and not any(c in s for c in ['c', 'n', 'o', 's', '=', '#', '(', ')', '[']):
        if any(char.isdigit() for char in s) and not re.search(r'[1-9].*[1-9]', s[:3]):
            return None, None, (
                f"'{s}' is a molecular formula, not a SMILES string. "
                "Use `calculate_formula_mass` for formulas, or pass a valid SMILES (e.g. 'Nc1ncnc2c1nc[nH]2' for adenine)."
            )

    # Attempt 1: Standard RDKit parse
    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol, Chem.CanonSmiles(s), None

    # Attempt 2: Auto-repair unkekulized pyrrole/purine/imidazole nitrogens (missing [nH])
    mol_no_san = Chem.MolFromSmiles(s, sanitize=False)
    if mol_no_san is not None:
        try:
            mol_no_san.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol_no_san)
            for atom in mol_no_san.GetAtoms():
                # Aromatic N with degree 2, 0 formal charge, and 0 explicit H often needs 1 H in 5-membered rings
                if atom.GetAtomicNum() == 7 and atom.GetIsAromatic() and atom.GetDegree() == 2 and atom.GetFormalCharge() == 0 and atom.GetNumExplicitHs() == 0:
                    test_mol = Chem.Mol(mol_no_san)
                    test_mol.GetAtomWithIdx(atom.GetIdx()).SetNumExplicitHs(1)
                    try:
                        Chem.SanitizeMol(test_mol)
                        fixed_smi = Chem.MolToSmiles(test_mol)
                        return test_mol, fixed_smi, None
                    except Exception:
                        continue
        except Exception:
            pass

    return None, None, (
        f"RDKit could not parse or kekulize '{s}'. "
        "Tip: In aromatic 5-membered rings (purines, adenine, histidine, indoles), "
        "explicitly write '[nH]' for the protonated ring nitrogen (e.g., adenine is 'Nc1ncnc2c1nc[nH]2')."
    )


def get_user_candidate_molecules() -> str:
    """Returns the user's input table of candidate molecules, classifications, SMILES, formulas, and exact masses."""
    candidates = get_candidates_from_table() if 'get_candidates_from_table' in globals() else []
    if not candidates:
        return "No candidate molecules found."

    rows = []
    for name, classification, smi in candidates:
        mol, clean_smi, _ = safe_mol_from_smiles(smi)
        if mol:
            em = Descriptors.ExactMolWt(mol)
            formula = rdMolDescriptors.CalcMolFormula(mol)
            rows.append({
                "Name": name,
                "Classification": classification,
                "SMILES": clean_smi,
                "Formula": formula,
                "Exact_Mass": round(em, 4),
                "[M+H]+_mz": round(em + 1.007276, 4)
            })
    return pd.DataFrame(rows).to_markdown(index=False)


def get_detected_peaks_summary() -> str:
    """Returns a full tabular summary of all deconvoluted HPLC-MS peaks, retention times, areas, dominant m/z ions, and assignments."""
    if 'detected_df' not in globals():
        return "Error: detected_df not found in memory. Run the peak detection cell first."

    cols = [c for c in ['peak_id', 'retention_time', 'peak start time', 'peak end time', 'normalised area', 'dominant m/z', 'Assignment', 'Confidence (%)'] if c in detected_df.columns]
    return detected_df[cols].to_markdown(index=False)


def get_peak_details(peak_id: int) -> dict:
    """Retrieves detailed analytical evidence, observed adducts, and raw MS1 ions for a specific peak ID."""
    result = {}
    if 'detected_df' in globals():
        row = detected_df[detected_df['peak_id'] == peak_id]
        if not row.empty:
            r_dict = row.iloc[0].to_dict()
            result["chromatogram_row"] = {k: str(v) for k, v in r_dict.items()}
            # Also query raw spectrum across this peak's start and end window if 'od' is loaded
            rt_start = r_dict.get('peak start time', np.nan)
            rt_end = r_dict.get('peak end time', np.nan)
            if 'od' in globals() and not pd.isna(rt_start) and not pd.isna(rt_end):
                agg = {}
                for spec in od:
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
                        mzs, ints = spec.get_peaks()
                        for m, inten in zip(np.round(mzs, 4), ints):
                            if inten > 1e3:
                                agg[m] = agg.get(m, 0.0) + float(inten)
                if agg:
                    top_ions = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:25]
                    max_i = top_ions[0][1]
                    result["raw_ms1_top_25_ions_in_peak_window"] = [
                        {"mz": float(m), "intensity": round(i, 1), "rel_pct": round((i / max_i) * 100, 2)}
                        for m, i in top_ions
                    ]

    if 'peak_assignments' in globals() and peak_id in peak_assignments:
        assign = peak_assignments[peak_id]
        result["assigned_compound"] = {
            "display_name": assign["display_name"],
            "formula": assign["formula"],
            "confidence": assign["confidence"],
            "evidence": assign["evidence"],
            "adducts_observed": [
                f"{a['adduct']} at {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm, intensity: {a['intensity']:.1e})"
                for a in assign["adducts"]
            ]
        }
    return result if result else {"error": f"Peak {peak_id} not found."}


def check_mz_in_peak(peak_id: int, target_mz: float, tolerance_da: float = 0.05) -> dict:
    """
    Directly searches all raw MS1 scans across a specific peak's retention time window
    to verify whether a specific m/z (e.g. a fragment ion) is present in that peak.
    """
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found in memory."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])
    dom_str = str(row.iloc[0]['dominant m/z'])

    matches = []
    max_scan_base_peak = 1.0
    for spec in od:
        if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
            mzs, ints = spec.get_peaks()
            if len(ints) > 0:
                max_scan_base_peak = max(max_scan_base_peak, float(np.max(ints)))
            mask = (mzs >= target_mz - tolerance_da) & (mzs <= target_mz + tolerance_da)
            if np.any(mask):
                for m, inten in zip(mzs[mask], ints[mask]):
                    matches.append({
                        "rt": round(float(spec.getRT()), 2),
                        "observed_mz": round(float(m), 4),
                        "error_da": round(float(m - target_mz), 4),
                        "ppm_error": round(float((m - target_mz) / target_mz * 1e6), 1),
                        "intensity": round(float(inten), 1)
                    })

    matches = sorted(matches, key=lambda x: x["intensity"], reverse=True)
    return {
        "peak_id": peak_id,
        "rt_window": f"{rt_start:.1f}s - {rt_end:.1f}s",
        "dominant_mz_summary": dom_str,
        "target_mz_searched": target_mz,
        "tolerance_da": tolerance_da,
        "found_in_raw_scans": len(matches) > 0,
        "max_observed_hit": matches[0] if matches else None,
        "top_5_scan_hits": matches[:5]
    }


def generate_in_silico_fragments(parent_smiles: str, check_peak_id: int = 0) -> dict:
    """
    Automatically generates chemically valid, RDKit-sanitized ESI+ fragments for a parent molecule
    (including glycosidic cleavage, neutral losses of H2O/NH3/CO, and single acyclic bond cleavages).
    Optionally checks if each fragment m/z is observed in `check_peak_id`.
    """
    mol, clean_parent, err = safe_mol_from_smiles(parent_smiles)
    if not mol:
        return {"error": err}

    parent_em = Descriptors.ExactMolWt(mol)
    parent_formula = rdMolDescriptors.CalcMolFormula(mol)
    parent_mh = round(parent_em + 1.007276, 4)

    fragments_dict = {}

    def _add_frag(frag_smi, pathway_label, is_intrinsic_cation=False):
        f_mol, f_clean, _ = safe_mol_from_smiles(frag_smi)
        if not f_mol:
            return
        em = Descriptors.ExactMolWt(f_mol)
        if em < 40.0 or em >= parent_em - 1.0:
            return
        form = rdMolDescriptors.CalcMolFormula(f_mol)
        f_charge = Chem.GetFormalCharge(f_mol)
        if is_intrinsic_cation or f_charge > 0:
            mz_val = round(em - 0.00054858, 4)
            ion_type = "[F]+ (oxocarbenium / intrinsic cation)"
        else:
            mz_val = round(em + 1.007276, 4)
            ion_type = "[F+H]+"

        key = (form, round(mz_val, 3))
        if key not in fragments_dict:
            entry = {
                "fragment_name": pathway_label,
                "smiles": f_clean,
                "formula": form,
                "neutral_exact_mass": round(em, 4),
                "ion_type": ion_type,
                "expected_esi_pos_mz": mz_val,
                "neutral_loss_from_parent_Da": round(parent_mh - mz_val, 4)
            }
            if check_peak_id > 0:
                hit_info = check_mz_in_peak(check_peak_id, mz_val, tolerance_da=0.05)
                entry["present_in_peak"] = hit_info.get("found_in_raw_scans", False)
                entry["peak_observation"] = hit_info.get("max_observed_hit", None)
            fragments_dict[key] = entry

    # 1. Check canonical nucleoside / riboside pathways if ribose/deoxyribose is present
    ribose_smarts = Chem.MolFromSmarts("[C@@H,C@H,CH]1O[C@H,C@@H,CH](CO)[C@@H,C@H,CH](O)[C@H,C@@H,CH]1")
    if mol.HasSubstructMatch(ribose_smarts):
        # Classic ribose oxocarbenium ion [C5H7O3]+ and dehydrated derivatives
        _add_frag("OC[C@H]1O[C+]=[C@H][C@@H]1O", "Ribose oxocarbenium fragment [C5H7O3]+", is_intrinsic_cation=True)
        _add_frag("O=CC1=CC(=O)CC1", "Dehydrated ribose fragment (C5H5O2+)", is_intrinsic_cation=False)

    # 2. Cleave all single non-ring bonds using RDKit FragmentOnBonds
    cleavable_bonds = []
    for bond in mol.GetBonds():
        if not bond.IsInRing() and bond.GetBondType() == Chem.BondType.SINGLE:
            cleavable_bonds.append(bond.GetIdx())

    for b_idx in cleavable_bonds:
        try:
            frag_mol = Chem.FragmentOnBonds(mol, [b_idx], addDummies=False)
            frags = Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False)
            for f in frags:
                raw_smi = Chem.MolToSmiles(f)
                _add_frag(raw_smi, "Single-bond cleavage fragment ([F+H]+)")
        except Exception:
            pass

    # 3. Common nucleoside cross-ring and base-loss pathways
    if "ncnc2c1ncn2" in clean_parent or "ncnc2c1nc[nH]2" in clean_parent:
        _add_frag("Nc1ncnc2c1nc[nH]2", "Protonated Adenine [B+H]+ (glycosidic cleavage, -Ribose -132.04 Da)")
        _add_frag("c1ncnc2c1nc[nH]2", "Purine core after NH3 loss from Adenine ([B+H-NH3]+, m/z 120.04)")
        _add_frag("Nc1ncnc2c1ncn2C=C", "0,2X Cross-ring cleavage (Adenine + C2H2O/vinyl remnant)")
        _add_frag("Nc1ncnc2c1ncn2CC=O", "0,2X Cross-ring ribose cleavage (Adenine + C2H3O, [M+H-C3H6O3]+)")
        _add_frag("Nc1ncnc2c1ncn2[C@@H](O)C=O", "0,3X Cross-ring ribose cleavage (Adenine + C2H3O2, [M+H-C3H6O2]+)")

    sorted_frags = sorted(fragments_dict.values(), key=lambda x: x["expected_esi_pos_mz"], reverse=True)
    return {
        "parent_smiles": clean_parent,
        "parent_formula": parent_formula,
        "parent_exact_mass": round(parent_em, 4),
        "parent_[M+H]+_mz": parent_mh,
        "generated_fragments": sorted_frags
    }


def analyze_structure_and_mass(smiles_or_formula: str, target_mz: float = 0.0) -> dict:
    """
    Calculates exact monoisotopic mass, molecular formula, and ESI m/z values ([M]+, [M+H]+, [M+Na]+)
    from EITHER a valid SMILES string OR a molecular formula (e.g., 'C5H5N5').
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles_or_formula)
    if mol is None:
        # Try evaluating as a chemical formula directly so it never crashes
        try:
            em = rdMolDescriptors.CalcExactMolWt(smiles_or_formula.strip())
            return {
                "input_interpreted_as": "Molecular Formula (no 2D topology)",
                "formula": smiles_or_formula.strip(),
                "exact_monoisotopic_mass": round(em, 5),
                "calculated_mz_species": {
                    "intrinsic_or_radical_[M]+": round(em - 0.00055, 4),
                    "protonated_[M+H]+": round(em + 1.007276, 4),
                    "sodiated_[M+Na]+": round(em + 22.989218, 4)
                },
                "note": err
            }
        except Exception:
            return {"error": err}

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    formal_charge = Chem.GetFormalCharge(mol)

    mz_intrinsic = exact_mass - (formal_charge * 0.00054858) if formal_charge != 0 else exact_mass
    mz_mh_plus = exact_mass + 1.007276 if formal_charge == 0 else mz_intrinsic

    res = {
        "canonical_smiles": clean_smi,
        "formula": formula,
        "formal_charge": formal_charge,
        "exact_monoisotopic_mass": round(exact_mass, 5),
        "calculated_mz_species": {
            "intrinsic_or_radical_[M]+": round(exact_mass - 0.00055, 4),
            "protonated_[M+H]+": round(mz_mh_plus, 4),
            "sodiated_[M+Na]+": round(exact_mass + 22.989218, 4) if formal_charge == 0 else "N/A"
        }
    }
    if target_mz > 0:
        diff_da = mz_mh_plus - target_mz
        res["target_mz_comparison"] = {
            "target_mz": target_mz,
            "error_Da": round(diff_da, 4),
            "ppm_error": round((diff_da / target_mz) * 1e6, 1)
        }
    return res


def generate_2d_structure_svg(smiles: str, label: str = "", width: int = 280, height: int = 210) -> dict:
    """
    Uses RDKit to generate a 2D chemical structure SVG from a SMILES string.
    Stores the SVG in memory and returns a short placeholder token (e.g. '{{SVG_1}}')
    that you place in your response where the image should appear.
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if not mol:
        return {"error": err}

    AllChem.Compute2DCoords(mol)
    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addStereoAnnotation = True
    opts.padding = 0.12

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    mh_mz = exact_mass + 1.007276 if Chem.GetFormalCharge(mol) == 0 else exact_mass
    legend_text = f"{label} | {formula} ([M+H]+: {mh_mz:.4f})" if label else f"{formula} (Exact: {exact_mass:.4f})"

    drawer.DrawMolecule(mol, legend=legend_text)
    drawer.FinishDrawing()
    svg_str = drawer.GetDrawingText()

    # Strip XML header so it embeds cleanly inline in HTML
    svg_clean = re.sub(r"<\?xml.*?\?>", "", svg_str, flags=re.DOTALL).strip()

    token_id = f"{{{{SVG_{len(_rendered_svg_registry) + 1}}}}}"
    _rendered_svg_registry[token_id] = (
        f"<div style='display:inline-block; margin:8px; padding:8px; border:1px solid #d1d5db; "
        f"border-radius:8px; background:white; text-align:center; box-shadow: 0 1px 3px rgba(0,0,0,0.08);'>"
        f"{svg_clean}</div>"
    )

    return {
        "status": "success",
        "canonical_smiles_drawn": clean_smi,
        "formula": formula,
        "exact_mass": round(exact_mass, 4),
        "expected_mh_mz": round(mh_mz, 4),
        "svg_placeholder_token": token_id,
        "instruction": f"Insert the exact token {token_id} on its own line in your response where you want this 2D structure displayed. Do NOT write raw SVG or base64."
    }

By providing these specific functions, analytical chemists can ask Gemini complex, multi-step questions directly in the chat interface, such as:

* *"What is the confidence score for the peak containing a pyrimidine ring?" (Gemini will call find_peaks_by_substructure, then get_peak_details).* *italicised text*

*  *"If I spiked this sample with Caffeine (CN1C=NC2=C1C(=O)N(C(=O)N2C)C), what [M+Na]+ m/z would I look for?" (Gemini will call calculate_expected_adduct_mz).*


You can pass these RDKit functions directly to the Gemini API so the model can autonomously invoke them when answering your natural language cheminformatics questions:

### Step 3 · Self-contained synthetic-chemistry LC-MS agent

This single cell is organised into nine sections; everything below runs client-side in Python, and Gemini only ever receives compact, distilled summaries.

- **Section 1 — Markdown & typography:** model replies are converted to HTML (`markdown` library, with a dependency-free fallback), sanitised, and wrapped in a styled container (`line-height 1.6; font-size 15px; padding 14px`) with proper table and code-block styling.
- **Section 2 — Preserved RDKit helpers:** SMILES repair, 2D SVG structure cards (`[[MOL_IMG_N]]` tokens) and the hover-card interactive chromatogram builder are kept unchanged.
- **Section 3 — Deterministic synthetic chemistry:** positive/negative adduct + dimer matrices, a bench-contaminant library (TPPO, DIPEA, Et₃N, DMAP, DCU, DIU, dibutyl phthalate, PEG ladders at Δ44.026), reaction-SMILES parsing with failure-mode mass deltas (hydrolysis, protodeboronation, des-halo, Boc/Fmoc/t-Bu cleavage), natural-abundance isotope envelopes (Cl 3:1, Br 1:1, S), neutral-loss detection and cLogP/TPSA-vs-retention-time sanity checks.
- **Section 4 — `SPECTRAL_STATE` + dashboard:** the full raw dataset stays in a local state object; `scipy.signal.find_peaks` picks the top 6 peaks, averages 3 apex scans, subtracts the pre-peak background, drops ions below 2 % and keeps the top centroids. The Plotly dashboard uses `make_subplots(rows=2, shared_xaxes=True)` — UV/DAD with relative area % on top, toggleable BPC/TIC below — with a configurable `uv_ms_delay_min = 0.08` dead-volume offset and colour-coded reaction markers.
- **Section 5 — External databases:** PubChem/ChEMBL/MassBank lookups are disk-cached (sqlite3) and executed in parallel (`ThreadPoolExecutor`), with responses trimmed to `{id, iupac_name, formula, exact_mass, canonical_smiles, relevance_score}`.
- **Sections 6–7 — Tools:** the existing peak-inspection and assignment tools plus new reaction-monitoring, neutral-loss, contaminant and elution-order tools.
- **Section 8 — Routing & context:** routine turns run on `gemini-2.5-flash` with a minimal thinking budget; structure-elucidation / mechanism questions escalate to `gemini-2.5-pro`; bulky historical tool outputs are pruned from the transcript.
- **Section 9 — UI:** the chat widget, model/routing dropdowns and the live dashboard; `update_lcms_plot(...)` re-renders locally in milliseconds and returns only a short confirmation string plus a `[[PLOTLY_CHROM_N]]` token.


In [ ]:
#@title Step 3 (Complete Self-Contained Script): RDKit + PubChem/ChEMBL/MassBank Tools + Interactive Plotly Replotting


# ==============================================================================
# SECTION 0 — Imports (stdlib-first; every optional dependency degrades safely)
# ==============================================================================

import re
import io
import os
import json
import math
import time
import uuid
import base64
import sqlite3
import hashlib
import functools
import threading
import urllib.request
import urllib.parse
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import pubchempy as pcp
from google.colab import userdata
from google import genai
from google.genai import types
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, HTML
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, Crippen, Draw
from rdkit.Chem.Draw import rdMolDraw2D

try:                                   # client-side peak picking (Feature 7)
    import scipy.signal as _scipy_signal
except Exception:
    _scipy_signal = None

try:                                   # Markdown -> styled HTML (Part 1)
    import markdown as _markdown_lib
except Exception:
    _markdown_lib = None


# ==============================================================================
# SECTION 1 — Native Markdown & styled typography rendering  (Part 1)
# ==============================================================================

MARKDOWN_CONTAINER_STYLE = (
    "line-height: 1.6; font-size: 15px; padding: 14px; color: #111827; background: #ffffff; "
    "border: 1px solid #e5e7eb; border-radius: 10px; "
    "font-family: -apple-system, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;"
)

_MARKDOWN_STYLE_BLOCK = """
<style>
.gemini-md h1, .gemini-md h2, .gemini-md h3, .gemini-md h4 { margin: 10px 0 6px; line-height: 1.3; color: #0f172a; }
.gemini-md p { margin: 8px 0; }
.gemini-md ul, .gemini-md ol { margin: 6px 0 6px 22px; }
.gemini-md li { margin: 3px 0; }
.gemini-md table { border-collapse: collapse; margin: 10px 0; font-size: 13.5px; }
.gemini-md th, .gemini-md td { border: 1px solid #d1d5db; padding: 6px 10px; text-align: left; }
.gemini-md th { background: #f3f4f6; font-weight: 700; }
.gemini-md tr:nth-child(even) td { background: #fafafa; }
.gemini-md code { background: #f3f4f6; border-radius: 4px; padding: 1px 5px; font-size: 13px;
                  font-family: 'SFMono-Regular', Consolas, 'Liberation Mono', monospace; }
.gemini-md pre { background: #0f172a; color: #e2e8f0; padding: 12px 14px; border-radius: 8px;
                 overflow-x: auto; line-height: 1.45; }
.gemini-md pre code { background: transparent; color: inherit; padding: 0; }
.gemini-md blockquote { border-left: 4px solid #93c5fd; margin: 8px 0; padding: 4px 12px; color: #374151; }
</style>
"""

_UNSAFE_TAG_RE = re.compile(r"<\s*/?\s*(script|iframe|object|embed|form|link|meta|base)\b[^>]*>", re.I)
_EVENT_ATTR_RE = re.compile(r"\son\w+\s*=\s*(\"[^\"]*\"|'[^']*'|[^\s>]+)", re.I)
_JS_URL_RE = re.compile(r"(href|src)\s*=\s*([\"']?)\s*javascript:[^\"'>\s]*\2", re.I)


def sanitize_model_html(html_str: str) -> str:
    """Strips <script>/<iframe> tags, inline on* event handlers and javascript: URLs from
    model-authored HTML before it is injected into the notebook DOM."""
    cleaned = _UNSAFE_TAG_RE.sub("", html_str or "")
    cleaned = _EVENT_ATTR_RE.sub("", cleaned)
    cleaned = _JS_URL_RE.sub("", cleaned)
    return cleaned


def _fallback_markdown_to_html(md_text: str) -> str:
    """Minimal, dependency-free Markdown -> HTML conversion (headings, lists, pipe tables,
    bold/italic, inline + fenced code) used when the `markdown` package is unavailable."""
    text = (md_text or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    blocks, in_code = [], False
    code_buffer, list_buffer, list_tag = [], [], "ul"
    table_buffer = []

    def _flush_list():
        if list_buffer:
            items = "".join(f"<li>{item}</li>" for item in list_buffer)
            blocks.append(f"<{list_tag}>{items}</{list_tag}>")
            list_buffer.clear()

    def _inline(chunk: str) -> str:
        chunk = re.sub(r"`([^`]+)`", r"<code>\1</code>", chunk)
        chunk = re.sub(r"\*\*([^*]+)\*\*", r"<strong>\1</strong>", chunk)
        chunk = re.sub(r"(?<!\*)\*([^*\n]+)\*(?!\*)", r"<em>\1</em>", chunk)
        return chunk

    def _flush_table():
        if not table_buffer:
            return
        rows = [[cell.strip() for cell in row.strip("|").split("|")] for row in table_buffer]
        table_buffer.clear()
        has_header = len(rows) > 1 and all(re.fullmatch(r":?-{2,}:?", cell) for cell in rows[1] if cell)
        html = ["<table>"]
        if has_header:
            html.append("<thead><tr>" + "".join(f"<th>{_inline(c)}</th>" for c in rows[0]) + "</tr></thead>")
        html.append("<tbody>")
        for row in (rows[2:] if has_header else rows):
            html.append("<tr>" + "".join(f"<td>{_inline(c)}</td>" for c in row) + "</tr>")
        html.append("</tbody></table>")
        blocks.append("".join(html))

    for line in text.split("\n"):
        if line.strip().startswith("```"):
            if in_code:
                blocks.append("<pre><code>" + "\n".join(code_buffer) + "</code></pre>")
                code_buffer = []
            else:
                _flush_list()
            in_code = not in_code
            continue
        if in_code:
            code_buffer.append(line)
            continue

        stripped = line.strip()
        if stripped.startswith("|") and stripped.endswith("|") and len(stripped) > 1:
            _flush_list()
            table_buffer.append(stripped)
            continue
        _flush_table()

        heading = re.match(r"^(#{1,4})\s+(.*)$", line.strip())
        bullet = re.match(r"^\s*[-*+]\s+(.*)$", line)
        numbered = re.match(r"^\s*\d+[.)]\s+(.*)$", line)
        if heading:
            _flush_list()
            level = len(heading.group(1))
            blocks.append(f"<h{level}>{_inline(heading.group(2))}</h{level}>")
        elif bullet or numbered:
            tag = "ul" if bullet else "ol"
            if list_buffer and tag != list_tag:
                _flush_list()
            list_tag = tag
            list_buffer.append(_inline((bullet or numbered).group(1)))
        elif not line.strip():
            _flush_list()
        else:
            _flush_list()
            blocks.append(f"<p>{_inline(line.strip())}</p>")

    if in_code and code_buffer:
        blocks.append("<pre><code>" + "\n".join(code_buffer) + "</code></pre>")
    _flush_table()
    _flush_list()
    return "".join(blocks)


def markdown_to_styled_html(md_text: str) -> str:
    """Converts Gemini Markdown into sanitized HTML wrapped in a styled container
    (line-height 1.6 / 15px / 14px padding) with proper table and code-block styling."""
    body = None
    if _markdown_lib is not None:
        try:
            body = _markdown_lib.markdown(
                md_text or "", extensions=["tables", "fenced_code", "sane_lists"]
            )
        except Exception:
            body = None
    if body is None:
        body = _fallback_markdown_to_html(md_text or "")
    return (
        f"{_MARKDOWN_STYLE_BLOCK}"
        f"<div class=\"gemini-md\" style=\"{MARKDOWN_CONTAINER_STYLE}\">{sanitize_model_html(body)}</div>"
    )


def display_markdown(md_text: str, styled: bool = True) -> None:
    """Renders a Markdown string in the Colab output cell: styled HTML when available,
    falling back to native `IPython.display.Markdown`."""
    if not (md_text or "").strip():
        return
    if styled:
        try:
            display(HTML(markdown_to_styled_html(md_text)))
            return
        except Exception:
            pass
    display(Markdown(md_text))


# ==============================================================================
# SECTION 2 — Preserved RDKit rendering / parsing helpers and the interactive
#             hover-card chromatogram builder (unchanged public behaviour)
# ==============================================================================

# 1. Silence RDKit C++ warnings
RDLogger.DisableLog('rdApp.*')

# Global dictionary mapping placeholder tokens ('[[MOL_IMG_X]]', '{{SVG_X}}', '[[PLOTLY_CHROM_X]]') to HTML blocks
_rendered_svg_registry = {}

if 'to_plotly_color' not in globals():
    def to_plotly_color(color_val):
        rgba = mcolors.to_rgba(color_val)
        return f"rgb({int(rgba[0]*255)}, {int(rgba[1]*255)}, {int(rgba[2]*255)})"


def _register_mol_svg(mol, label: str = "", width: int = 300, height: int = 220) -> str:
    """
    Renders an RDKit Mol object to a Base64-encoded SVG <img> HTML card,
    stores it in `_rendered_svg_registry` under both `[[MOL_IMG_N]]` and `{{SVG_N}}`,
    and returns the token `[[MOL_IMG_N]]`.
    """
    mol_copy = Chem.Mol(mol)
    AllChem.Compute2DCoords(mol_copy)

    drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
    opts = drawer.drawOptions()
    opts.addStereoAnnotation = True
    opts.padding = 0.12

    exact_mass = Descriptors.ExactMolWt(mol_copy)
    formula = rdMolDescriptors.CalcMolFormula(mol_copy)
    f_charge = Chem.GetFormalCharge(mol_copy)
    mh_mz = exact_mass + 1.007276 if f_charge == 0 else exact_mass - (f_charge * 0.00054858)

    drawer.DrawMolecule(mol_copy)
    drawer.FinishDrawing()
    svg_str = drawer.GetDrawingText()

    b64_svg = base64.b64encode(svg_str.encode('utf-8')).decode('utf-8')
    mol_count = sum(1 for k in _rendered_svg_registry if k.startswith("[[MOL_IMG_")) + 1

    caption_title = f"<div style='font-weight:700; font-size:12px; color:#111827; margin-bottom:4px;'>{label}</div>" if label else ""
    caption_sub = f"<div style='font-size:11px; color:#374151; margin-top:4px;'><b>{formula}</b> | Exact: {exact_mass:.4f} | m/z: {mh_mz:.4f}</div>"

    html_card = (
        f"<div style='display:inline-block; margin:10px 6px; padding:10px 14px; border:2px solid #cbd5e1; "
        f"border-radius:10px; background:#ffffff; text-align:center; box-shadow:0 2px 6px rgba(0,0,0,0.15);'>"
        f"{caption_title}"
        f"<img src='data:image/svg+xml;base64,{b64_svg}' width='{width}' height='{height}' style='display:block; margin:0 auto; background:#ffffff;'/>"
        f"{caption_sub}"
        f"</div>"
    )

    token_primary = f"[[MOL_IMG_{mol_count}]]"
    token_alt = f"{{{{SVG_{mol_count}}}}}"
    _rendered_svg_registry[token_primary] = html_card
    _rendered_svg_registry[token_alt] = html_card
    return token_primary


def safe_mol_from_smiles(smiles_or_str: str):
    """Auto-repairs unkekulized pyrrole/purine/imidazole nitrogens (missing [nH])."""
    s = smiles_or_str.strip()
    if not s:
        return None, None, "Empty input string."

    if re.fullmatch(r'([A-Z][a-z]?\d*)+([+-]\d*)?', s) and not any(c in s for c in ['c', 'n', 'o', 's', '=', '#', '(', ')', '[']):
        if any(char.isdigit() for char in s) and not re.search(r'[1-9].*[1-9]', s[:3]):
            return None, None, f"'{s}' is a molecular formula, not a SMILES string."

    mol = Chem.MolFromSmiles(s)
    if mol is not None:
        return mol, Chem.CanonSmiles(s), None

    mol_no_san = Chem.MolFromSmiles(s, sanitize=False)
    if mol_no_san is not None:
        try:
            mol_no_san.UpdatePropertyCache(strict=False)
            Chem.GetSymmSSSR(mol_no_san)
            for atom in mol_no_san.GetAtoms():
                if atom.GetAtomicNum() == 7 and atom.GetIsAromatic() and atom.GetDegree() == 2 and atom.GetFormalCharge() == 0 and atom.GetNumExplicitHs() == 0:
                    test_mol = Chem.Mol(mol_no_san)
                    test_mol.GetAtomWithIdx(atom.GetIdx()).SetNumExplicitHs(1)
                    try:
                        Chem.SanitizeMol(test_mol)
                        return test_mol, Chem.MolToSmiles(test_mol), None
                    except Exception:
                        continue
        except Exception:
            pass

    return None, None, f"RDKit could not parse '{s}'. Ensure aromatic 5-membered ring NH groups are written as '[nH]'."


# --- SELF-CONTAINED INTERACTIVE PLOTLY CHROMATOGRAM BUILDER ---

def _peak_label_for_legend(p_id: int) -> str:
    if 'peak_assignments' in globals() and p_id in peak_assignments:
        a_info = peak_assignments[p_id]
        iso_short = " [Isobaric Isomers]" if a_info.get("is_isobaric", False) else ""
        adduct_summary = ", ".join([f"{x['adduct']} ({x['obs_mz']:.2f})" for x in a_info['adducts']])
        return f"Peak {p_id}: {a_info['display_name']}{iso_short} [{adduct_summary}] (Conf: {a_info['confidence']:.0f}%)"
    if 'detected_df' in globals():
        dom_rows = detected_df.loc[detected_df['peak_id'] == p_id, 'dominant m/z']
        if len(dom_rows):
            dom_str = dom_rows.values[0]
            first_ion = str(dom_str).split(',')[0] if dom_str != "None" else "N/A"
            return f"Peak {p_id}: Unassigned (Base m/z: {first_ion})"
    return f"Peak {p_id}: Unassigned"


def build_interactive_chromatogram_html(div_id: str = "hplc_interactive_chrom_div") -> str:
    """
    Rebuilds the interactive Plotly chromatogram HTML with floating RDKit 2D structure hover cards
    from the live `detected_df` and `peak_assignments` dictionaries, and updates `_notebook_figure_registry`.
    """
    FIG_WIDTH_PX, FIG_HEIGHT_PX = 1220, 760
    MARGIN_L, MARGIN_R, MARGIN_T, MARGIN_B = 80, 40, 65, 210
    X_RANGE = [190, 310]

    time_arr = chrom.df[chrom.time_col].values
    raw_signal = chrom.df[chrom.int_col].values
    convolved = np.sum(chrom.unmixed_chromatograms, axis=1) if chrom.peaks is not None else None
    y_max_plot = float(np.max(raw_signal))
    prom_val = globals().get('current_prominence_val', 0.05)
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    hover_cards_by_peak = {}
    qc_struct_images = {}

    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        norm_area = peak_row.get('normalised area', 'N/A')
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        if 'peak_assignments' in globals() and p_id in peak_assignments:
            assign = peak_assignments[p_id]
            mol_images = [Draw.MolToImage(m, size=(175, 145)) for m in assign["mols"]]
            if len(mol_images) > 1:
                total_w = sum(im.width for im in mol_images)
                struct_img = Image.new('RGBA', (total_w, mol_images[0].height), (255, 255, 255, 255))
                x_off = 0
                for im in mol_images:
                    struct_img.paste(im, (x_off, 0))
                    x_off += im.width
            else:
                struct_img = mol_images[0]

            qc_struct_images[p_id] = struct_img
            buf = io.BytesIO()
            struct_img.save(buf, format='PNG')
            b64_png = base64.b64encode(buf.getvalue()).decode('utf-8')

            adduct_lines = "<br>".join([
                f"<b>{a['adduct']}</b>: {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ])
            iso_badge = (
                f"<div style='color:#b45309; font-weight:600; font-size:11px; margin-top:2px;'>"
                f"⚠️ Isobaric Isomers ({assign['formula']})</div>"
            ) if assign.get("is_isobaric", False) else ""

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:10px; background:#ffffff; "
                f"padding:10px 12px; box-shadow:0 8px 20px rgba(0,0,0,0.18); font-family:sans-serif; "
                f"color:#111827; min-width:220px; max-width:390px; text-align:center;'>"
                f"<div style='font-weight:700; font-size:13px; border-bottom:1px solid #e5e7eb; "
                f"padding-bottom:4px; margin-bottom:4px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s): {assign['display_name']}</div>"
                f"{iso_badge}"
                f"<div style='font-size:11.5px; color:#374151; line-height:1.45; margin-bottom:6px;'>"
                f"{adduct_lines}<br>"
                f"<b>Area:</b> {norm_area} &nbsp;|&nbsp; "
                f"<b>Confidence:</b> <span style='color:#047857; font-weight:700;'>{assign['confidence']:.1f}%</span>"
                f"</div>"
                f"<img src='data:image/png;base64,{b64_png}' "
                f"style='max-width:100%; height:auto; display:block; margin:0 auto; border-radius:4px;'/>"
                f"</div>"
            )
        else:
            dom_str = str(peak_row.get('dominant m/z', 'None'))
            dom_tokens = [x.strip() for x in dom_str.split(',') if x.strip() and x.strip() != 'None']
            top_ions_str = "<br>".join(dom_tokens[:4]) if dom_tokens else "N/A"

            hover_cards_by_peak[p_id] = (
                f"<div style='border:2px solid {peak_color}; border-radius:8px; background:#f9fafb; "
                f"padding:8px 12px; box-shadow:0 6px 16px rgba(0,0,0,0.15); font-family:sans-serif; "
                f"color:#1f2937; min-width:190px; text-align:left;'>"
                f"<div style='font-weight:700; font-size:12.5px; border-bottom:1px solid #d1d5db; "
                f"padding-bottom:3px; margin-bottom:5px;'>"
                f"Peak {p_id} ({rt_apex:.1f}s) — Unassigned</div>"
                f"<div style='font-size:11px; color:#4b5563; line-height:1.4;'>"
                f"<b>Norm. Area:</b> {norm_area}<br>"
                f"<b>Dominant m/z:</b><br>{top_ions_str}</div>"
                f"</div>"
            )

    chrom_fig = go.Figure()
    chrom_fig.add_trace(go.Scatter(
        x=time_arr, y=raw_signal, mode='lines', name='raw chromatogram',
        line=dict(color='black', width=1.6),
        hovertemplate='<b>Raw Chromatogram</b><br>RT=%{x:.2f}s<br>Signal=%{y:.3e}<extra></extra>'
    ))
    if convolved is not None:
        chrom_fig.add_trace(go.Scatter(
            x=time_arr, y=convolved, mode='lines', name='inferred mixture',
            line=dict(color='red', width=1.4, dash='dash'),
            hovertemplate='<b>Inferred Mixture</b><br>RT=%{x:.2f}s<br>Mixture=%{y:.3e}<extra></extra>'
        ))

    if chrom.peaks is not None:
        for g, _ in chrom.peaks.groupby('peak_id'):
            p_id = int(g)
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            component = chrom.unmixed_chromatograms[:, p_id - 1]
            chrom_fig.add_trace(go.Scatter(
                x=time_arr, y=component, mode='lines', fill='tozeroy',
                line=dict(color=peak_color, width=1.0), opacity=0.55,
                name=_peak_label_for_legend(p_id),
                meta=p_id,
                hoverinfo='none'
            ))

    shapes, compact_annotations = [], []
    for _, peak_row in detected_df.iterrows():
        p_id = int(peak_row['peak_id'])
        rt_apex = float(peak_row['retention_time'])
        c_idx = int(np.argmin(np.abs(time_arr - rt_apex)))
        apex_y = float(raw_signal[c_idx])
        peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])

        shapes.append(dict(
            type='line', x0=rt_apex, x1=rt_apex, y0=0, y1=apex_y,
            line=dict(color='dodgerblue', width=1.1, dash='dot'), opacity=0.55
        ))

        badge_text = (
            f"<b>P{p_id}: {peak_assignments[p_id]['display_name']}</b>"
            if ('peak_assignments' in globals() and p_id in peak_assignments)
            else f"<b>P{p_id}</b>"
        )
        compact_annotations.append(dict(
            x=rt_apex, y=apex_y + y_max_plot * 0.035, xref='x', yref='y',
            text=badge_text,
            showarrow=False,
            font=dict(size=10, color='#111827'),
            bgcolor='rgba(255,255,255,0.88)',
            bordercolor=peak_color,
            borderwidth=1.5,
            borderpad=2
        ))

        chrom_fig.add_trace(go.Scatter(
            x=[rt_apex], y=[apex_y],
            mode='markers',
            marker=dict(size=9, color=peak_color, line=dict(width=1.5, color='white')),
            showlegend=False,
            meta=p_id,
            hoverinfo='none'
        ))

    chrom_fig.update_layout(
        shapes=shapes,
        annotations=compact_annotations,
        title=dict(
            text=(
                f"HPLC Chromatogram Deconvolution & Peak Assignments (Prominence: {prom_val:.4f}) "
                f"— Hover over any peak to view 2D structure & MS1 details"
            ),
            font=dict(size=14.5, color='#111827')
        ),
        xaxis=dict(
            title='Retention Time (seconds)',
            range=X_RANGE,
            rangeslider=dict(visible=True, thickness=0.10)
        ),
        yaxis=dict(
            title='Total Ion Count (baseline corrected)',
            range=[0, y_max_plot * 1.18]
        ),
        legend=dict(
            orientation='h',
            x=0.5, y=-0.26,
            xanchor='center', yanchor='top',
            font=dict(size=10),
            bgcolor='rgba(255,255,255,0.9)',
            bordercolor='#d1d5db',
            borderwidth=1
        ),
        margin=dict(l=MARGIN_L, r=MARGIN_R, t=MARGIN_T, b=MARGIN_B),
        template='plotly_white',
        width=FIG_WIDTH_PX,
        height=FIG_HEIGHT_PX,
        hovermode='closest'
    )

    globals()['_last_chromatogram_figure'] = chrom_fig

    # Also update the static Kaleido PNG snapshot in `_notebook_figure_registry` for Step 4 Vision QC
    try:
        qc_fig = go.Figure(chrom_fig)
        qc_images = []
        qc_annotations = list(compact_annotations)
        banner_xs = np.linspace(200.0, 300.0, max(len(qc_struct_images), 1))
        for idx_p, (p_id, struct_img) in enumerate(sorted(qc_struct_images.items())):
            bx = float(banner_xs[idx_p])
            by = y_max_plot * 1.48
            assign = peak_assignments[p_id]
            peak_color = to_plotly_color(colors[(p_id - 1) % len(colors)])
            qc_annotations.append(dict(
                x=bx, y=by + y_max_plot * 0.04, xref='x', yref='y',
                text=f"<b>Peak {p_id}: {assign['display_name']}</b> ({assign['confidence']:.0f}%)",
                showarrow=False, font=dict(size=9.5, color='#111827'),
                bgcolor='white', bordercolor=peak_color, borderwidth=1.5
            ))
            qc_images.append(dict(
                source=struct_img, xref='x', yref='y',
                x=bx, y=by,
                sizex=18.0, sizey=y_max_plot * 0.32,
                xanchor='center', yanchor='top', layer='above'
            ))
        qc_fig.update_layout(
            yaxis=dict(title='Total Ion Count (baseline corrected)', range=[0, y_max_plot * 1.62]),
            annotations=qc_annotations,
            images=qc_images
        )
        globals().setdefault('_notebook_figure_registry', {})['deconvoluted_chromatogram_with_structures'] = qc_fig.to_image(format='png', scale=2)
    except Exception:
        pass

    fig_html = chrom_fig.to_html(include_plotlyjs='cdn', full_html=False, div_id=div_id)
    cards_json = json.dumps({str(k): v for k, v in hover_cards_by_peak.items()})

    return f"""
    <div style="position:relative; width:{FIG_WIDTH_PX}px; margin: 12px 0;">
        {fig_html}
        <div id="{div_id}_tooltip" style="display:none; position:fixed; pointer-events:none; z-index:99999; transition: opacity 0.08s ease-in-out;"></div>
    </div>
    <script>
    (function() {{
        var cards = {cards_json};
        var gd = document.getElementById('{div_id}');
        var tooltip = document.getElementById('{div_id}_tooltip');

        function attachHoverEvents() {{
            if (!gd || typeof gd.on !== 'function') {{
                setTimeout(attachHoverEvents, 150);
                return;
            }}
            gd.on('plotly_hover', function(data) {{
                if (!data || !data.points || data.points.length === 0) return;
                var pt = data.points[0];
                var p_id = pt.data.meta;
                if (p_id && cards[String(p_id)]) {{
                    tooltip.innerHTML = cards[String(p_id)];
                    tooltip.style.display = 'block';
                    if (data.event) {{
                        var x = data.event.clientX + 18;
                        var y = data.event.clientY - 30;
                        var rect = tooltip.getBoundingClientRect();
                        if (x + rect.width > window.innerWidth - 15) x = data.event.clientX - rect.width - 18;
                        if (y + rect.height > window.innerHeight - 15) y = window.innerHeight - rect.height - 15;
                        if (y < 10) y = 10;
                        tooltip.style.left = x + 'px';
                        tooltip.style.top = y + 'px';
                    }}
                }}
            }});
            gd.on('plotly_unhover', function() {{
                tooltip.style.display = 'none';
            }});
            gd.addEventListener('mousemove', function(e) {{
                if (tooltip.style.display === 'block') {{
                    var x = e.clientX + 18;
                    var y = e.clientY - 30;
                    var rect = tooltip.getBoundingClientRect();
                    if (x + rect.width > window.innerWidth - 15) x = e.clientX - rect.width - 18;
                    if (y + rect.height > window.innerHeight - 15) y = window.innerHeight - rect.height - 15;
                    if (y < 10) y = 10;
                    tooltip.style.left = x + 'px';
                    tooltip.style.top = y + 'px';
                }}
            }});
        }}
        attachHoverEvents();
    }})();
    </script>
    """


# ==============================================================================
# SECTION 3 — Deterministic synthetic-chemistry layer (Features 1, 2, 3, 5, 6)
#             Every value below is computed in Python *before* Gemini is called.
# ==============================================================================

PROTON_MASS = 1.007276
ELECTRON_MASS = 0.000548580

# --- Feature 1: adduct / dimer matrix -----------------------------------------
POSITIVE_ADDUCTS = [
    # (label, M multiplier, m/z shift, charge)
    ("[M+H]+",       1,  1.0073, 1),
    ("[M+NH4]+",     1, 18.0338, 1),
    ("[M+Na]+",      1, 22.9892, 1),
    ("[M+K]+",       1, 38.9632, 1),
    ("[M+CH3CN+H]+", 1, 42.0338, 1),
    ("[M+2H]2+",     1,  2.0146, 2),
    ("[2M+H]+",      2,  1.0073, 1),
    ("[2M+Na]+",     2, 22.9892, 1),
]

NEGATIVE_ADDUCTS = [
    ("[M-H]-",        1, -1.0073, 1),
    ("[M+Cl]-",       1, 34.9694, 1),
    ("[M+HCOO]-",     1, 44.9982, 1),
    ("[M+CH3COO]-",   1, 59.0139, 1),
    ("[M-2H]2-",      1, -2.0146, 2),
    ("[2M-H]-",       2, -1.0073, 1),
]

# Common synthetic bench contaminants, quoted as the observed ion m/z.
BENCH_CONTAMINANTS = [
    {"name": "TPPO (triphenylphosphine oxide)", "mz": 279.093, "ion": "[M+H]+", "mode": "positive",
     "origin": "Wittig / Mitsunobu / Appel / Staudinger byproduct"},
    {"name": "DIPEA (Hunig's base)", "mz": 130.159, "ion": "[M+H]+", "mode": "positive",
     "origin": "amide coupling / alkylation base"},
    {"name": "Et3N (triethylamine)", "mz": 102.128, "ion": "[M+H]+", "mode": "positive",
     "origin": "base / HPLC mobile-phase modifier"},
    {"name": "DMAP", "mz": 123.092, "ion": "[M+H]+", "mode": "positive",
     "origin": "acylation catalyst"},
    {"name": "DCU (N,N'-dicyclohexylurea)", "mz": 225.196, "ion": "[M+H]+", "mode": "positive",
     "origin": "DCC coupling byproduct"},
    {"name": "DIU (N,N'-diisopropylurea)", "mz": 145.134, "ion": "[M+H]+", "mode": "positive",
     "origin": "DIC coupling byproduct"},
    {"name": "Dibutyl phthalate", "mz": 279.159, "ion": "[M+H]+", "mode": "positive",
     "origin": "plasticiser leached from tubing / vial caps"},
]

PEG_REPEAT_DA = 44.026          # -CH2CH2O- oligomer spacing (PEG-200/400 series)
PEG_ADDUCT_OFFSETS = {"[M+H]+": 19.0178, "[M+NH4]+": 36.0444, "[M+Na]+": 40.9998}

# --- Feature 5: classic in-source neutral losses -------------------------------
# (label, monoisotopic delta, nominal delta quoted by bench chemists)
NEUTRAL_LOSS_LIBRARY = [
    ("H2O", 18.0106, 18.01),
    ("NH3", 17.0265, 17.03),
    ("CO / N2", 27.9949, 28.00),
    ("HCl", 35.9767, 36.46),
    ("CO2", 43.9898, 44.00),
    ("isobutene (t-Bu / Boc in-source)", 56.0626, 56.06),
    ("Boc (-CO2 -isobutene)", 100.0524, 100.05),
]

# --- Feature 2: synthetic failure-mode mass deltas -----------------------------
REACTION_FAILURE_MODES = [
    ("Unreacted starting material", 0.0, "unreacted"),
    ("Hydrolysis (+H2O)", 18.0106, "hydrolysis"),
    ("Protodeboronation (-B(OH)2 + H)", -43.9900, "deboronation"),
    ("Des-bromo (-Br + H)", -77.9105, "dehalogenation"),
    ("Des-chloro (-Cl + H)", -33.9610, "dehalogenation"),
    ("Boc cleavage (-Boc + H)", -100.0524, "deprotection"),
    ("In-source isobutene loss (-C4H8)", -56.0626, "deprotection"),
    ("Fmoc cleavage (-Fmoc + H)", -222.0681, "deprotection"),
    ("t-Bu ester cleavage (-C4H8)", -56.0626, "deprotection"),
]

REACTION_MARKER_COLORS = {
    "starting_material": "#2563eb",
    "product": "#059669",
    "unreacted": "#2563eb",
    "hydrolysis": "#d97706",
    "deprotection": "#7c3aed",
    "dehalogenation": "#dc2626",
    "deboronation": "#db2777",
    "reagent": "#0891b2",
    "other": "#6b7280",
}

# --- Feature 3: natural isotope abundances -------------------------------------
ELEMENT_ISOTOPES = {
    "H":  [(1.00782503, 0.999885), (2.01410178, 0.000115)],
    "B":  [(10.0129370, 0.199), (11.0093055, 0.801)],
    "C":  [(12.0, 0.9893), (13.00335484, 0.0107)],
    "N":  [(14.00307401, 0.99636), (15.00010890, 0.00364)],
    "O":  [(15.99491462, 0.99757), (16.99913170, 0.00038), (17.99915961, 0.00205)],
    "F":  [(18.99840322, 1.0)],
    "Na": [(22.98976928, 1.0)],
    "Si": [(27.97692653, 0.92223), (28.97649470, 0.04685), (29.97377017, 0.03092)],
    "P":  [(30.97376163, 1.0)],
    "S":  [(31.97207100, 0.9499), (32.97145876, 0.0075), (33.96786690, 0.0425), (35.96708076, 0.0001)],
    "Cl": [(34.96885268, 0.7576), (36.96590259, 0.2424)],
    "K":  [(38.96370649, 0.932581), (39.96399817, 0.000117), (40.96182526, 0.067302)],
    "Br": [(78.91833710, 0.5069), (80.91628970, 0.4931)],
    "I":  [(126.90447300, 1.0)],
}

_FORMULA_TOKEN_RE = re.compile(r"([A-Z][a-z]?)(\d*)")


# ------------------------------------------------------------------------------
# Feature 1 — automated adduct, dimer & bench-artifact matrix
# ------------------------------------------------------------------------------

def expected_ion_table(exact_mass: float, modes: tuple = ("positive", "negative")) -> dict:
    """Deterministic adduct/dimer m/z matrix for a neutral monoisotopic mass (no RDKit needed)."""
    table = {}
    if "positive" in modes:
        table["positive_mode"] = [
            {"adduct": label, "mz": round((exact_mass * mult + shift) / z, 4)}
            for label, mult, shift, z in POSITIVE_ADDUCTS
        ]
    if "negative" in modes:
        table["negative_mode"] = [
            {"adduct": label, "mz": round((exact_mass * mult + shift) / z, 4)}
            for label, mult, shift, z in NEGATIVE_ADDUCTS
        ]
    return table


def match_bench_contaminants(mz_values, tol_da: float = 0.02) -> list:
    """Flags common synthetic bench artifacts (TPPO, DIPEA, Et3N, DMAP, DCU, DIU, dibutyl
    phthalate) in a list of observed m/z values (floats or `(mz, rel_intensity)` pairs)."""
    hits = []
    for entry in mz_values or []:
        mz = float(entry[0]) if isinstance(entry, (list, tuple)) else float(entry)
        rel = float(entry[1]) if isinstance(entry, (list, tuple)) and len(entry) > 1 else None
        for contaminant in BENCH_CONTAMINANTS:
            delta = mz - contaminant["mz"]
            if abs(delta) <= tol_da:
                hit = {
                    "observed_mz": round(mz, 4),
                    "contaminant": contaminant["name"],
                    "ion": contaminant["ion"],
                    "delta_mDa": round(delta * 1000.0, 1),
                    "origin": contaminant["origin"],
                }
                if rel is not None:
                    hit["rel_intensity_pct"] = round(rel, 1)
                hits.append(hit)
    return hits


def detect_peg_series(mz_values, tol_da: float = 0.02, min_series: int = 3) -> list:
    """Detects PEG-200/400 oligomer ladders (constant 44.026 Da spacing) in an ion list."""
    masses = sorted({round(float(v[0]) if isinstance(v, (list, tuple)) else float(v), 4)
                     for v in (mz_values or [])})
    used, series = set(), []
    for i, start in enumerate(masses):
        if start in used:
            continue
        ladder, cursor = [start], start
        for candidate in masses[i + 1:]:
            if abs(candidate - (cursor + PEG_REPEAT_DA)) <= tol_da:
                ladder.append(candidate)
                cursor = candidate
        if len(ladder) >= min_series:
            used.update(ladder)
            adduct = "unknown"
            for name, offset in PEG_ADDUCT_OFFSETS.items():
                n_units = (ladder[0] - offset) / PEG_REPEAT_DA
                if abs(n_units - round(n_units)) <= (tol_da / PEG_REPEAT_DA) + 1e-3 and round(n_units) >= 1:
                    adduct = f"{name} (n={int(round(n_units))})"
                    break
            series.append({
                "series": "PEG oligomers (delta m/z = 44.026)",
                "members": ladder,
                "length": len(ladder),
                "likely_adduct": adduct,
            })
    return series


def flag_contaminants(mz_values, tol_da: float = 0.02) -> dict:
    """Combined bench-artifact + PEG ladder screen applied to a peak's ion list."""
    return {
        "bench_artifacts": match_bench_contaminants(mz_values, tol_da=tol_da),
        "polymer_series": detect_peg_series(mz_values, tol_da=tol_da),
    }


def calculate_expected_ions(smiles: str) -> dict:
    """Returns the deterministic positive/negative adduct + dimer m/z matrix, exact mass,
    formula, cLogP and TPSA for a SMILES string (RDKit `Descriptors.ExactMolWt`)."""
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if mol is None:
        return {"error": err}
    exact_mass = Descriptors.ExactMolWt(mol)
    result = {
        "canonical_smiles": clean_smi,
        "formula": rdMolDescriptors.CalcMolFormula(mol),
        "exact_mass": round(exact_mass, 4),
        "cLogP": round(Crippen.MolLogP(mol), 2),
        "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 1),
    }
    result.update(expected_ion_table(exact_mass))
    return result


# ------------------------------------------------------------------------------
# Feature 3 — theoretical isotope envelopes (halogen / sulfur confirmation)
# ------------------------------------------------------------------------------

def parse_molecular_formula(formula: str) -> dict:
    """Parses 'C10H15N3O5', '[C9H8ClNO2]+' or 'C6H5(CH3)2' into an {element: count} dict."""
    text = str(formula or "").strip().replace(" ", "")
    text = re.sub(r"^\[|\][+-]?\d*$", "", text)
    text = re.sub(r"[+-]+\d*$", "", text)
    counts, stack, current = {}, [], {}
    i = 0
    while i < len(text):
        char = text[i]
        if char == "(":
            stack.append(current)
            current = {}
            i += 1
        elif char == ")":
            i += 1
            digits = ""
            while i < len(text) and text[i].isdigit():
                digits += text[i]
                i += 1
            multiplier = int(digits) if digits else 1
            parent = stack.pop() if stack else {}
            for element, n in current.items():
                parent[element] = parent.get(element, 0) + n * multiplier
            current = parent
        else:
            match = _FORMULA_TOKEN_RE.match(text, i)
            if not match or not match.group(1):
                raise ValueError(f"Unparsable molecular formula near '{text[i:]}'")
            element = match.group(1)
            if element not in ELEMENT_ISOTOPES:
                raise ValueError(f"Unsupported element '{element}' in formula '{formula}'")
            current[element] = current.get(element, 0) + (int(match.group(2)) if match.group(2) else 1)
            i = match.end()
    for element, n in current.items():
        counts[element] = counts.get(element, 0) + n
    if not counts:
        raise ValueError(f"No elements parsed from formula '{formula}'")
    return counts


def _convolve_isotope_dists(dist_a: dict, dist_b: dict, max_offset: int = 8) -> dict:
    """Convolves two {nucleon_offset: (probability, mean_mass)} isotope distributions."""
    combined = {}
    for offset_a, (prob_a, mass_a) in dist_a.items():
        for offset_b, (prob_b, mass_b) in dist_b.items():
            offset = offset_a + offset_b
            if offset > max_offset:
                continue
            prob = prob_a * prob_b
            if prob <= 0.0:
                continue
            mass = mass_a + mass_b
            if offset in combined:
                old_prob, old_mass = combined[offset]
                total = old_prob + prob
                combined[offset] = (total, (old_mass * old_prob + mass * prob) / total)
            else:
                combined[offset] = (prob, mass)
    return combined


def isotope_pattern_from_formula(formula: str, adduct_shift: float = PROTON_MASS,
                                 charge: int = 1, max_peaks: int = 5,
                                 min_rel_pct: float = 0.5) -> list:
    """Binomial/polynomial natural-abundance isotope envelope (M, M+1, M+2, M+4 ...) for a
    molecular formula, returned as `[{isotope, mz, rel_intensity}]` normalised to 100%."""
    counts = parse_molecular_formula(formula)
    max_offset = max(2, max_peaks + 2)
    envelope = {0: (1.0, 0.0)}
    for element, n in sorted(counts.items()):
        isotopes = ELEMENT_ISOTOPES[element]
        lightest = isotopes[0][0]
        single = {}
        for mass, abundance in isotopes:
            if abundance <= 0.0:
                continue
            offset = int(round(mass - lightest))
            if offset > max_offset:
                continue
            if offset in single:
                prob, mean = single[offset]
                total = prob + abundance
                single[offset] = (total, (mean * prob + mass * abundance) / total)
            else:
                single[offset] = (abundance, mass)
        element_dist = {0: (1.0, 0.0)}
        for _ in range(n):
            element_dist = _convolve_isotope_dists(element_dist, single, max_offset=max_offset)
        envelope = _convolve_isotope_dists(envelope, element_dist, max_offset=max_offset)

    if not envelope:
        return []
    base_prob = max(prob for prob, _ in envelope.values())
    charge = max(1, abs(int(charge)))
    peaks = []
    for offset in sorted(envelope):
        prob, mass = envelope[offset]
        rel = (prob / base_prob) * 100.0
        if rel < min_rel_pct:
            continue
        peaks.append({
            "isotope": "M" if offset == 0 else f"M+{offset}",
            "mz": round((mass + adduct_shift) / charge, 4),
            "rel_intensity": round(rel, 2),
        })
    return peaks[:max_peaks]


def theoretical_isotope_envelope(smiles_or_formula: str, adduct: str = "[M+H]+",
                                 max_peaks: int = 5) -> dict:
    """Theoretical isotope distribution for a SMILES *or* molecular formula, including a
    halogen/sulfur diagnostic (Cl 3:1, Br 1:1, S ~4.4% M+2)."""
    adduct_lookup = {label: (mult, shift, z) for label, mult, shift, z in POSITIVE_ADDUCTS + NEGATIVE_ADDUCTS}
    mult, shift, charge = adduct_lookup.get(adduct, (1, PROTON_MASS, 1))
    if mult != 1:
        return {"error": f"Isotope overlays are only defined for monomeric adducts, not {adduct}."}

    mol, _clean, _err = safe_mol_from_smiles(smiles_or_formula)
    formula = rdMolDescriptors.CalcMolFormula(mol) if mol is not None else smiles_or_formula
    try:
        peaks = isotope_pattern_from_formula(formula, adduct_shift=shift, charge=charge, max_peaks=max_peaks)
        counts = parse_molecular_formula(formula)
    except (ValueError, TypeError) as exc:
        return {"error": f"Could not build an isotope envelope for '{smiles_or_formula}': {exc}"}

    diagnostics = []
    m2 = next((p["rel_intensity"] for p in peaks if p["isotope"] == "M+2"), 0.0)
    if counts.get("Cl"):
        diagnostics.append(f"{counts['Cl']} Cl -> M+2 approx {m2:.0f}% (1 Cl = 32%, 2 Cl = 65%)")
    if counts.get("Br"):
        diagnostics.append(f"{counts['Br']} Br -> M+2 approx {m2:.0f}% (1 Br = 97%, near 1:1 doublet)")
    if counts.get("S"):
        diagnostics.append(f"{counts['S']} S -> M+2 approx {m2:.0f}% (1 S = 4.4%)")
    if not diagnostics:
        diagnostics.append(f"No Cl/Br/S present; M+2 approx {m2:.1f}% (13C2 only)")

    return {
        "formula": formula,
        "adduct": adduct,
        "isotope_peaks": peaks,
        "halogen_diagnostic": diagnostics,
    }


# ------------------------------------------------------------------------------
# Feature 5 — in-source fragmentation / neutral-loss annotator
# ------------------------------------------------------------------------------

def detect_neutral_losses(ions, tol_da: float = 0.012, top_n: int = 10) -> list:
    """Deterministic pairwise delta-m/z scan across the top-N ions of a spectrum, annotating
    classic neutral losses (H2O, NH3, CO/N2, HCl, CO2, isobutene, Boc)."""
    normalised = []
    for ion in ions or []:
        if isinstance(ion, dict):
            normalised.append((float(ion.get("mz", 0.0)), float(ion.get("rel", ion.get("rel_intensity", 0.0)))))
        elif isinstance(ion, (list, tuple)) and ion:
            normalised.append((float(ion[0]), float(ion[1]) if len(ion) > 1 else 0.0))
        else:
            normalised.append((float(ion), 0.0))
    normalised = sorted(normalised, key=lambda x: x[1], reverse=True)[:top_n]

    annotations = []
    for i, (mz_hi, rel_hi) in enumerate(normalised):
        for mz_lo, rel_lo in normalised[i + 1:]:
            delta = mz_hi - mz_lo
            if delta <= 0:
                continue
            for label, exact_loss, nominal in NEUTRAL_LOSS_LIBRARY:
                if abs(delta - exact_loss) <= tol_da:
                    annotations.append({
                        "precursor_mz": round(mz_hi, 4),
                        "fragment_mz": round(mz_lo, 4),
                        "delta_mz": round(delta, 4),
                        "neutral_loss": label,
                        "nominal_loss": nominal,
                        "fragment_rel_intensity": round(rel_lo, 1),
                        "precursor_rel_intensity": round(rel_hi, 1),
                    })
                    break
    return sorted(annotations, key=lambda a: a["precursor_rel_intensity"], reverse=True)


# ------------------------------------------------------------------------------
# Feature 6 — cLogP / TPSA vs. retention-time sanity check (reverse-phase C18)
# ------------------------------------------------------------------------------

def compute_polarity_descriptors(smiles: str) -> dict:
    """Crippen cLogP + TPSA + exact mass for one SMILES (used for C18 elution-order checks)."""
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if mol is None:
        return {"error": err}
    return {
        "canonical_smiles": clean_smi,
        "formula": rdMolDescriptors.CalcMolFormula(mol),
        "exact_mass": round(Descriptors.ExactMolWt(mol), 4),
        "cLogP": round(Crippen.MolLogP(mol), 2),
        "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 1),
    }


def _rank(values) -> list:
    """Average-tie ranking helper (1 = smallest)."""
    order = sorted(range(len(values)), key=lambda i: values[i])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:
            j += 1
        mean_rank = (i + j) / 2.0 + 1.0
        for k in range(i, j + 1):
            ranks[order[k]] = mean_rank
        i = j + 1
    return ranks


def check_rt_polarity_consistency(entries) -> dict:
    """Spearman rank correlation between cLogP and retention time for a list of
    `{"name", "cLogP", "rt"}` records; on C18 reverse-phase, higher cLogP must elute later."""
    usable = [e for e in (entries or []) if e.get("cLogP") is not None and e.get("rt") is not None]
    if len(usable) < 2:
        return {"verdict": "insufficient_data", "n": len(usable), "spearman_rho": None, "outliers": []}

    logp_ranks = _rank([float(e["cLogP"]) for e in usable])
    rt_ranks = _rank([float(e["rt"]) for e in usable])
    n = len(usable)
    d_squared = sum((a - b) ** 2 for a, b in zip(logp_ranks, rt_ranks))
    rho = 1.0 - (6.0 * d_squared) / (n * (n * n - 1)) if n > 1 else 0.0

    outliers = [
        {"name": e.get("name", "?"), "cLogP": e["cLogP"], "rt": e["rt"],
         "rank_gap": round(lr - rr, 1)}
        for e, lr, rr in zip(usable, logp_ranks, rt_ranks) if abs(lr - rr) >= 2.0
    ]
    if rho >= 0.6:
        verdict = "consistent"
    elif rho >= 0.0:
        verdict = "weakly_consistent"
    else:
        verdict = "inconsistent"
    return {
        "verdict": verdict,
        "n": n,
        "spearman_rho": round(rho, 3),
        "outliers": outliers,
        "note": "C18 reverse-phase: higher cLogP / lower TPSA should elute later.",
    }


# ------------------------------------------------------------------------------
# Feature 2 — reaction monitoring (SM -> product + common byproducts)
# ------------------------------------------------------------------------------

_ROLE_ALIASES = {
    "sm": "starting_material", "startingmaterial": "starting_material",
    "starting material": "starting_material", "substrate": "starting_material",
    "reagent": "reagent", "reagents": "reagent", "catalyst": "reagent", "base": "reagent",
    "product": "product", "target": "product", "targetproduct": "product",
    "target product": "product", "desired": "product",
}


_ROLE_LABEL_RE = re.compile(r"^\s*([A-Za-z][A-Za-z \-_]*)\s*[:=]\s*(.+)$")


def _match_role_label(token: str):
    """Returns `(role, label, smiles)` when a token is written as `role: SMILES`.
    SMILES containing '=' (e.g. `CC(=O)O`) never match because the prefix must be a
    recognised role alias."""
    match = _ROLE_LABEL_RE.match(token or "")
    if not match:
        return None
    key = match.group(1).strip().lower().replace("-", " ").replace("_", " ")
    role = _ROLE_ALIASES.get(key, _ROLE_ALIASES.get(key.replace(" ", "")))
    if not role:
        return None
    return role, match.group(1).strip(), match.group(2)


def parse_reaction_input(user_input: str) -> dict:
    """Parses a single SMILES, a Reaction SMILES (`SM1.SM2>>Product`, `SM>Reagents>Product`)
    or a labelled comma-separated list (`SM: CCO, reagent: ClC(=O)C, product: CCOC(C)=O`)
    into `{"starting_materials": [...], "reagents": [...], "products": [...]}`."""
    text = (user_input or "").strip()
    result = {"starting_materials": [], "reagents": [], "products": [], "input_type": "empty"}
    if not text:
        return result

    def _add(role, label, smiles):
        smiles = smiles.strip()
        if smiles:
            result[role].append({"role": role[:-1] if role.endswith("s") else role,
                                 "label": label or smiles, "smiles": smiles})

    if ">>" in text or re.search(r"[^>]>[^>]", text):
        result["input_type"] = "reaction_smiles"
        parts = text.split(">>") if ">>" in text else text.split(">")
        parts = [p.strip() for p in parts]
        reactants = parts[0] if parts else ""
        agents = parts[1] if len(parts) == 3 else ""
        products = parts[-1] if len(parts) > 1 else ""
        for idx, smi in enumerate([s for s in reactants.split(".") if s.strip()]):
            _add("starting_materials", f"SM{idx + 1}", smi)
        for idx, smi in enumerate([s for s in agents.split(".") if s.strip()]):
            _add("reagents", f"Reagent{idx + 1}", smi)
        for idx, smi in enumerate([s for s in products.split(".") if s.strip()]):
            _add("products", f"Product{idx + 1}", smi)
        return result

    tokens = [t.strip() for t in text.split(",") if t.strip()]
    labelled = {token: _match_role_label(token) for token in tokens}
    if len(tokens) == 1 and labelled[tokens[0]] is None:
        result["input_type"] = "single_smiles"
        _add("products", "Product", tokens[0])
        return result

    result["input_type"] = "component_list"
    unlabelled = []
    for token in tokens:
        parsed_label = labelled[token]
        if parsed_label:
            role, label, smiles = parsed_label
            _add(role + "s" if not role.endswith("s") else role, label, smiles)
            continue
        unlabelled.append(token)

    for idx, token in enumerate(unlabelled):
        if idx == len(unlabelled) - 1 and not result["products"]:
            _add("products", "Product", token)
        else:
            _add("starting_materials", f"SM{idx + 1}", token)
    return result


def reaction_failure_mode_masses(exact_mass: float, label: str = "", role: str = "product") -> list:
    """Deterministic m/z table of classic synthetic failure modes relative to one component:
    unreacted SM, hydrolysis, protodeboronation/des-halo, Boc/Fmoc/t-Bu cleavage."""
    rows = []
    for mode_label, delta, category in REACTION_FAILURE_MODES:
        neutral = exact_mass + delta
        if neutral <= 20.0:
            continue
        rows.append({
            "component": label,
            "component_role": role,
            "failure_mode": mode_label,
            "category": category,
            "delta_Da": round(delta, 4),
            "neutral_mass": round(neutral, 4),
            "[M+H]+": round(neutral + 1.0073, 4),
            "[M+Na]+": round(neutral + 22.9892, 4),
            "[M-H]-": round(neutral - 1.0073, 4),
            "color": REACTION_MARKER_COLORS.get(category, REACTION_MARKER_COLORS["other"]),
        })
    return rows


# ==============================================================================
# SECTION 4 — SPECTRAL_STATE, client-side distillation & Plotly dashboard
#             (Features 4, 7, 8 — Gemini never sees a raw coordinate array)
# ==============================================================================

UV_MS_DELAY_MIN = 0.08           # dead-volume delay between the DAD cell and the MS source
DEFAULT_TOP_CHROM_PEAKS = 6      # Feature 7: top-N chromatographic peaks kept
DEFAULT_MIN_REL_PCT = 2.0        # Feature 7: noise floor (% of base peak)
DEFAULT_TOP_IONS = 10            # Feature 7: centroids retained per peak (local state)
LLM_TOP_IONS = 6                 # Feature 7: centroids actually serialised for Gemini
BACKGROUND_OFFSET_MIN = 0.05     # background scan taken t_onset - 0.05 min
APEX_SCANS_TO_AVERAGE = 3

SPECTRAL_STATE = {
    "hydrated": False,
    "source": "",
    "rt_unit": "s",
    "scans": [],              # [{"rt": float, "mz": array, "int": array}] (MS1 only)
    "times": [],
    "tic": [],
    "bpc": [],
    "uv": None,               # {"times": [...], "signal": [...], "label": "..."}
    "uv_ms_delay_min": UV_MS_DELAY_MIN,
    "chrom_peaks": [],
    "distilled": None,
    "reaction_markers": [],
    "annotations": {},
    "figure": None,
    "ms_scan_figure": None,
    "last_render_ms": 0.0,
}


def _rt_scale(rt_unit: str = None) -> float:
    """Minutes -> chromatogram RT unit conversion factor (60 for seconds, 1 for minutes)."""
    unit = (rt_unit or SPECTRAL_STATE.get("rt_unit") or "s").lower()
    return 60.0 if unit.startswith("s") else 1.0


def map_uv_rt_to_ms_rt(uv_rt: float, delay_min: float = UV_MS_DELAY_MIN, rt_unit: str = "s") -> float:
    """Applies the configurable UV->MS dead-volume delay when mapping a UV apex to its MS scan."""
    return float(uv_rt) + float(delay_min) * _rt_scale(rt_unit)


def relative_area_percent(areas) -> list:
    """Normalises peak areas to % of total (UV areas are far more reliable than TIC for yield)."""
    total = float(sum(abs(float(a)) for a in areas or []))
    if total <= 0:
        return [0.0 for _ in (areas or [])]
    return [round(abs(float(a)) / total * 100.0, 2) for a in areas]


def _window_stats(mz_array, int_array, low: float, high: float):
    """(max, sum) intensity inside [low, high]; vectorised with numpy when available."""
    try:
        import numpy as _np
        mzs = _np.asarray(mz_array)
        ints = _np.asarray(int_array)
        mask = (mzs >= low) & (mzs <= high)
        if not bool(mask.any()):
            return 0.0, 0.0
        selected = ints[mask]
        return float(selected.max()), float(selected.sum())
    except Exception:
        best = total = 0.0
        for mz, intensity in zip(mz_array, int_array):
            if low <= mz <= high:
                total += float(intensity)
                best = max(best, float(intensity))
        return best, total


def _find_peaks_pure(values, prominence: float) -> list:
    """Dependency-free local-maximum + prominence peak picker (scipy fallback)."""
    series = [float(v) for v in values]
    n = len(series)
    indices = []
    for i in range(1, n - 1):
        if series[i] >= series[i - 1] and series[i] > series[i + 1]:
            left_min = series[i]
            j = i - 1
            while j >= 0 and series[j] <= series[i]:
                left_min = min(left_min, series[j])
                j -= 1
            right_min = series[i]
            k = i + 1
            while k < n and series[k] <= series[i]:
                right_min = min(right_min, series[k])
                k += 1
            if series[i] - max(left_min, right_min) >= prominence:
                indices.append(i)
    return indices


def _peak_bounds(values, apex_idx: int, floor_frac: float = 0.05) -> tuple:
    """Walks down both flanks of an apex until the signal turns back up or hits the floor."""
    series = [float(v) for v in values]
    floor = series[apex_idx] * floor_frac
    left = apex_idx
    while left > 0 and series[left - 1] <= series[left] and series[left] > floor:
        left -= 1
    right = apex_idx
    last = len(series) - 1
    while right < last and series[right + 1] <= series[right] and series[right] > floor:
        right += 1
    return left, right


def find_chromatographic_peaks(times, signal, top_n: int = DEFAULT_TOP_CHROM_PEAKS,
                               prominence_frac: float = 0.05) -> list:
    """Client-side peak picking (`scipy.signal.find_peaks` when available) returning the top-N
    peaks by area with apex/onset/offset retention times and relative area %."""
    times = [float(t) for t in (times or [])]
    signal = [float(s) for s in (signal or [])]
    if len(times) < 3 or len(signal) != len(times):
        return []

    span = max(signal) - min(signal)
    if span <= 0:
        return []
    normalised = [(s - min(signal)) / span for s in signal]

    if _scipy_signal is not None:
        apexes = list(_scipy_signal.find_peaks(normalised, prominence=prominence_frac)[0])
    else:
        apexes = _find_peaks_pure(normalised, prominence_frac)
    if not apexes:
        apexes = [max(range(len(signal)), key=lambda i: signal[i])]

    records = []
    for apex in apexes:
        left, right = _peak_bounds(signal, int(apex))
        area = 0.0
        for i in range(left, right):
            area += (times[i + 1] - times[i]) * (signal[i + 1] + signal[i]) / 2.0
        records.append({
            "apex_index": int(apex),
            "rt": round(times[int(apex)], 4),
            "rt_start": round(times[left], 4),
            "rt_end": round(times[right], 4),
            "apex_intensity": float(signal[int(apex)]),
            "area": float(area),
        })

    records = sorted(records, key=lambda r: r["area"], reverse=True)[:max(1, int(top_n))]
    records = sorted(records, key=lambda r: r["rt"])
    for pct, record in zip(relative_area_percent([r["area"] for r in records]), records):
        record["area_pct"] = pct
    for i, record in enumerate(records, start=1):
        record["peak_id"] = i
    return records


def average_scans(spectra) -> dict:
    """Averages a list of `{mz: intensity}` scan dictionaries (apex averaging, Feature 7)."""
    usable = [s for s in (spectra or []) if s]
    if not usable:
        return {}
    summed = {}
    for spectrum in usable:
        for mz, intensity in spectrum.items():
            summed[mz] = summed.get(mz, 0.0) + float(intensity)
    return {mz: value / len(usable) for mz, value in summed.items()}


def subtract_background(spectrum: dict, background: dict, scale: float = 1.0) -> dict:
    """Subtracts a pre-peak background scan (mobile phase / column bleed) from a peak spectrum."""
    cleaned = {}
    for mz, intensity in (spectrum or {}).items():
        residual = float(intensity) - float((background or {}).get(mz, 0.0)) * float(scale)
        if residual > 0:
            cleaned[mz] = residual
    return cleaned


def top_centroids(spectrum: dict, min_rel_pct: float = DEFAULT_MIN_REL_PCT,
                  top_n: int = DEFAULT_TOP_IONS) -> list:
    """Filters a spectrum to the top-N centroids above `min_rel_pct` % of the base peak."""
    if not spectrum:
        return []
    ordered = sorted(spectrum.items(), key=lambda kv: kv[1], reverse=True)
    base = float(ordered[0][1])
    if base <= 0:
        return []
    centroids = []
    for mz, intensity in ordered:
        rel = float(intensity) / base * 100.0
        if rel < min_rel_pct:
            break
        centroids.append({"mz": round(float(mz), 4), "rel": round(rel, 1)})
        if len(centroids) >= top_n:
            break
    return centroids


def extract_xic(scans, target_mz: float, ppm_tolerance: float = 15.0) -> tuple:
    """Extracted-ion chromatogram for one m/z (local slicing only — never sent to the model)."""
    tolerance = float(target_mz) * float(ppm_tolerance) * 1e-6
    low, high = float(target_mz) - tolerance, float(target_mz) + tolerance
    times, intensities = [], []
    for scan in scans or []:
        _best, total = _window_stats(scan["mz"], scan["int"], low, high)
        times.append(float(scan["rt"]))
        intensities.append(total)
    return times, intensities


def _scan_to_dict(scan, decimals: int = 3) -> dict:
    """Bins one raw scan into a `{rounded_mz: intensity}` dictionary."""
    binned = {}
    for mz, intensity in zip(scan["mz"], scan["int"]):
        key = round(float(mz), decimals)
        binned[key] = binned.get(key, 0.0) + float(intensity)
    return binned


def _nearest_scan_index(rt: float, times) -> int:
    """Index of the scan closest to `rt` (empty-safe)."""
    if not len(times):
        return -1
    return min(range(len(times)), key=lambda i: abs(float(times[i]) - float(rt)))


def _locate_uv_trace() -> dict:
    """Finds a UV/DAD trace from notebook globals (`uv_df`/`dad_df`/`uv_trace`) or from the
    non-TIC chromatograms embedded in the pyOpenMS experiment; returns None when absent."""
    preset = SPECTRAL_STATE.get("uv")
    if preset and preset.get("times"):
        return preset

    for name in ("uv_df", "dad_df", "uv_trace"):
        candidate = globals().get(name)
        if isinstance(candidate, dict) and candidate.get("times"):
            return {"times": list(candidate["times"]), "signal": list(candidate["signal"]),
                    "label": candidate.get("label", "UV/DAD")}
        if candidate is not None and hasattr(candidate, "columns"):
            time_col = next((c for c in candidate.columns if str(c).lower() in ("time", "rt", "retention_time")), None)
            signal_col = next((c for c in candidate.columns
                               if str(c).lower() not in ("time", "rt", "retention_time")), None)
            if time_col is not None and signal_col is not None:
                return {"times": [float(v) for v in candidate[time_col].values],
                        "signal": [float(v) for v in candidate[signal_col].values],
                        "label": f"UV/DAD ({signal_col})"}

    experiment = globals().get("od")
    try:
        for chromatogram in (experiment.getChromatograms() if experiment is not None else []):
            name = (chromatogram.getNativeID() or "").lower()
            if "tic" in name or "bpc" in name:
                continue
            rts, intensities = chromatogram.get_peaks()
            if len(rts) > 2:
                return {"times": [float(v) for v in rts], "signal": [float(v) for v in intensities],
                        "label": f"UV/DAD ({chromatogram.getNativeID()})"}
    except Exception:
        pass
    return None


def hydrate_spectral_state(force: bool = False) -> dict:
    """Loads the full raw spectral dataset into the local `SPECTRAL_STATE` object once
    (MS1 scans, TIC, BPC, UV/DAD trace, detected chromatographic peaks)."""
    if SPECTRAL_STATE["hydrated"] and not force:
        return {"status": "cached", "scans": len(SPECTRAL_STATE["scans"]),
                "peaks": len(SPECTRAL_STATE["chrom_peaks"])}

    scans, times, tic, bpc, source = [], [], [], [], ""
    experiment = globals().get("od")
    if experiment is not None:
        try:
            for spectrum in experiment:
                if spectrum.getMSLevel() != 1:
                    continue
                mzs, intensities = spectrum.get_peaks()
                if len(mzs) == 0:
                    continue
                rt = float(spectrum.getRT())
                scans.append({"rt": rt, "mz": mzs, "int": intensities})
                times.append(rt)
                tic.append(float(sum(intensities)) if not hasattr(intensities, "sum") else float(intensities.sum()))
                bpc.append(float(max(intensities)) if not hasattr(intensities, "max") else float(intensities.max()))
            source = "pyOpenMS MSExperiment `od`"
        except Exception as exc:
            source = f"pyOpenMS load failed ({exc})"

    hplc = globals().get("hplc_df")
    if not times and hplc is not None and hasattr(hplc, "columns"):
        times = [float(v) for v in hplc["Time"].values]
        tic = [float(v) for v in hplc["Signal"].values]
        bpc = list(tic)
        source = source or "hplc_df TIC trace"

    SPECTRAL_STATE.update({
        "scans": scans,
        "times": times,
        "tic": tic,
        "bpc": bpc,
        "rt_unit": "s" if (times and max(times) > 90) else "min",
        "source": source or "no spectral data found",
    })
    SPECTRAL_STATE["uv"] = _locate_uv_trace()

    uv = SPECTRAL_STATE["uv"]
    if uv and uv.get("times"):
        SPECTRAL_STATE["chrom_peaks"] = find_chromatographic_peaks(uv["times"], uv["signal"])
    else:
        SPECTRAL_STATE["chrom_peaks"] = find_chromatographic_peaks(times, tic)

    SPECTRAL_STATE["hydrated"] = True
    SPECTRAL_STATE["distilled"] = None
    return {"status": "hydrated", "scans": len(scans), "peaks": len(SPECTRAL_STATE["chrom_peaks"]),
            "rt_unit": SPECTRAL_STATE["rt_unit"], "uv": bool(uv), "source": SPECTRAL_STATE["source"]}


def _candidate_polarity_rows() -> list:
    """cLogP / TPSA / exact mass for every candidate molecule in the user's SMILES table."""
    rows = []
    candidates = get_candidates_from_table() if "get_candidates_from_table" in globals() else []
    for name, classification, smiles in candidates:
        descriptors = compute_polarity_descriptors(smiles)
        if "error" in descriptors:
            continue
        descriptors.update({"name": name, "classification": classification})
        rows.append(descriptors)
    return rows


def distill_peak_summary(top_n: int = DEFAULT_TOP_CHROM_PEAKS, min_rel_pct: float = DEFAULT_MIN_REL_PCT,
                         top_ions: int = DEFAULT_TOP_IONS,
                         background_offset_min: float = BACKGROUND_OFFSET_MIN) -> dict:
    """Feature 7 — client-side peak picking + apex averaging + background subtraction +
    noise filtering, producing the compact structured payload Gemini actually receives."""
    hydrate_spectral_state()
    scans = SPECTRAL_STATE["scans"]
    scan_times = [s["rt"] for s in scans]
    offset = background_offset_min * _rt_scale()

    peaks_payload = []
    for peak in SPECTRAL_STATE["chrom_peaks"]:
        record = {
            "id": peak["peak_id"],
            "rt": peak["rt"],
            "area_pct": peak.get("area_pct", 0.0),
            "ions": [],
            "losses": [],
            "flags": [],
        }
        if scans:
            ms_rt = map_uv_rt_to_ms_rt(peak["rt"], SPECTRAL_STATE["uv_ms_delay_min"],
                                       SPECTRAL_STATE["rt_unit"]) if SPECTRAL_STATE["uv"] else peak["rt"]
            apex_idx = _nearest_scan_index(ms_rt, scan_times)
            half = max(0, APEX_SCANS_TO_AVERAGE // 2)
            window = [_scan_to_dict(scans[i]) for i in
                      range(max(0, apex_idx - half), min(len(scans), apex_idx + half + 1))]
            averaged = average_scans(window)

            background_idx = _nearest_scan_index(peak["rt_start"] - offset, scan_times)
            background = _scan_to_dict(scans[background_idx]) if 0 <= background_idx < len(scans) else {}
            if background_idx == apex_idx:
                background = {}
            cleaned = subtract_background(averaged, background)

            centroids = top_centroids(cleaned, min_rel_pct=min_rel_pct, top_n=top_ions)
            record["ions"] = [[c["mz"], c["rel"]] for c in centroids]
            record["losses"] = [
                f"{loss['precursor_mz']}->{loss['fragment_mz']} (-{loss['delta_mz']}, {loss['neutral_loss']})"
                for loss in detect_neutral_losses(centroids)[:3]
            ]
            contaminants = flag_contaminants([c["mz"] for c in centroids])
            record["flags"] = [h["contaminant"] for h in contaminants["bench_artifacts"]]
            record["flags"] += [s["series"] for s in contaminants["polymer_series"]]
        peaks_payload.append(record)

    polarity_rows = _candidate_polarity_rows()
    payload = {
        "rt_unit": SPECTRAL_STATE["rt_unit"],
        "uv_ms_delay_min": SPECTRAL_STATE["uv_ms_delay_min"],
        "detector": SPECTRAL_STATE["uv"]["label"] if SPECTRAL_STATE["uv"] else "TIC (no UV/DAD trace found)",
        "peaks": peaks_payload,
        "candidates": [
            {"name": row["name"], "formula": row["formula"], "mass": row["exact_mass"],
             "cLogP": row["cLogP"], "TPSA": row["TPSA_A2"]}
            for row in polarity_rows
        ],
    }
    if SPECTRAL_STATE["reaction_markers"]:
        payload["reaction_markers"] = [
            {"label": marker["label"], "mz": marker["mz"], "rt": marker.get("rt")}
            for marker in SPECTRAL_STATE["reaction_markers"]
        ]
    SPECTRAL_STATE["distilled"] = payload
    return payload


def get_distilled_peak_summary(refresh: bool = False, max_ions_per_peak: int = LLM_TOP_IONS) -> str:
    """Compact JSON summary (<300 tokens) of the top chromatographic peaks: retention time,
    relative area %, background-subtracted top centroids, neutral losses, contaminant flags and
    candidate cLogP/TPSA. Use this instead of asking for raw scans."""
    if refresh or not SPECTRAL_STATE.get("distilled"):
        distill_peak_summary()
    payload = json.loads(json.dumps(SPECTRAL_STATE["distilled"]))
    for peak in payload.get("peaks", []):
        peak["ions"] = peak["ions"][:max(1, int(max_ions_per_peak))]
    return json.dumps(payload, separators=(",", ":"))


# ------------------------------------------------------------------------------
# Feature 4 + 8 — dual UV/DAD + BPC/TIC dashboard and the parameterised replot tool
# ------------------------------------------------------------------------------

def build_ms_scan_figure(rt: float, overlay_formula: str = None, adduct: str = "[M+H]+",
                         mz_window: tuple = None) -> go.Figure:
    """Plots the MS scan nearest `rt` as centroid sticks, optionally overlaying the theoretical
    isotope envelope (Feature 3) as semi-transparent hollow bars (opacity 0.45)."""
    hydrate_spectral_state()
    scans = SPECTRAL_STATE["scans"]
    figure = go.Figure()
    if not scans:
        figure.update_layout(title="No MS1 scans available", template="plotly_white", height=320)
        return figure

    index = _nearest_scan_index(rt, [s["rt"] for s in scans])
    scan = scans[index]
    centroids = top_centroids(_scan_to_dict(scan), min_rel_pct=0.5, top_n=60)
    mzs = [c["mz"] for c in centroids]
    rels = [c["rel"] for c in centroids]

    figure.add_trace(go.Bar(
        x=mzs, y=rels, name=f"Experimental scan @ {scan['rt']:.2f} {SPECTRAL_STATE['rt_unit']}",
        marker=dict(color="#1f2937"), width=0.25,
        hovertemplate="m/z %{x:.4f}<br>%{y:.1f}% base peak<extra></extra>",
    ))

    if overlay_formula:
        envelope = theoretical_isotope_envelope(overlay_formula, adduct=adduct)
        if "error" not in envelope and envelope["isotope_peaks"]:
            figure.add_trace(go.Bar(
                x=[p["mz"] for p in envelope["isotope_peaks"]],
                y=[p["rel_intensity"] for p in envelope["isotope_peaks"]],
                name=f"Theoretical {envelope['formula']} {adduct}",
                opacity=0.45, width=0.45,
                marker=dict(color="rgba(220,38,38,0.25)", line=dict(color="#dc2626", width=2)),
                hovertemplate="%{text}<br>m/z %{x:.4f}<br>%{y:.1f}% theoretical<extra></extra>",
                text=[p["isotope"] for p in envelope["isotope_peaks"]],
            ))
            figure.add_annotation(
                x=0.01, y=1.12, xref="paper", yref="paper", showarrow=False,
                text="<b>Isotope QC:</b> " + "; ".join(envelope["halogen_diagnostic"]),
                font=dict(size=11, color="#b91c1c"), align="left",
            )

    if mz_window:
        figure.update_xaxes(range=[float(mz_window[0]), float(mz_window[1])])
    figure.update_layout(
        title=f"MS1 scan @ RT {scan['rt']:.2f} {SPECTRAL_STATE['rt_unit']}",
        xaxis_title="m/z", yaxis_title="Relative intensity (%)",
        template="plotly_white", height=360, barmode="overlay",
        margin=dict(l=70, r=30, t=70, b=50),
        legend=dict(orientation="h", y=-0.25),
    )
    return figure


def build_lcms_dashboard_figure(rt_window=None, extracted_ion_mz=None, ppm_tolerance: float = 15.0,
                                show_tic: bool = True, show_bpc: bool = True,
                                peak_annotations: dict = None) -> go.Figure:
    """Feature 4 — two stacked, x-axis-linked subplots: row 1 UV/DAD with integrated relative
    area % annotations, row 2 toggleable BPC/TIC plus any requested XIC traces and the
    colour-coded reaction-monitoring markers."""
    hydrate_spectral_state()
    unit = SPECTRAL_STATE["rt_unit"]
    uv = SPECTRAL_STATE["uv"]
    figure = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
        subplot_titles=(
            uv["label"] + " — relative area % (preferred for yield estimation)"
            if uv else "UV/DAD unavailable — TIC shown as area proxy",
            "MS: Base Peak Chromatogram / Total Ion Current",
        ),
    )

    if uv and uv.get("times"):
        figure.add_trace(go.Scatter(x=uv["times"], y=uv["signal"], mode="lines",
                                    name=uv["label"], line=dict(color="#111827", width=1.6)),
                         row=1, col=1)
    elif SPECTRAL_STATE["times"]:
        figure.add_trace(go.Scatter(x=SPECTRAL_STATE["times"], y=SPECTRAL_STATE["tic"], mode="lines",
                                    name="TIC (UV proxy)", line=dict(color="#111827", width=1.4)),
                         row=1, col=1)

    for peak in SPECTRAL_STATE["chrom_peaks"]:
        label = (peak_annotations or {}).get(str(peak["peak_id"]), "")
        figure.add_annotation(
            x=peak["rt"], y=peak["apex_intensity"], row=1, col=1,
            text=f"<b>P{peak['peak_id']}</b> {peak.get('area_pct', 0.0):.1f}%"
                 + (f"<br>{label}" if label else ""),
            showarrow=True, arrowhead=0, arrowwidth=1, ax=0, ay=-26,
            font=dict(size=10, color="#111827"), bgcolor="rgba(255,255,255,0.88)",
            bordercolor="#9ca3af", borderwidth=1, borderpad=2,
        )

    if SPECTRAL_STATE["times"]:
        if show_bpc:
            figure.add_trace(go.Scatter(x=SPECTRAL_STATE["times"], y=SPECTRAL_STATE["bpc"], mode="lines",
                                        name="BPC", line=dict(color="#2563eb", width=1.3)),
                             row=2, col=1)
        if show_tic:
            figure.add_trace(go.Scatter(x=SPECTRAL_STATE["times"], y=SPECTRAL_STATE["tic"], mode="lines",
                                        name="TIC", line=dict(color="#9ca3af", width=1.1, dash="dot"),
                                        visible=True if not show_bpc else "legendonly"),
                             row=2, col=1)

    palette = ["#dc2626", "#059669", "#7c3aed", "#d97706", "#0891b2", "#db2777"]
    for i, mz in enumerate(extracted_ion_mz or []):
        times, intensities = extract_xic(SPECTRAL_STATE["scans"], float(mz), ppm_tolerance)
        if not times:
            continue
        figure.add_trace(go.Scatter(x=times, y=intensities, mode="lines",
                                    name=f"XIC {float(mz):.4f} (+/-{ppm_tolerance:.0f} ppm)",
                                    line=dict(color=palette[i % len(palette)], width=1.6)),
                         row=2, col=1)

    for marker in SPECTRAL_STATE["reaction_markers"]:
        if marker.get("rt") is None:
            continue
        figure.add_vline(
            x=marker["rt"], line_width=1.3, line_dash="dash",
            line_color=marker.get("color", "#6b7280"),
            annotation_text=marker["label"], annotation_position="top",
            annotation_font_size=9, annotation_font_color=marker.get("color", "#6b7280"),
            row=2, col=1,
        )
        figure.add_vline(x=marker["rt"], line_width=1.0, line_dash="dash",
                         line_color=marker.get("color", "#6b7280"), row=1, col=1)

    if rt_window and len(rt_window) == 2:
        figure.update_xaxes(range=[float(rt_window[0]), float(rt_window[1])])

    figure.update_yaxes(title_text="UV absorbance (mAU)" if uv else "TIC", row=1, col=1)
    figure.update_yaxes(title_text="Ion current", row=2, col=1)
    figure.update_xaxes(title_text=f"Retention time ({unit})", row=2, col=1)
    figure.update_layout(
        template="plotly_white", height=720, width=1180, hovermode="x unified",
        title=dict(text=f"LC-UV/MS dashboard — UV->MS delay {SPECTRAL_STATE['uv_ms_delay_min']:.2f} min",
                   font=dict(size=14.5)),
        legend=dict(orientation="h", y=-0.14, x=0.5, xanchor="center", font=dict(size=10)),
        margin=dict(l=80, r=40, t=80, b=90),
    )
    return figure


def _register_plot_html(html_block: str) -> str:
    """Stores rendered Plotly HTML under a fresh `[[PLOTLY_CHROM_N]]` token."""
    index = sum(1 for key in _rendered_svg_registry if key.startswith("[[PLOTLY_CHROM_")) + 1
    token = f"[[PLOTLY_CHROM_{index}]]"
    _rendered_svg_registry[token] = html_block
    return token


def update_lcms_plot(
    rt_window: list[float] | None = None,
    extracted_ion_mz: list[float] | None = None,
    ppm_tolerance: float = 15.0,
    select_scan_rt: float | None = None,
    overlay_formula_isotopes: str | None = None,
    peak_annotations: dict[str, str] | None = None
) -> str:
    """Re-renders the interactive LC-UV/MS dashboard locally (zoom window, extracted ion
    chromatograms, selected MS scan, theoretical isotope overlay, peak labels) and returns a
    short confirmation string plus the `[[PLOTLY_CHROM_N]]` token to place in your reply.
    All heavy slicing happens in Python — no coordinate arrays are returned to the model."""
    started = time.perf_counter()
    try:
        dashboard = build_lcms_dashboard_figure(
            rt_window=rt_window, extracted_ion_mz=extracted_ion_mz,
            ppm_tolerance=ppm_tolerance, peak_annotations=peak_annotations,
        )
        SPECTRAL_STATE["figure"] = dashboard
        blocks = [dashboard.to_html(include_plotlyjs="cdn", full_html=False,
                                    div_id=f"lcms_dash_{uuid.uuid4().hex[:6]}")]

        scan_note = ""
        if select_scan_rt is not None or overlay_formula_isotopes:
            scan_rt = select_scan_rt
            if scan_rt is None:
                scan_rt = SPECTRAL_STATE["chrom_peaks"][0]["rt"] if SPECTRAL_STATE["chrom_peaks"] else 0.0
            scan_figure = build_ms_scan_figure(scan_rt, overlay_formula=overlay_formula_isotopes)
            SPECTRAL_STATE["ms_scan_figure"] = scan_figure
            blocks.append(scan_figure.to_html(include_plotlyjs=False, full_html=False,
                                              div_id=f"lcms_scan_{uuid.uuid4().hex[:6]}"))
            scan_note = f" | MS scan @ {float(scan_rt):.2f}"
            if overlay_formula_isotopes:
                scan_note += f" + theoretical {overlay_formula_isotopes} isotopes"

        token = _register_plot_html("<div>" + "".join(blocks) + "</div>")
        elapsed_ms = (time.perf_counter() - started) * 1000.0
        SPECTRAL_STATE["last_render_ms"] = elapsed_ms
        if peak_annotations:
            SPECTRAL_STATE["annotations"].update(peak_annotations)

        summary = (
            f"Plot updated in {elapsed_ms:.0f} ms | window={rt_window or 'full'} | "
            f"XIC traces={len(extracted_ion_mz or [])} (+/-{ppm_tolerance:.0f} ppm)"
            f"{scan_note} | place {token} on its own line to render it."
        )
        return summary
    except Exception as exc:
        return f"Plot update failed: {exc}"


# ==============================================================================
# SECTION 5 — Cached + parallel external database tools (Feature 9)
#             sqlite3 disk cache (stdlib) -> repeated lookups cost ~0 ms, and
#             every response is trimmed to ~6 fields (<100 tokens) per hit.
# ==============================================================================

CHEM_CACHE_DIR = os.environ.get("LCMS_CHEM_CACHE_DIR", os.path.join(os.getcwd(), ".lcms_chem_cache"))
CHEM_CACHE_TTL_SECONDS = 7 * 24 * 3600
MAX_HITS_RETURNED_TO_LLM = 3


class ChemLookupCache:
    """Persistent key/value cache for external chemical database lookups (sqlite3, stdlib only).
    Falls back to a process-local dict if the database cannot be opened."""

    def __init__(self, path: str = None, ttl_seconds: int = CHEM_CACHE_TTL_SECONDS):
        self.ttl = ttl_seconds
        self.hits = 0
        self.misses = 0
        self._lock = threading.Lock()
        self._memory = {}
        self._conn = None
        self.path = path
        try:
            if path is None:
                os.makedirs(CHEM_CACHE_DIR, exist_ok=True)
                path = os.path.join(CHEM_CACHE_DIR, "lookups.sqlite")
            self._conn = sqlite3.connect(path, check_same_thread=False)
            self._conn.execute(
                "CREATE TABLE IF NOT EXISTS lookups (key TEXT PRIMARY KEY, created REAL, payload TEXT)"
            )
            self._conn.commit()
            self.path = path
        except Exception:
            self._conn = None

    @staticmethod
    def make_key(namespace: str, payload) -> str:
        digest = hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode("utf-8")).hexdigest()
        return f"{namespace}:{digest}"

    def get(self, key: str):
        with self._lock:
            entry = self._memory.get(key)
            if entry is None and self._conn is not None:
                try:
                    row = self._conn.execute(
                        "SELECT created, payload FROM lookups WHERE key = ?", (key,)
                    ).fetchone()
                    if row:
                        entry = (float(row[0]), json.loads(row[1]))
                except Exception:
                    entry = None
            if entry is None:
                self.misses += 1
                return None
            created, value = entry
            if self.ttl and (time.time() - created) > self.ttl:
                self.misses += 1
                return None
            self._memory[key] = entry
            self.hits += 1
            return value

    def set(self, key: str, value) -> None:
        with self._lock:
            entry = (time.time(), value)
            self._memory[key] = entry
            if self._conn is not None:
                try:
                    self._conn.execute(
                        "INSERT OR REPLACE INTO lookups (key, created, payload) VALUES (?, ?, ?)",
                        (key, entry[0], json.dumps(value, default=str)),
                    )
                    self._conn.commit()
                except Exception:
                    pass

    def stats(self) -> dict:
        return {"hits": self.hits, "misses": self.misses, "backend": "sqlite" if self._conn else "memory",
                "path": self.path}


CHEM_CACHE = ChemLookupCache()


def disk_cached(namespace: str, cache=None):
    """Decorator adding persistent caching to an external-database fetch function."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            store = cache or CHEM_CACHE
            key = ChemLookupCache.make_key(namespace, {"args": args, "kwargs": kwargs})
            cached_value = store.get(key)
            if cached_value is not None:
                result = dict(cached_value)
                result["cache"] = "hit"
                return result
            result = func(*args, **kwargs)
            if isinstance(result, dict) and "error" not in result:
                store.set(key, result)
            return result
        return wrapper
    return decorator


def relevance_from_ppm(calc_mz: float, observed_mz: float, ppm_span: float = 20.0) -> float:
    """Maps a mass error onto a 0-1 relevance score (1.0 when no observed m/z was supplied)."""
    if not observed_mz or observed_mz <= 0 or not calc_mz:
        return 1.0
    ppm = abs((observed_mz - calc_mz) / calc_mz) * 1e6
    return round(max(0.0, 1.0 - ppm / ppm_span), 3)


def trim_database_hit(record: dict, observed_mh_mz: float = 0.0) -> dict:
    """Truncates a verbose PubChem/ChEMBL record to `{id, iupac_name, formula, exact_mass,
    canonical_smiles, relevance_score}` (~100 tokens max) before it reaches the model."""
    exact_mass = record.get("exact_mass")
    calc_mh = (float(exact_mass) + PROTON_MASS) if exact_mass else 0.0
    return {
        "id": record.get("id"),
        "iupac_name": (record.get("iupac_name") or record.get("name") or "")[:90],
        "formula": record.get("formula"),
        "exact_mass": round(float(exact_mass), 4) if exact_mass else None,
        "canonical_smiles": (record.get("smiles") or "")[:160],
        "relevance_score": relevance_from_ppm(calc_mh, observed_mh_mz),
    }


@disk_cached("pubchem")
def _fetch_pubchem_raw(compound_name: str = "", molecular_formula: str = "",
                       observed_mh_mz: float = 0.0, mass_window_da: float = 0.015,
                       max_results: int = 5) -> dict:
    """Raw (cached) PubChem PUG-REST fetch — returns a small list of identifier records."""
    compounds = []
    if compound_name.strip():
        compounds = pcp.get_compounds(compound_name.strip(), "name")[:max_results]
    elif molecular_formula.strip():
        compounds = pcp.get_compounds(molecular_formula.strip(), "formula")[:max_results]
    elif observed_mh_mz > 0:
        neutral_mass = observed_mh_mz - PROTON_MASS
        low = round(neutral_mass - mass_window_da, 4)
        high = round(neutral_mass + mass_window_da, 4)
        url = (f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/exact_mass/range/"
               f"{low}/{high}/cids/JSON")
        request = urllib.request.Request(url, headers={"User-Agent": "Colab-LCMS-Agent/2.0"})
        with urllib.request.urlopen(request, timeout=10) as response:
            cid_data = json.loads(response.read().decode("utf-8"))
        cids = cid_data.get("IdentifierList", {}).get("CID", [])[:max_results]
        compounds = pcp.get_compounds(cids, "cid") if cids else []

    records = []
    for compound in compounds[:max_results]:
        smiles = getattr(compound, "isomeric_smiles", None) or getattr(compound, "canonical_smiles", None)
        if not smiles:
            continue
        synonyms = (getattr(compound, "synonyms", None) or [])[:1]
        records.append({
            "id": f"CID {compound.cid}",
            "name": synonyms[0] if synonyms else None,
            "iupac_name": getattr(compound, "iupac_name", None),
            "formula": getattr(compound, "molecular_formula", None),
            "smiles": smiles,
        })
    return {"records": records}


@disk_cached("chembl")
def _fetch_chembl_raw(molecular_formula: str = "", inchikey: str = "", max_results: int = 5) -> dict:
    """Raw (cached) ChEMBL REST fetch by full molecular formula or standard InChIKey."""
    if inchikey:
        url = ("https://www.ebi.ac.uk/chembl/api/data/molecule.json"
               f"?molecule_structures__standard_inchi_key={urllib.parse.quote(inchikey)}")
    elif molecular_formula.strip():
        url = ("https://www.ebi.ac.uk/chembl/api/data/molecule.json"
               f"?molecule_properties__full_molformula={urllib.parse.quote(molecular_formula.strip())}"
               f"&limit={max_results}")
    else:
        return {"error": "Provide either `molecular_formula` or `smiles`."}

    request = urllib.request.Request(
        url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/2.0"})
    with urllib.request.urlopen(request, timeout=10) as response:
        data = json.loads(response.read().decode("utf-8"))

    records = []
    for entry in data.get("molecules", [])[:max_results]:
        structures = entry.get("molecule_structures") or {}
        properties = entry.get("molecule_properties") or {}
        if not structures.get("canonical_smiles"):
            continue
        records.append({
            "id": entry.get("molecule_chembl_id"),
            "name": entry.get("pref_name"),
            "iupac_name": entry.get("pref_name"),
            "formula": properties.get("full_molformula"),
            "smiles": structures.get("canonical_smiles"),
            "alogp": properties.get("alogp"),
            "psa": properties.get("psa"),
            "max_phase": entry.get("max_phase"),
        })
    return {"records": records}


@disk_cached("massbank")
def _fetch_massbank_raw(inchikey: str = "", compound_name: str = "", exact_mass: float = 0.0,
                        max_records: int = 3) -> dict:
    """Raw (cached) MassBank Europe fetch returning only the top 10 MS/MS fragments per record."""
    params = {"ion_mode": "POSITIVE", "ms_type": "MS2"}
    if inchikey:
        params["inchi_key"] = inchikey
    elif compound_name.strip():
        params["compound_name"] = compound_name.strip()
    elif exact_mass > 0:
        params["exact_mass"] = f"{exact_mass:.4f}"
        params["mass_tolerance"] = "0.02"
    else:
        return {"error": "Provide a SMILES string, compound name, or exact_mass."}

    search_url = f"https://api.massbank.eu/records/search?{urllib.parse.urlencode(params)}"
    request = urllib.request.Request(
        search_url, headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/2.0"})
    with urllib.request.urlopen(request, timeout=10) as response:
        search_result = json.loads(response.read().decode("utf-8"))

    items = search_result if isinstance(search_result, list) else search_result.get(
        "results", search_result.get("data", []))
    records = []
    for item in items[:max_records]:
        accession = item if isinstance(item, str) else item.get("accession")
        if not accession:
            continue
        safe_accession = urllib.parse.quote(str(accession), safe="")
        record_request = urllib.request.Request(
            f"https://api.massbank.eu/records/{safe_accession}",
            headers={"Accept": "application/json", "User-Agent": "Colab-LCMS-Agent/2.0"})
        with urllib.request.urlopen(record_request, timeout=8) as record_response:
            record_data = json.loads(record_response.read().decode("utf-8"))
        peaks = record_data.get("peak", {}).get("peak", {}).get("values", [])
        top_peaks = sorted(peaks, key=lambda p: p.get("rel", p.get("intensity", 0)), reverse=True)[:10]
        records.append({
            "accession": accession,
            "title": (record_data.get("record_title") or "")[:90],
            "precursor_mz": record_data.get("ms_focused_ion", {}).get("precursor_m/z"),
            "top_ms2_fragments": [
                {"mz": round(float(p.get("mz", 0)), 4), "rel": p.get("rel", p.get("intensity", 0))}
                for p in top_peaks
            ],
        })
    return {"records": records}


def search_pubchem_by_mass_or_formula(observed_mh_mz: float = 0.0, molecular_formula: str = "",
                                      compound_name: str = "",
                                      mass_window_da: float = 0.015,
                                      max_results: int = MAX_HITS_RETURNED_TO_LLM) -> dict:
    """Searches PubChem by name, molecular formula or observed [M+H]+ m/z. Results are disk
    cached and trimmed to {id, iupac_name, formula, exact_mass, canonical_smiles,
    relevance_score} plus a pre-rendered 2D structure `image_token`."""
    try:
        raw = _fetch_pubchem_raw(compound_name=compound_name, molecular_formula=molecular_formula,
                                 observed_mh_mz=observed_mh_mz, mass_window_da=mass_window_da,
                                 max_results=max(max_results, MAX_HITS_RETURNED_TO_LLM))
        if "error" in raw:
            return raw
        hits = []
        for record in raw.get("records", [])[:max_results]:
            mol, clean_smiles, _err = safe_mol_from_smiles(record.get("smiles", ""))
            if not mol:
                continue
            enriched = dict(record)
            enriched["exact_mass"] = Descriptors.ExactMolWt(mol)
            enriched["smiles"] = clean_smiles
            hit = trim_database_hit(enriched, observed_mh_mz)
            hit["image_token"] = _register_mol_svg(
                mol, label=f"{record.get('name') or record.get('id')}")
            hits.append(hit)
        return {"source": "PubChem", "query_mh_mz": observed_mh_mz, "cache": raw.get("cache", "miss"),
                "hits": hits}
    except Exception as exc:
        return {"error": f"PubChem search failed: {exc}"}


def search_chembl_metabolites_and_isomers(molecular_formula: str = "", smiles: str = "",
                                          max_results: int = MAX_HITS_RETURNED_TO_LLM) -> dict:
    """Queries ChEMBL by molecular formula or SMILES (via InChIKey). Cached and trimmed, with
    ChEMBL ALogP/PSA plus RDKit cLogP retained for reverse-phase elution-order reasoning."""
    try:
        inchikey = ""
        if smiles.strip():
            mol, _clean, err = safe_mol_from_smiles(smiles)
            if not mol:
                return {"error": err}
            inchikey = Chem.MolToInchiKey(mol)
        raw = _fetch_chembl_raw(molecular_formula=molecular_formula, inchikey=inchikey,
                                max_results=max(max_results, MAX_HITS_RETURNED_TO_LLM))
        if "error" in raw:
            return raw
        hits = []
        for record in raw.get("records", [])[:max_results]:
            mol, clean_smiles, _err = safe_mol_from_smiles(record.get("smiles", ""))
            if not mol:
                continue
            enriched = dict(record)
            enriched["exact_mass"] = Descriptors.ExactMolWt(mol)
            enriched["smiles"] = clean_smiles
            hit = trim_database_hit(enriched)
            hit.update({
                "chembl_alogp": record.get("alogp"),
                "psa": record.get("psa"),
                "rdkit_cLogP": round(Crippen.MolLogP(mol), 2),
                "image_token": _register_mol_svg(mol, label=f"{record.get('name') or record.get('id')}"),
            })
            hits.append(hit)
        return {"source": "ChEMBL", "cache": raw.get("cache", "miss"), "hits": hits,
                "retention_note": "On C18 reverse-phase, lower cLogP / higher PSA elutes earlier."}
    except Exception as exc:
        return {"error": f"ChEMBL query failed: {exc}"}


def query_massbank_reference_spectrum(compound_name_or_smiles: str = "", exact_mass: float = 0.0,
                                      max_records: int = MAX_HITS_RETURNED_TO_LLM) -> dict:
    """Queries MassBank Europe for experimental ESI+ MS/MS reference spectra (cached; only the
    top 10 fragments per record are returned)."""
    try:
        inchikey, name = "", ""
        mol, _clean, _err = safe_mol_from_smiles(compound_name_or_smiles)
        if mol:
            inchikey = Chem.MolToInchiKey(mol)
        else:
            name = compound_name_or_smiles
        raw = _fetch_massbank_raw(inchikey=inchikey, compound_name=name, exact_mass=exact_mass,
                                  max_records=max_records)
        if "error" in raw:
            return raw
        return {"source": "MassBank", "cache": raw.get("cache", "miss"),
                "reference_spectra": raw.get("records", [])[:max_records]}
    except Exception as exc:
        return {"warning": f"Live MassBank lookup returned: {exc}. "
                           "Use `disambiguate_isobaric_peak_isomers` for deterministic MS1 matching."}


def lookup_compound_everywhere(compound_name: str = "", molecular_formula: str = "",
                               observed_mh_mz: float = 0.0, smiles: str = "",
                               max_results: int = MAX_HITS_RETURNED_TO_LLM) -> dict:
    """Runs the PubChem, ChEMBL and MassBank lookups **in parallel** (ThreadPoolExecutor) and
    returns one bounded, trimmed payload — the fastest way to identify an unknown peak."""
    jobs = {
        "pubchem": lambda: search_pubchem_by_mass_or_formula(
            observed_mh_mz=observed_mh_mz, molecular_formula=molecular_formula,
            compound_name=compound_name, max_results=max_results),
        "chembl": lambda: search_chembl_metabolites_and_isomers(
            molecular_formula=molecular_formula, smiles=smiles, max_results=max_results),
        "massbank": lambda: query_massbank_reference_spectrum(
            compound_name_or_smiles=smiles or compound_name, exact_mass=max(0.0, observed_mh_mz - PROTON_MASS),
            max_records=max_results),
    }
    started = time.perf_counter()
    results = {}
    with ThreadPoolExecutor(max_workers=len(jobs)) as pool:
        futures = {pool.submit(job): name for name, job in jobs.items()}
        for future in as_completed(futures):
            name = futures[future]
            try:
                results[name] = future.result()
            except Exception as exc:
                results[name] = {"error": str(exc)}
    results["elapsed_ms"] = round((time.perf_counter() - started) * 1000.0, 1)
    results["cache_stats"] = CHEM_CACHE.stats()
    return results


# ==============================================================================
# SECTION 6 — Preserved peak-inspection, fragmentation and assignment tools
# ==============================================================================

def get_user_candidate_molecules() -> str:
    """Returns the user's candidate molecules with classifications, SMILES, formulas, exact masses,
    key adduct m/z values and the cLogP / TPSA used for reverse-phase elution-order checks."""
    candidates = get_candidates_from_table() if 'get_candidates_from_table' in globals() else []
    if not candidates:
        return "No candidate molecules found."

    rows = []
    for name, classification, smi in candidates:
        mol, clean_smi, _ = safe_mol_from_smiles(smi)
        if mol:
            em = Descriptors.ExactMolWt(mol)
            formula = rdMolDescriptors.CalcMolFormula(mol)
            rows.append({
                "Name": name,
                "Classification": classification,
                "SMILES": clean_smi,
                "Formula": formula,
                "Exact_Mass": round(em, 4),
                "[M+H]+_mz": round(em + 1.0073, 4),
                "[M+Na]+_mz": round(em + 22.9892, 4),
                "[M-H]-_mz": round(em - 1.0073, 4),
                "cLogP": round(Crippen.MolLogP(mol), 2),
                "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 1)
            })
    return pd.DataFrame(rows).to_markdown(index=False)


def get_detected_peaks_summary() -> str:
    """Returns a full tabular summary of all deconvoluted HPLC-MS peaks, retention times, areas, dominant m/z ions, and assignments."""
    if 'detected_df' not in globals():
        return "Error: detected_df not found in memory."
    cols = [c for c in ['peak_id', 'retention_time', 'peak start time', 'peak end time', 'normalised area', 'dominant m/z', 'Assignment', 'Confidence (%)'] if c in detected_df.columns]
    return detected_df[cols].to_markdown(index=False)


def get_peak_details(peak_id: int) -> dict:
    """Retrieves detailed analytical evidence, observed adducts, and raw MS1 ions for a specific peak ID."""
    result = {}
    if 'detected_df' in globals():
        row = detected_df[detected_df['peak_id'] == peak_id]
        if not row.empty:
            r_dict = row.iloc[0].to_dict()
            result["chromatogram_row"] = {k: str(v) for k, v in r_dict.items()}
            rt_start = r_dict.get('peak start time', np.nan)
            rt_end = r_dict.get('peak end time', np.nan)
            if 'od' in globals() and not pd.isna(rt_start) and not pd.isna(rt_end):
                agg = {}
                for spec in od:
                    if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
                        mzs, ints = spec.get_peaks()
                        for m, inten in zip(np.round(mzs, 4), ints):
                            if inten > 1e3:
                                agg[m] = agg.get(m, 0.0) + float(inten)
                if agg:
                    top_ions = sorted(agg.items(), key=lambda x: x[1], reverse=True)[:25]
                    max_i = top_ions[0][1]
                    result["raw_ms1_top_25_ions_in_peak_window"] = [
                        {"mz": float(m), "intensity": round(i, 1), "rel_pct": round((i / max_i) * 100, 2)}
                        for m, i in top_ions
                    ]

    if 'peak_assignments' in globals() and peak_id in peak_assignments:
        assign = peak_assignments[peak_id]
        result["assigned_compound"] = {
            "display_name": assign["display_name"],
            "formula": assign["formula"],
            "confidence": assign["confidence"],
            "evidence": assign["evidence"],
            "adducts_observed": [
                f"{a['adduct']} at {a['obs_mz']:.4f} m/z ({a['ppm']:+.1f} ppm)"
                for a in assign["adducts"]
            ]
        }
    return result if result else {"error": f"Peak {peak_id} not found."}


def check_mz_in_peak(peak_id: int, target_mz: float, tolerance_da: float = 0.05) -> dict:
    """Checks all raw MS1 scans across a specific peak's retention time window for a target m/z."""
    if 'detected_df' not in globals() or 'od' not in globals():
        return {"error": "detected_df or pyOpenMS 'od' experiment not found in memory."}

    row = detected_df[detected_df['peak_id'] == peak_id]
    if row.empty:
        return {"error": f"Peak {peak_id} not found in detected_df."}

    rt_start = float(row.iloc[0]['peak start time'])
    rt_end = float(row.iloc[0]['peak end time'])

    matches = []
    for spec in od:
        if spec.getMSLevel() == 1 and rt_start <= spec.getRT() <= rt_end:
            mzs, ints = spec.get_peaks()
            mask = (mzs >= target_mz - tolerance_da) & (mzs <= target_mz + tolerance_da)
            if np.any(mask):
                for m, inten in zip(mzs[mask], ints[mask]):
                    matches.append({
                        "rt": round(float(spec.getRT()), 2),
                        "observed_mz": round(float(m), 4),
                        "error_da": round(float(m - target_mz), 4),
                        "ppm_error": round(float((m - target_mz) / target_mz * 1e6), 1),
                        "intensity": round(float(inten), 1)
                    })

    matches = sorted(matches, key=lambda x: x["intensity"], reverse=True)
    return {
        "peak_id": peak_id,
        "rt_window": f"{rt_start:.1f}s - {rt_end:.1f}s",
        "target_mz_searched": target_mz,
        "found_in_raw_scans": len(matches) > 0,
        "max_observed_hit": matches[0] if matches else None
    }


def generate_in_silico_fragments(parent_smiles: str, check_peak_id: int = 0) -> dict:
    """
    Generates RDKit-sanitized ESI+ fragments for a parent molecule, automatically pre-renders
    the 2D SVG structure image for the parent AND every fragment (returning an `image_token`
    like `[[MOL_IMG_1]]` for each), and checks if each fragment m/z is present in `check_peak_id`.
    """
    mol, clean_parent, err = safe_mol_from_smiles(parent_smiles)
    if not mol:
        return {"error": err}

    parent_em = Descriptors.ExactMolWt(mol)
    parent_formula = rdMolDescriptors.CalcMolFormula(mol)
    parent_mh = round(parent_em + 1.007276, 4)
    parent_img_token = _register_mol_svg(mol, label="Parent Molecule")

    fragments_dict = {}

    def _add_frag(frag_smi, pathway_label, is_intrinsic_cation=False):
        f_mol, f_clean, _ = safe_mol_from_smiles(frag_smi)
        if not f_mol:
            return
        em = Descriptors.ExactMolWt(f_mol)
        if em < 40.0 or em >= parent_em - 1.0:
            return
        form = rdMolDescriptors.CalcMolFormula(f_mol)
        f_charge = Chem.GetFormalCharge(f_mol)
        if is_intrinsic_cation or f_charge > 0:
            mz_val = round(em - 0.00054858, 4)
            ion_type = "[F]+ (intrinsic cation)"
        else:
            mz_val = round(em + 1.007276, 4)
            ion_type = "[F+H]+"

        key = (form, round(mz_val, 3))
        if key not in fragments_dict:
            entry = {
                "fragment_name": pathway_label,
                "smiles": f_clean,
                "formula": form,
                "neutral_exact_mass": round(em, 4),
                "ion_type": ion_type,
                "expected_esi_pos_mz": mz_val,
                "neutral_loss_from_parent_Da": round(parent_mh - mz_val, 4),
                "_temp_mol": f_mol
            }
            if check_peak_id > 0:
                hit_info = check_mz_in_peak(check_peak_id, mz_val, tolerance_da=0.05)
                entry["present_in_peak"] = hit_info.get("found_in_raw_scans", False)
                entry["peak_observation"] = hit_info.get("max_observed_hit", None)
            fragments_dict[key] = entry

    # 1. Canonical nucleoside / ribose oxocarbenium pathways
    ribose_smarts = Chem.MolFromSmarts("[C@@H,C@H,CH]1O[C@H,C@@H,CH](CO)[C@@H,C@H,CH](O)[C@H,C@@H,CH]1")
    if mol.HasSubstructMatch(ribose_smarts):
        _add_frag("OC[C@H]1O[C+]=[C@H][C@@H]1O", "Ribose oxocarbenium fragment [C5H7O3]+", is_intrinsic_cation=True)

    # 2. Single non-ring bond cleavages
    cleavable_bonds = [b.GetIdx() for b in mol.GetBonds() if not b.IsInRing() and b.GetBondType() == Chem.BondType.SINGLE]
    for b_idx in cleavable_bonds:
        try:
            frag_mol = Chem.FragmentOnBonds(mol, [b_idx], addDummies=False)
            for f in Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False):
                _add_frag(Chem.MolToSmiles(f), "Single-bond cleavage fragment ([F+H]+)")
        except Exception:
            pass

    # 3. Purine/Adenosine cross-ring and base-loss pathways
    if "ncnc2c1ncn2" in clean_parent or "ncnc2c1nc[nH]2" in clean_parent:
        _add_frag("Nc1ncnc2c1nc[nH]2", "Protonated Adenine [B+H]+ (glycosidic cleavage, -Ribose)")
        _add_frag("c1ncnc2c1nc[nH]2", "Purine core after NH3 loss ([B+H-NH3]+)")
        _add_frag("Nc1ncnc2c1ncn2C=C", "0,2X Cross-ring cleavage (Adenine + vinyl remnant)")
        _add_frag("Nc1ncnc2c1ncn2CC=O", "0,2X Cross-ring ribose cleavage (Adenine + C2H3O)")
        _add_frag("Nc1ncnc2c1ncn2[C@@H](O)C=O", "0,3X Cross-ring ribose cleavage (Adenine + C2H3O2)")

    sorted_frags = sorted(fragments_dict.values(), key=lambda x: x["expected_esi_pos_mz"], reverse=True)
    for f_entry in sorted_frags:
        f_mol = f_entry.pop("_temp_mol")
        f_entry["image_token"] = _register_mol_svg(f_mol, label=f_entry["fragment_name"])

    return {
        "parent_smiles": clean_parent,
        "parent_formula": parent_formula,
        "parent_exact_mass": round(parent_em, 4),
        "parent_[M+H]+_mz": parent_mh,
        "parent_image_token": parent_img_token,
        "generated_fragments": sorted_frags,
        "rendering_instruction": (
            "Each molecule above already has its 2D image pre-rendered! "
            "Simply include the exact `parent_image_token` or `image_token` (e.g. [[MOL_IMG_1]]) "
            "on its own line in your response where you want that structure displayed."
        )
    }


def analyze_structure_and_mass(smiles_or_formula: str, target_mz: float = 0.0, label: str = "") -> dict:
    """
    Calculates exact monoisotopic mass, formula, and ESI m/z values ([M]+, [M+H]+, [M+Na]+),
    and automatically pre-renders the 2D SVG structure image (returning `image_token`).
    """
    mol, clean_smi, err = safe_mol_from_smiles(smiles_or_formula)
    if mol is None:
        try:
            em = rdMolDescriptors.CalcExactMolWt(smiles_or_formula.strip())
            return {
                "input_interpreted_as": "Molecular Formula (no 2D structure)",
                "formula": smiles_or_formula.strip(),
                "exact_monoisotopic_mass": round(em, 5),
                "protonated_[M+H]+": round(em + 1.007276, 4)
            }
        except Exception:
            return {"error": err}

    exact_mass = Descriptors.ExactMolWt(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    formal_charge = Chem.GetFormalCharge(mol)
    mz_mh_plus = exact_mass + 1.007276 if formal_charge == 0 else exact_mass - (formal_charge * 0.00054858)

    img_token = _register_mol_svg(mol, label=label or formula)

    res = {
        "canonical_smiles": clean_smi,
        "formula": formula,
        "exact_monoisotopic_mass": round(exact_mass, 5),
        "protonated_[M+H]+": round(mz_mh_plus, 4),
        "rdkit_cLogP": round(Crippen.MolLogP(mol), 3),
        "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "image_token": img_token,
        "instruction": f"Place {img_token} on its own line in your response to display the 2D structure."
    }
    if target_mz > 0:
        diff_da = mz_mh_plus - target_mz
        res["target_mz_comparison"] = {
            "target_mz": target_mz,
            "error_Da": round(diff_da, 4),
            "ppm_error": round((diff_da / target_mz) * 1e6, 1)
        }
    return res


def generate_2d_structure_svg(smiles: str, label: str = "") -> dict:
    """Explicit tool to generate a 2D chemical structure SVG and return its `image_token` (e.g. [[MOL_IMG_1]])."""
    mol, clean_smi, err = safe_mol_from_smiles(smiles)
    if not mol:
        return {"error": err}
    token = _register_mol_svg(mol, label=label)
    return {
        "status": "success",
        "canonical_smiles": clean_smi,
        "image_token": token,
        "instruction": f"Insert the exact token {token} on its own line in your response to display this structure."
    }

def disambiguate_isobaric_peak_isomers(
    peak_id: int,
    candidate_names: list[str],
    candidate_smiles: list[str]
) -> dict:
    """
    Resolves isobaric isomers (e.g., 2'-O-methylcytidine vs 5-methylcytidine) for a given `peak_id`
    by checking raw MS1 scans for isomer-specific glycosidic cleavage ions ([BH2]+ losing
    Ribose -132.0423 Da vs 2'-O-Methylribose -146.0579 Da vs Deoxyribose -116.0473 Da)
    and comparing RDKit Crippen cLogP & TPSA to predict reverse-phase C18 elution order.
    """
    if len(candidate_names) != len(candidate_smiles):
        return {"error": "candidate_names and candidate_smiles must have the same length."}

    isomer_reports = []
    for name, smi in zip(candidate_names, candidate_smiles):
        mol, clean_smi, err = safe_mol_from_smiles(smi)
        if not mol:
            isomer_reports.append({"name": name, "error": err})
            continue

        em = Descriptors.ExactMolWt(mol)
        mh_mz = round(em + 1.007276, 4)
        clogp = round(Crippen.MolLogP(mol), 3)
        tpsa = round(rdMolDescriptors.CalcTPSA(mol), 2)
        img_token = _register_mol_svg(mol, label=f"Candidate: {name}")

        frag_res = generate_in_silico_fragments(clean_smi, check_peak_id=peak_id)
        observed_diagnostic_frags = [
            {
                "fragment_name": f["fragment_name"],
                "expected_mz": f["expected_esi_pos_mz"],
                "neutral_loss_Da": f["neutral_loss_from_parent_Da"],
                "observed_in_peak": f["peak_observation"],
                "fragment_image_token": f["image_token"]
            }
            for f in frag_res.get("generated_fragments", [])
            if f.get("present_in_peak")
        ]

        sugar_checks = []
        for loss_label, loss_da in [
            ("Unmodified Ribose loss (-132.0423 Da -> modified nucleobase [BH2]+)", 132.042259),
            ("2'-O-Methylribose loss (-146.0579 Da -> unmodified nucleobase [BH2]+)", 146.057909),
            ("2'-Deoxyribose loss (-116.0473 Da -> nucleobase [BH2]+)", 116.047344),
        ]:
            expected_bh2_mz = round(mh_mz - loss_da, 4)
            if expected_bh2_mz > 70.0:
                scan_hit = check_mz_in_peak(peak_id, expected_bh2_mz, tolerance_da=0.025)
                sugar_checks.append({
                    "pathway": loss_label,
                    "target_bh2_mz": expected_bh2_mz,
                    "detected_in_raw_ms1": scan_hit.get("found_in_raw_scans", False),
                    "max_hit": scan_hit.get("max_observed_hit")
                })

        isomer_reports.append({
            "candidate_name": name,
            "canonical_smiles": clean_smi,
            "formula": rdMolDescriptors.CalcMolFormula(mol),
            "parent_[M+H]+_mz": mh_mz,
            "rdkit_cLogP": clogp,
            "TPSA_A2": tpsa,
            "structure_image_token": img_token,
            "observed_in_silico_fragments": observed_diagnostic_frags,
            "diagnostic_glycosidic_cleavage_checks": sugar_checks
        })

    return {
        "peak_id": peak_id,
        "isobaric_comparison": isomer_reports,
        "interpretation_guide": (
            "1. Check `diagnostic_glycosidic_cleavage_checks` and `observed_in_silico_fragments` for direct in-source fragment hits. "
            "2. For reverse-phase C18 elution order, lower `rdkit_cLogP` / higher `TPSA_A2` (more polar, e.g. 5-methylcytidine with 3 free ribose OHs) "
            "elutes EARLIER than higher `rdkit_cLogP` / lower `TPSA_A2` (e.g. 2'-O-methylcytidine with a methylated 2'-OH)."
        )
    }


def assign_peaks_and_replot_chromatogram(
    peak_ids: list[int],
    compound_names: list[str],
    smiles_list: list[str],
    classifications: list[str] = None
) -> dict:
    """
    Assigns one or more peaks (including previously unassigned peaks or resolved isobaric peaks)
    to specific chemical structures, calculates mass accuracy & confidence scores against raw MS1
    scans in `od`, updates `detected_df` and `peak_assignments` in place, and generates a brand-new
    interactive Plotly chromatogram with floating RDKit 2D hover cards.
    Returns a `plot_token` (e.g. `[[PLOTLY_CHROM_1]]`) that you MUST place on its own line in your response.
    """
    global detected_df, peak_assignments

    if 'detected_df' not in globals() or 'chrom' not in globals() or chrom is None:
        return {"error": "detected_df or chrom not initialized. Run Cell 4 first."}

    if not (len(peak_ids) == len(compound_names) == len(smiles_list)):
        return {"error": "peak_ids, compound_names, and smiles_list must have the exact same length."}

    if not classifications or len(classifications) != len(peak_ids):
        classifications = ["Metabolite"] * len(peak_ids)

    adduct_defs = globals().get('esi_pos_adducts', [
        ("[M+H]+",   1, 1.007276,  1),
        ("[M+NH4]+", 1, 18.033823, 1),
        ("[M+Na]+",  1, 22.989218, 1),
        ("[M+K]+",   1, 38.963158, 1),
        ("[2M+H]+",  2, 1.007276,  1),
    ])

    updated_summary = []
    for p_id, name, smi, cls_label in zip(peak_ids, compound_names, smiles_list, classifications):
        p_id = int(p_id)
        mol, clean_smi, err = safe_mol_from_smiles(smi)
        if not mol:
            updated_summary.append({"peak_id": p_id, "status": "failed", "error": err})
            continue

        em = round(Descriptors.ExactMolWt(mol), 4)
        formula = rdMolDescriptors.CalcMolFormula(mol)

        # Screen all standard ESI+ adducts in the peak's raw MS1 scans
        observed_adducts = []
        for adduct_name, mult, shift, z in adduct_defs:
            calc_mz = (em * mult + shift) / z
            hit = check_mz_in_peak(p_id, calc_mz, tolerance_da=0.03)
            if hit.get("found_in_raw_scans") and hit.get("max_observed_hit"):
                best_hit = hit["max_observed_hit"]
                observed_adducts.append({
                    "adduct": adduct_name,
                    "obs_mz": best_hit["observed_mz"],
                    "calc_mz": calc_mz,
                    "ppm": best_hit["ppm_error"],
                    "intensity": best_hit["intensity"]
                })

        if not observed_adducts:
            calc_mh = em + 1.007276
            observed_adducts.append({
                "adduct": "[M+H]+",
                "obs_mz": round(calc_mh, 4),
                "calc_mz": calc_mh,
                "ppm": 0.0,
                "intensity": 1e6
            })
        else:
            observed_adducts = sorted(observed_adducts, key=lambda x: x["intensity"], reverse=True)

        row_mask = detected_df['peak_id'] == p_id
        dom_str = detected_df.loc[row_mask, 'dominant m/z'].values[0] if row_mask.any() else "None"
        dom_parsed = parse_dominant_mz_string(dom_str) if 'parse_dominant_mz_string' in globals() else []

        best_group = {
            "exact_mass": em,
            "formula": formula,
            "mols": [mol],
            "adducts": observed_adducts
        }
        if 'compute_confidence_from_dominant_mz' in globals():
            conf_score, evidence_str = compute_confidence_from_dominant_mz(
                best_group, dom_parsed, globals().get('peak_ms1_spectra', {}).get(p_id, {})
            )
        else:
            conf_score, evidence_str = 88.0, f"Assigned ({observed_adducts[0]['ppm']:+.1f} ppm)"

        peak_assignments[p_id] = {
            "display_name": name,
            "full_assignment": f"{name} ({cls_label})",
            "is_isobaric": False,
            "formula": formula,
            "mols": [mol],
            "adducts": observed_adducts,
            "confidence": conf_score,
            "evidence": evidence_str
        }

        top_a = observed_adducts[0]
        if row_mask.any():
            detected_df.loc[row_mask, 'Assignment'] = (
                f"{name} ({cls_label}) [{top_a['adduct']}: {top_a['obs_mz']:.4f} m/z ({top_a['ppm']:+.1f} ppm)]"
            )
            detected_df.loc[row_mask, 'Confidence (%)'] = f"{conf_score:.1f}% ({evidence_str})"

        struct_token = _register_mol_svg(mol, label=f"Peak {p_id}: {name}")
        updated_summary.append({
            "peak_id": p_id,
            "assigned_name": name,
            "classification": cls_label,
            "formula": formula,
            "primary_adduct": top_a["adduct"],
            "observed_mz": top_a["obs_mz"],
            "ppm_error": top_a["ppm"],
            "confidence_pct": conf_score,
            "evidence": evidence_str,
            "structure_image_token": struct_token
        })

    # Render fresh interactive Plotly chromatogram with a unique DOM div ID and register its placeholder token
    unique_div_id = f"hplc_chrom_replot_{uuid.uuid4().hex[:6]}"
    interactive_html = build_interactive_chromatogram_html(div_id=unique_div_id)

    plot_idx = sum(1 for k in _rendered_svg_registry if k.startswith("[[PLOTLY_CHROM_")) + 1
    plot_token = f"[[PLOTLY_CHROM_{plot_idx}]]"
    _rendered_svg_registry[plot_token] = interactive_html

    return {
        "status": "success",
        "updated_peaks": updated_summary,
        "plot_token": plot_token,
        "rendering_instruction": (
            f"CRITICAL: Place the exact token {plot_token} on its own line in your response "
            "to render the updated interactive Plotly HPLC chromatogram with floating 2D hover cards!"
        )
    }


# ==============================================================================
# SECTION 7 — Reaction-monitoring, neutral-loss, polarity & isotope Gemini tools
#             (thin wrappers over the deterministic helpers in SECTION 3/4)
# ==============================================================================

def _apex_rt_for_mz(mz: float, ppm_tolerance: float = 20.0, min_rel_to_base: float = 0.01):
    """Locates the retention time at which an exact m/z maximises (local XIC scan)."""
    scans = SPECTRAL_STATE.get("scans") or []
    if not scans:
        return None
    times, intensities = extract_xic(scans, mz, ppm_tolerance)
    if not intensities:
        return None
    best_index = max(range(len(intensities)), key=lambda i: intensities[i])
    best_intensity = intensities[best_index]
    if best_intensity <= 0:
        return None
    reference = max(SPECTRAL_STATE.get("bpc") or [0.0]) or best_intensity
    if best_intensity < reference * min_rel_to_base:
        return None
    return {"rt": round(times[best_index], 3), "intensity": best_intensity,
            "rel_to_base_pct": round(best_intensity / reference * 100.0, 2)}


def analyze_reaction_mixture(reaction_input: str, ppm_tolerance: float = 20.0) -> dict:
    """Reaction-monitoring mode. Accepts a single SMILES, a Reaction SMILES (`SM1.SM2>>Product`)
    or a labelled component list, then computes exact masses, adduct ions, cLogP/TPSA and the
    classic failure-mode m/z values (unreacted SM, hydrolysis, protodeboronation, des-halo,
    Boc/Fmoc/t-Bu cleavage), searches the raw scans for each of them and draws colour-coded
    dashed markers on the chromatogram at the retention times where they maximise."""
    parsed = parse_reaction_input(reaction_input)
    if parsed["input_type"] == "empty":
        return {"error": "Provide a SMILES, Reaction SMILES (`SM>>Product`) or labelled component list."}

    hydrate_spectral_state()
    components, markers, failure_rows = [], [], []

    for role_key, role_label in (("starting_materials", "starting_material"),
                                 ("reagents", "reagent"),
                                 ("products", "product")):
        for component in parsed[role_key]:
            mol, clean_smiles, err = safe_mol_from_smiles(component["smiles"])
            if not mol:
                components.append({"label": component["label"], "role": role_label, "error": err})
                continue
            exact_mass = Descriptors.ExactMolWt(mol)
            summary = {
                "label": component["label"],
                "role": role_label,
                "canonical_smiles": clean_smiles,
                "formula": rdMolDescriptors.CalcMolFormula(mol),
                "exact_mass": round(exact_mass, 4),
                "cLogP": round(Crippen.MolLogP(mol), 2),
                "TPSA_A2": round(rdMolDescriptors.CalcTPSA(mol), 1),
                "[M+H]+": round(exact_mass + 1.0073, 4),
                "image_token": _register_mol_svg(mol, label=f"{component['label']} ({role_label})"),
            }
            observation = _apex_rt_for_mz(summary["[M+H]+"], ppm_tolerance)
            if observation:
                summary["observed_rt"] = observation["rt"]
                summary["rel_to_base_pct"] = observation["rel_to_base_pct"]
                markers.append({
                    "label": f"{component['label']} [M+H]+",
                    "mz": summary["[M+H]+"], "rt": observation["rt"],
                    "color": REACTION_MARKER_COLORS.get(role_label, REACTION_MARKER_COLORS["other"]),
                    "category": role_label,
                })
            components.append(summary)

            if role_label == "reagent":
                continue
            for row in reaction_failure_mode_masses(exact_mass, component["label"], role_label):
                if row["failure_mode"].startswith("Unreacted") and role_label == "product":
                    continue
                hit = _apex_rt_for_mz(row["[M+H]+"], ppm_tolerance)
                row["observed"] = bool(hit)
                if hit:
                    row["observed_rt"] = hit["rt"]
                    row["rel_to_base_pct"] = hit["rel_to_base_pct"]
                    markers.append({
                        "label": f"{row['failure_mode']} ({component['label']})",
                        "mz": row["[M+H]+"], "rt": hit["rt"], "color": row["color"],
                        "category": row["category"],
                    })
                failure_rows.append(row)

    SPECTRAL_STATE["reaction_markers"] = markers
    SPECTRAL_STATE["distilled"] = None
    plot_note = update_lcms_plot(extracted_ion_mz=[m["mz"] for m in markers][:4],
                                 ppm_tolerance=ppm_tolerance)

    polarity_entries = [
        {"name": c["label"], "cLogP": c.get("cLogP"), "rt": c.get("observed_rt")}
        for c in components if c.get("observed_rt") is not None
    ]
    return {
        "input_type": parsed["input_type"],
        "components": components,
        "failure_modes_detected": [r for r in failure_rows if r.get("observed")][:12],
        "failure_modes_absent": [
            {"failure_mode": r["failure_mode"], "component": r["component"], "[M+H]+": r["[M+H]+"]}
            for r in failure_rows if not r.get("observed")
        ][:12],
        "elution_order_check": check_rt_polarity_consistency(polarity_entries),
        "plot": plot_note,
    }


def annotate_peak_neutral_losses(peak_id: int = 0) -> dict:
    """Deterministic in-source neutral-loss annotation (H2O, NH3, CO/N2, HCl, CO2, isobutene,
    Boc) across the top 10 background-subtracted ions of one distilled chromatographic peak."""
    if not SPECTRAL_STATE.get("distilled"):
        distill_peak_summary()
    peaks = (SPECTRAL_STATE.get("distilled") or {}).get("peaks", [])
    if not peaks:
        return {"error": "No distilled peaks available — run the peak detection cell first."}
    selected = [p for p in peaks if p["id"] == int(peak_id)] if peak_id else peaks
    return {
        "rt_unit": SPECTRAL_STATE["rt_unit"],
        "peaks": [
            {
                "id": peak["id"],
                "rt": peak["rt"],
                "ions": peak["ions"][:10],
                "neutral_losses": detect_neutral_losses(
                    [{"mz": ion[0], "rel": ion[1]} for ion in peak["ions"]])[:6],
                "contaminant_flags": peak.get("flags", []),
            }
            for peak in selected[:DEFAULT_TOP_CHROM_PEAKS]
        ],
    }


def check_elution_order_consistency(names: list[str] = None, smiles_list: list[str] = None,
                                    retention_times: list[float] = None) -> dict:
    """Checks whether the observed C18 reverse-phase elution order is consistent with the
    calculated lipophilicity (Crippen cLogP) and TPSA of the supplied molecules."""
    entries = []
    if names and smiles_list and retention_times:
        for name, smiles, rt in zip(names, smiles_list, retention_times):
            descriptors = compute_polarity_descriptors(smiles)
            if "error" in descriptors:
                continue
            entries.append({"name": name, "cLogP": descriptors["cLogP"],
                            "TPSA_A2": descriptors["TPSA_A2"], "rt": float(rt)})
    else:
        for marker in SPECTRAL_STATE.get("reaction_markers", []):
            if marker.get("rt") is not None:
                entries.append({"name": marker["label"], "cLogP": None, "rt": marker["rt"]})
    return {"entries": entries, "assessment": check_rt_polarity_consistency(entries)}


def screen_bench_contaminants(peak_id: int = 0) -> dict:
    """Flags common synthetic bench contaminants (TPPO, DIPEA, Et3N, DMAP, DCU, DIU, dibutyl
    phthalate, PEG oligomer ladders) in the distilled peak ion lists before interpretation."""
    if not SPECTRAL_STATE.get("distilled"):
        distill_peak_summary()
    peaks = (SPECTRAL_STATE.get("distilled") or {}).get("peaks", [])
    selected = [p for p in peaks if p["id"] == int(peak_id)] if peak_id else peaks
    return {
        "peaks": [
            {"id": peak["id"], "rt": peak["rt"],
             **flag_contaminants([ion[0] for ion in peak["ions"]])}
            for peak in selected
        ]
    }


# ==============================================================================
# SECTION 8 — Tiered model routing & context management (Feature 10)
# ==============================================================================

FAST_MODEL_DEFAULT = "gemini-2.5-flash"
DEEP_MODEL_DEFAULT = "gemini-2.5-pro"
FAST_THINKING_BUDGET = 0          # minimal thinking -> sub-second interactive turns
DEEP_THINKING_BUDGET = 4096       # structure elucidation / mechanism reasoning
TOOL_OUTPUT_MAX_CHARS = 400
PRUNE_AFTER_EXCHANGES = 3

DEEP_REASONING_TRIGGERS = (
    "elucidat", "unknown structure", "propose a structure", "de novo",
    "mechanism", "fragmentation pathway", "multi-step", "rationalise", "rationalize",
    "why does", "step by step", "think hard", "deep dive",
)


def estimate_tokens(text: str) -> int:
    """Rough 4-characters-per-token estimate used for the context budget badge."""
    return max(1, int(len(text or "") / 4))


def route_model(user_text: str, mode: str = "auto", fast_model: str = FAST_MODEL_DEFAULT,
                deep_model: str = DEEP_MODEL_DEFAULT) -> dict:
    """Tiered routing: interactive chat / peak matching / replotting stay on Flash with a
    minimal thinking budget; complex structure-elucidation or mechanism questions escalate
    to Pro. `mode` is one of 'auto', 'fast', 'deep'."""
    normalised = (user_text or "").lower()
    if mode == "fast":
        return {"model": fast_model, "thinking_budget": FAST_THINKING_BUDGET, "tier": "fast",
                "reason": "forced fast mode"}
    if mode == "deep":
        return {"model": deep_model, "thinking_budget": DEEP_THINKING_BUDGET, "tier": "deep",
                "reason": "forced deep-reasoning mode"}
    trigger = next((t for t in DEEP_REASONING_TRIGGERS if t in normalised), None)
    if trigger:
        return {"model": deep_model, "thinking_budget": DEEP_THINKING_BUDGET, "tier": "deep",
                "reason": f"matched deep-reasoning trigger '{trigger}'"}
    return {"model": fast_model, "thinking_budget": FAST_THINKING_BUDGET, "tier": "fast",
            "reason": "routine interactive turn"}


def summarize_tool_payload(payload, max_chars: int = TOOL_OUTPUT_MAX_CHARS) -> str:
    """Collapses a bulky tool response into a short placeholder kept in the chat history."""
    try:
        text = payload if isinstance(payload, str) else json.dumps(payload, default=str)
    except Exception:
        text = str(payload)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + f"... [pruned {len(text) - max_chars} chars]"


def prune_history_contents(contents, keep_last_tool_turns: int = 2,
                           max_chars: int = TOOL_OUTPUT_MAX_CHARS):
    """Replaces bulky historical function-response payloads with short summaries, keeping the
    most recent `keep_last_tool_turns` tool results intact (Feature 10 context management)."""
    contents = list(contents or [])
    tool_indices = [
        i for i, content in enumerate(contents)
        if any(getattr(part, "function_response", None) is not None
               for part in (getattr(content, "parts", None) or []))
    ]
    prunable = set(tool_indices[:-keep_last_tool_turns] if keep_last_tool_turns else tool_indices)
    for index in prunable:
        for part in (getattr(contents[index], "parts", None) or []):
            function_response = getattr(part, "function_response", None)
            if function_response is None:
                continue
            payload = getattr(function_response, "response", None)
            if payload is None:
                continue
            try:
                function_response.response = {"pruned_summary": summarize_tool_payload(payload, max_chars)}
            except Exception:
                pass
    return contents


# ==============================================================================
# SECTION 9 — Tool binding, trimmed system instruction & stateful chat UI
# ==============================================================================

client = genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))

rdkit_tools = [
    # Deterministic synthetic-chemistry layer (Features 1, 2, 3, 5, 6)
    calculate_expected_ions,
    analyze_reaction_mixture,
    theoretical_isotope_envelope,
    annotate_peak_neutral_losses,
    screen_bench_contaminants,
    check_elution_order_consistency,
    # Distilled spectral access + local replotting (Features 4, 7, 8)
    get_distilled_peak_summary,
    update_lcms_plot,
    # Existing RDKit / peak tools
    get_user_candidate_molecules,
    get_detected_peaks_summary,
    get_peak_details,
    check_mz_in_peak,
    generate_in_silico_fragments,
    analyze_structure_and_mass,
    generate_2d_structure_svg,
    # Cached + parallel external databases (Feature 9)
    lookup_compound_everywhere,
    search_pubchem_by_mass_or_formula,
    search_chembl_metabolites_and_isomers,
    query_massbank_reference_spectrum,
    disambiguate_isobaric_peak_isomers,
    assign_peaks_and_replot_chromatogram,
]


def _validated_tools(tools):
    """Drops any callable whose signature the installed google-genai SDK cannot turn into a
    function declaration, so one unsupported annotation can never break the whole chat."""
    try:
        validated = []
        for tool in tools:
            try:
                types.FunctionDeclaration.from_callable(callable=tool, client=client)
                validated.append(tool)
            except Exception as exc:
                print(f"⚠️ Tool '{getattr(tool, '__name__', tool)}' skipped: {exc}")
        return validated or list(tools)
    except Exception:
        return list(tools)


_SYSTEM_INSTRUCTION_CACHE = {"text": None}


def build_system_instruction(force_refresh: bool = False) -> str:
    """Static, cacheable system prompt: a compact candidate table plus the distilled (<300
    token) peak payload instead of raw scan tables."""
    if _SYSTEM_INSTRUCTION_CACHE["text"] and not force_refresh:
        return _SYSTEM_INSTRUCTION_CACHE["text"]

    try:
        distilled = get_distilled_peak_summary(refresh=force_refresh)
    except Exception as exc:
        distilled = json.dumps({"error": f"distillation unavailable: {exc}"})

    candidates = []
    for row in _candidate_polarity_rows():
        candidates.append(
            f"- {row['name']} ({row.get('classification', 'n/a')}): {row['formula']}, "
            f"M={row['exact_mass']}, [M+H]+={round(row['exact_mass'] + 1.0073, 4)}, "
            f"cLogP={row['cLogP']}, TPSA={row['TPSA_A2']}"
        )

    instruction = f"""You are an expert synthetic/medicinal chemistry LC-MS assistant embedded in a Colab notebook.

### Live experimental context (pre-computed client-side — do NOT ask for raw scans)
**Candidate molecules**
{chr(10).join(candidates) if candidates else "- (none supplied)"}

**Distilled peak payload** (top peaks, background-subtracted centroids, >=2% of base peak):
{distilled}

### Operating rules
1. **Deterministic chemistry first.** Call `calculate_expected_ions` for adduct/dimer matrices,
   `analyze_reaction_mixture` for SM -> product reaction monitoring and failure-mode deltas,
   `theoretical_isotope_envelope` for Cl/Br/S confirmation, `annotate_peak_neutral_losses` for
   in-source losses, and `screen_bench_contaminants` before assigning an unknown ion.
2. **Never request or echo raw m/z arrays.** Use `get_distilled_peak_summary`; use
   `check_mz_in_peak` or `get_peak_details` only for one specific ion or peak.
3. **Plot changes go through `update_lcms_plot`** (zoom window, XIC m/z list, selected scan,
   isotope overlay, peak labels). It renders locally and returns a `[[PLOTLY_CHROM_N]]` token —
   copy that token onto its own line in your reply so the figure renders.
4. **Structure images**: tools return `image_token` values such as `[[MOL_IMG_1]]`; place the exact
   token on its own line. Never invent tokens.
5. **Database lookups**: prefer `lookup_compound_everywhere` (PubChem + ChEMBL + MassBank in
   parallel, disk cached, trimmed to 6 fields per hit).
6. **Chromatography sanity**: UV/DAD relative area % is the yield estimate, not TIC. On C18, higher
   cLogP / lower TPSA must elute later — call `check_elution_order_consistency` when assignments
   look chemically odd. UV apexes map to MS scans with a {UV_MS_DELAY_MIN:.2f} min dead-volume delay.
7. **Formatting**: answer in concise GitHub-flavoured Markdown — short sections, bullet lists and
   Markdown tables (m/z, ppm error, assignment, evidence). Keep raw numbers to 4 decimal places.
"""
    _SYSTEM_INSTRUCTION_CACHE["text"] = instruction
    return instruction


curated_stable_models = [
    'gemini-2.5-flash',
    'gemini-2.5-flash-lite',
    'gemini-2.5-pro',
    'gemini-3.8-flash',
    'gemini-3.6-flash',
    'gemini-3.5-flash',
    'gemini-3.5-flash-lite',
    'gemini-3.1-flash-lite',
]

try:
    api_models = [
        m.name.replace('models/', '')
        for m in client.models.list()
        if 'generateContent' in (getattr(m, 'supported_actions', None) or [])
        and m.name.startswith('models/gemini-')
        and not any(x in m.name for x in ['preview', 'exp', 'image', 'tts', 'live', 'transcribe', 'embedding', '1.5', '2.0'])
    ]
    for m_id in sorted(api_models, reverse=True):
        if m_id not in curated_stable_models:
            curated_stable_models.append(m_id)
except Exception:
    pass


def _dropdown_value(options, preferred):
    return preferred if preferred in options else options[0]


model_dropdown = widgets.Dropdown(
    options=curated_stable_models,
    value=_dropdown_value(curated_stable_models, FAST_MODEL_DEFAULT),
    description='Fast model:',
    layout=widgets.Layout(width='300px')
)

deep_model_dropdown = widgets.Dropdown(
    options=curated_stable_models,
    value=_dropdown_value(curated_stable_models, DEEP_MODEL_DEFAULT),
    description='Deep model:',
    layout=widgets.Layout(width='300px')
)

routing_dropdown = widgets.Dropdown(
    options=[('Auto (Flash → Pro on hard questions)', 'auto'),
             ('Always fast (Flash)', 'fast'),
             ('Always deep (Pro)', 'deep')],
    value='auto',
    description='Routing:',
    layout=widgets.Layout(width='340px')
)

reset_button = widgets.Button(
    description='Reset Chat / Refresh Data',
    button_style='warning',
    icon='refresh',
    layout=widgets.Layout(width='210px')
)

refresh_plot_button = widgets.Button(
    description='Refresh Dashboard',
    button_style='info',
    icon='area-chart',
    layout=widgets.Layout(width='190px')
)

chat_input = widgets.Textarea(
    value='',
    placeholder='e.g. "Monitor BrC1=CC=C(C=O)C=C1.OB(O)c1ccccc1>>O=Cc1ccc(-c2ccccc2)cc1 — is there any protodeboronation or des-bromo byproduct?"',
    description='Input:',
    layout=widgets.Layout(width='85%', height='90px')
)

submit_button = widgets.Button(description='Send', button_style='primary', icon='paper-plane')

plot_area = widgets.Output(layout=widgets.Layout(border='1px solid #e5e7eb', padding='6px'))
output_area = widgets.Output(
    layout=widgets.Layout(border='1px solid #e5e7eb', padding='12px', height='700px', overflow_y='auto')
)

_chat_sessions = {}
chat_session = None
active_model_name = None
chat_history_ui = []


def _create_chat(model_name: str, thinking_budget: int = FAST_THINKING_BUDGET, history=None):
    """Creates a chat session bound to the validated tool list, with an optional thinking budget."""
    base_kwargs = dict(
        system_instruction=build_system_instruction(),
        tools=_validated_tools(rdkit_tools),
        temperature=0.2,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=False, maximum_remote_calls=30
        ),
    )
    try:
        config = types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget), **base_kwargs
        )
        return client.chats.create(model=model_name, config=config, history=history or [])
    except Exception:
        config = types.GenerateContentConfig(**base_kwargs)
        return client.chats.create(model=model_name, config=config, history=history or [])


def init_chat_session(force_reset: bool = False):
    """Initialises (or resets) the chat sessions and re-syncs the distilled spectral context."""
    global chat_session, active_model_name, chat_history_ui
    if force_reset:
        _chat_sessions.clear()
        chat_history_ui.clear()
        _rendered_svg_registry.clear()
        _SYSTEM_INSTRUCTION_CACHE["text"] = None
        SPECTRAL_STATE["reaction_markers"] = []
        hydrate_spectral_state(force=True)
        build_system_instruction(force_refresh=True)
        with output_area:
            clear_output()
            print("🔄 Chat reset. Spectral state re-hydrated and distilled context refreshed.")
    if chat_session is None:
        active_model_name = model_dropdown.value
        chat_session = _get_session({"model": active_model_name,
                                     "thinking_budget": FAST_THINKING_BUDGET})
    return chat_session


def _get_session(route: dict):
    """Returns (creating if needed) the chat session for a routed model, carrying over the
    pruned conversation history when the tier switches."""
    global chat_session, active_model_name
    model_name = route["model"]
    session = _chat_sessions.get(model_name)
    if session is None:
        history = []
        previous = _chat_sessions.get(active_model_name)
        if previous is not None:
            try:
                history = prune_history_contents(previous.get_history())
            except Exception:
                history = []
        session = _create_chat(model_name, route.get("thinking_budget", FAST_THINKING_BUDGET), history)
        _chat_sessions[model_name] = session
    active_model_name = model_name
    chat_session = session
    return session


def _compact_session(route: dict):
    """Rebuilds the active session from a pruned transcript once tool outputs pile up."""
    model_name = route["model"]
    session = _chat_sessions.get(model_name)
    if session is None:
        return
    try:
        history = session.get_history()
        if len(history) <= PRUNE_AFTER_EXCHANGES * 2:
            return
        _chat_sessions[model_name] = _create_chat(
            model_name, route.get("thinking_budget", FAST_THINKING_BUDGET),
            prune_history_contents(history)
        )
    except Exception:
        pass


def extract_response_text(response) -> str:
    """Concatenates every text part of a Gemini response."""
    text_chunks = []
    if getattr(response, "candidates", None):
        for part in response.candidates[0].content.parts:
            if getattr(part, 'text', None):
                text_chunks.append(part.text)
    return "".join(text_chunks) if text_chunks else "*(No text returned.)*"


def render_message_with_svgs(markdown_text: str):
    """Splits a reply on `[[MOL_IMG_X]]`, `{{SVG_X}}` and `[[PLOTLY_CHROM_X]]` tokens, rendering
    registered structure cards / interactive figures as HTML and all prose as styled Markdown."""
    pattern = r'(\[\[MOL_IMG_\d+\]\]|\{\{SVG_\d+\}\}|\[\[PLOTLY_CHROM_\d+\]\])'
    for part in re.split(pattern, markdown_text or ""):
        stripped = part.strip()
        if stripped in _rendered_svg_registry:
            display(HTML(_rendered_svg_registry[stripped]))
        elif stripped:
            display_markdown(part)


def refresh_dashboard(_button=None):
    """Renders the dual UV/DAD + BPC/TIC dashboard into the plot area."""
    with plot_area:
        clear_output(wait=True)
        try:
            status = hydrate_spectral_state()
            figure = build_lcms_dashboard_figure()
            SPECTRAL_STATE["figure"] = figure
            display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False,
                                        div_id=f"lcms_dash_{uuid.uuid4().hex[:6]}")))
            try:
                globals().setdefault('_notebook_figure_registry', {})['lcms_uv_ms_dashboard'] = \
                    figure.to_image(format='png', scale=2)
            except Exception:
                pass
            print(f"📈 {status.get('scans', 0)} MS1 scans | "
                  f"{len(SPECTRAL_STATE['chrom_peaks'])} peaks | source: {SPECTRAL_STATE['source']}")
        except Exception as exc:
            print(f"⚠️ Dashboard unavailable: {exc}")


def on_submit_clicked(b):
    user_text = chat_input.value.strip()
    if not user_text:
        return

    chat_input.disabled = True
    submit_button.disabled = True
    started = time.perf_counter()

    route = route_model(user_text, mode=routing_dropdown.value,
                        fast_model=model_dropdown.value, deep_model=deep_model_dropdown.value)
    init_chat_session()
    session = _get_session(route)

    with output_area:
        print(f"👤 You: {user_text}\n")
        print(f"🤖 Routing → {route['model']} ({route['tier']}: {route['reason']})...\n")

    try:
        response = session.send_message(user_text)
        reply_text = extract_response_text(response)
        elapsed = time.perf_counter() - started

        chat_history_ui.append(("user", user_text))
        chat_history_ui.append((
            "model",
            f"**🤖 Gemini [{route['model']} · {route['tier']}] · {elapsed:.1f}s · "
            f"~{estimate_tokens(reply_text)} output tokens**\n\n{reply_text}"
        ))
        _compact_session(route)

        with output_area:
            clear_output(wait=True)
            for role, message in chat_history_ui:
                if role == "user":
                    display_markdown(f"**👤 You:** {message}")
                else:
                    render_message_with_svgs(message)
                display(HTML("<hr style='border:0; border-top:1px solid #e5e7eb; margin:14px 0;'>"))

    except Exception as exc:
        with output_area:
            print(f"❌ Error: {exc}")

    finally:
        chat_input.value = ''
        chat_input.disabled = False
        submit_button.disabled = False


def on_reset_clicked(b):
    init_chat_session(force_reset=True)
    refresh_dashboard()


submit_button.on_click(on_submit_clicked)
reset_button.on_click(on_reset_clicked)
refresh_plot_button.on_click(refresh_dashboard)

ui = widgets.VBox([
    widgets.HTML(
        "<h3>💬 Gemini + RDKit / PubChem / ChEMBL / MassBank LC-MS Assistant</h3>"
        "<div style='font-size:12.5px; color:#4b5563;'>Adduct & contaminant matrices, reaction "
        "monitoring, isotope overlays and client-side peak distillation run locally; Gemini only "
        "ever sees the compact summary.</div>"
    ),
    widgets.HBox([model_dropdown, deep_model_dropdown]),
    widgets.HBox([routing_dropdown, reset_button, refresh_plot_button]),
    plot_area,
    chat_input,
    submit_button,
    output_area
])

display(ui)

try:
    refresh_dashboard()
except Exception as _dashboard_error:
    print(f"⚠️ Initial dashboard render skipped: {_dashboard_error}")


---
### Step 4: Multimodal Vision QC — Let Gemini *See* the Chromatogram

Until now, Gemini has only reasoned over numeric data (`detected_df`, dominant m/z lists). The cell below instead feeds the **actual rendered plot images** — the raw HPLC trace from Cell 4 and the deconvoluted, annotated chromatogram with embedded 2D structure cards from Cell 7 — directly into Gemini's multimodal vision context. This lets the model visually spot peak shapes and areas, baseline distortions, peak tailing/fronting, co-elution anomalies, and whether the drawn chemical structures look consistent with their assigned peaks — things a numeric table alone can miss.

### Step 4 · Multimodal vision QC

- **What it does:** sends the *rendered* chromatogram images (not just numbers) to Gemini so it can spot baseline drift, tailing, shoulders and co-elution that numeric tables hide.
- **How:** every figure cached in `_notebook_figure_registry` — including the new UV/MS dashboard — is rasterised to PNG and attached as an image part alongside the QC prompt.
- **Re-run with your own question**, e.g. `run_multimodal_chromatogram_vision_qc("Does the structure card on Peak 2 look right?")`.


In [ ]:
#@title Step 4: Multimodal Vision QC — Feed Chromatogram & Structure Images to Gemini
import io
from google import genai
from google.genai import types
from google.colab import userdata
from IPython.display import display, Markdown


def _figure_to_png_bytes(fig_or_bytes, dpi: int = 160) -> bytes:
    """Rasterizes a chromatogram image to PNG bytes. Accepts either a matplotlib `Figure`
    (rasterized via `savefig`, even if already auto-closed by the inline backend — the
    Figure object itself remains valid and renderable) or raw PNG bytes already exported
    from Cell 7's interactive Plotly chromatogram via `fig.to_image()`/kaleido, so Gemini
    sees the exact same zoomable chromatogram the user inspected."""
    if isinstance(fig_or_bytes, (bytes, bytearray)):
        return bytes(fig_or_bytes)
    buf = io.BytesIO()
    fig_or_bytes.savefig(buf, format='png', dpi=dpi, bbox_inches='tight')
    buf.seek(0)
    return buf.getvalue()


def collect_vision_qc_images() -> dict:
    """
    Gathers every chromatogram image stashed by Cells 4 & 7 in `_notebook_figure_registry`.
    Cell 4 caches a matplotlib Figure (auto-closed by the inline backend, but still
    rasterizable). Cell 7's chromatogram is an interactive Plotly figure; rather than
    re-rendering it statically, Cell 7 exports the *exact* figure the user is
    zooming/panning through to PNG bytes (via kaleido) and caches those bytes here, so
    this QC step sees precisely what the user sees.
    """
    registry = globals().get('_notebook_figure_registry', {})
    images = {}
    for name, fig in registry.items():
        try:
            images[name] = _figure_to_png_bytes(fig)
        except Exception as e:
            print(f"⚠️ Could not rasterize '{name}': {e}")
    return images


_vision_client = globals().get('client') or genai.Client(api_key=userdata.get("KDD_GEMINI_API_KEY"))
_vision_model_name = globals()['model_dropdown'].value if 'model_dropdown' in globals() else 'gemini-3.8-flash'

VISION_QC_PROMPT = """You are an expert analytical chemist visually inspecting these HPLC/MS chromatogram images (not just numeric data). Examine every image provided and report, peak-by-peak where possible:

1. **Peaks & areas** — how many distinct peaks are visible, their approximate retention times, and relative areas.
2. **Baseline distortions** — drift, noise, humps, or a non-zero baseline between peaks.
3. **Peak tailing / fronting** — any visible asymmetry versus an ideal Gaussian peak shape.
4. **Co-elution anomalies** — shoulders, unresolved doublets, or overlapping peaks that may hide a second compound.
5. **Chemical structures** — for any 2D molecular structure drawings embedded on the chromatogram, describe the structure and flag if it looks inconsistent with the peak it is attached to (e.g. wrong ring system, mismatched size).

Present your findings as a concise markdown report with one section per image."""


def run_multimodal_chromatogram_vision_qc(user_question: str = None) -> str:
    """Sends the cached chromatogram/structure images into Gemini's vision context and displays its QC report."""
    images = collect_vision_qc_images()
    if not images:
        msg = "⚠️ No chromatogram figures found in `_notebook_figure_registry`. Run Cells 4 and 7 first."
        display(Markdown(msg))
        return msg

    parts = [types.Part.from_text(text=user_question or VISION_QC_PROMPT)]
    for name, png_bytes in images.items():
        parts.append(types.Part.from_text(text=f"--- Image: {name} ---"))
        parts.append(types.Part.from_bytes(data=png_bytes, mime_type='image/png'))

    response = _vision_client.models.generate_content(
        model=_vision_model_name,
        contents=[types.Content(role='user', parts=parts)],
    )

    report = getattr(response, 'text', None) or "*(No text returned.)*"
    display(Markdown(f"### 🔬 Gemini Multimodal Vision QC Report ({len(images)} image(s) analyzed)\n\n{report}"))
    return report


# Run once automatically with the default QC prompt.
# Re-run any time with a custom, targeted question, e.g.:
# run_multimodal_chromatogram_vision_qc("Does the structure card on Peak 2 look right for adenosine?")
_ = run_multimodal_chromatogram_vision_qc()